# PaperVis

Publication figures.


## Setup

Shared imports and helpers.


In [ ]:
# Common setup for Figures 1-9
# Run this cell first. It contains shared imports, paths, helpers, dataset labels, and the shared figure note.
import os
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Mapping, Sequence, Tuple
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import FixedLocator, Formatter
from matplotlib.gridspec import GridSpec
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, display
import scienceplots  # noqa: F401

# =============================================================================
# SHARED PARETO CORE — Figures 4, 6, 7, 8, and 9
# =============================================================================
# Figure cells keep only data discovery, ablation encoding, labels, and their
# per-panel configuration. The common Pareto mechanics live here.

PARETO_SECONDS_PER_MINUTE = 60.0
PARETO_JOULES_PER_MWH = 3600000000.0


@dataclass(frozen=True)
class ParetoBaselinePoint:
    workload: str
    label: str
    waiting_minutes: float
    wasted_energy_mwh: float
    metrics_path: Path


def pareto_frontier(
    coordinates: Sequence[Tuple[float, float]],
) -> List[Tuple[float, float]]:
    """Return the lower-left Pareto frontier for two minimized metrics."""
    ordered = sorted(coordinates, key=lambda point: (point[0], point[1]))
    frontier: List[Tuple[float, float]] = []
    best_energy = math.inf

    for waiting, energy in ordered:
        if energy < best_energy:
            frontier.append((waiting, energy))
            best_energy = energy

    return frontier


def pareto_extract_timeout(run_name: str) -> int | None:
    """Extract ``timeout-N`` from a run-directory name."""
    match = re.search(
        r'(?:^|_)timeout-(\d+)(?:_|$)',
        run_name,
        flags=re.IGNORECASE,
    )
    return int(match.group(1)) if match else None


def pareto_strict_numeric(
    series: pd.Series,
    column: str,
    path: Path,
) -> pd.Series:
    """Validate a metric column and return finite floating-point values."""
    if series.isna().any():
        rows = series.index[series.isna()].tolist()
        raise ValueError(
            f'Null value in {path}, column {column!r}, rows {rows}'
        )

    try:
        numeric = pd.to_numeric(series, errors='raise')
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f'Non-numeric value in {path}, column {column!r}'
        ) from exc

    values = numeric.to_numpy(dtype=float)
    invalid = ~np.isfinite(values)
    if invalid.any():
        rows = numeric.index[invalid].tolist()
        raise ValueError(
            f'Non-finite value in {path}, column {column!r}, rows {rows}'
        )

    return numeric.astype(float)


def pareto_read_metrics(
    metrics_path: Path,
    *,
    waiting_divisor: float = PARETO_SECONDS_PER_MINUTE,
    energy_divisor: float = PARETO_JOULES_PER_MWH,
    source_role: str = 'pareto_metrics',
) -> Tuple[float, float]:
    """Read one Pareto metrics row with explicit unit conversion."""
    if not metrics_path.is_file():
        raise FileNotFoundError(f'Missing metrics file: {metrics_path}')

    if waiting_divisor <= 0.0 or energy_divisor <= 0.0:
        raise ValueError('Metric conversion divisors must be positive.')

    register_source_file(metrics_path, source_role)
    frame = pd.read_csv(metrics_path)

    if len(frame) != 1:
        raise ValueError(
            f'{metrics_path} must contain exactly one data row; '
            f'found {len(frame)}'
        )

    required = ('mean_waiting_time', 'total_energy_waste')
    require_columns(frame, required, metrics_path)

    waiting_value = float(
        pareto_strict_numeric(
            frame['mean_waiting_time'],
            'mean_waiting_time',
            metrics_path,
        ).iloc[0]
    )
    energy_value = float(
        pareto_strict_numeric(
            frame['total_energy_waste'],
            'total_energy_waste',
            metrics_path,
        ).iloc[0]
    )

    if waiting_value < 0.0:
        raise ValueError(f'Negative mean_waiting_time in {metrics_path}')
    if energy_value < 0.0:
        raise ValueError(f'Negative total_energy_waste in {metrics_path}')

    return (
        waiting_value / float(waiting_divisor),
        energy_value / float(energy_divisor),
    )


def pareto_heuristic_oracle_workload_dir(
    root: Path,
    platform: str,
    workload: str,
) -> Path:
    """Return the shared Heuristic-IPM workload directory."""
    return Path(root) / platform / workload


def pareto_apply_axis_tick_format(
    ax: plt.Axes,
    *,
    axis: str,
    decimals: int = 2,
    integer_tolerance: float = 1e-9,
) -> None:
    """Use three major ticks and compact numeric labels on one axis."""
    if axis not in {'x', 'y'}:
        raise ValueError("axis must be either 'x' or 'y'.")
    if decimals < 0:
        raise ValueError('decimals must be non-negative.')

    matplotlib_axis = ax.xaxis if axis == 'x' else ax.yaxis
    matplotlib_axis.set_major_locator(
        matplotlib.ticker.MaxNLocator(nbins=3, min_n_ticks=3)
    )

    def format_tick(value, position):
        if abs(value - round(value)) < integer_tolerance:
            return str(int(round(value)))
        return f'{value:.{decimals}f}'.rstrip('0').rstrip('.')

    matplotlib_axis.set_major_formatter(
        matplotlib.ticker.FuncFormatter(format_tick)
    )


def pareto_apply_xy_tick_format(
    ax: plt.Axes,
    *,
    decimals: int = 2,
    integer_tolerance: float = 1e-9,
) -> None:
    """Apply the same three-tick formatting to both Pareto axes."""
    pareto_apply_axis_tick_format(
        ax,
        axis='x',
        decimals=decimals,
        integer_tolerance=integer_tolerance,
    )
    pareto_apply_axis_tick_format(
        ax,
        axis='y',
        decimals=decimals,
        integer_tolerance=integer_tolerance,
    )


def pareto_keep_annotations_inside_axes(
    fig: object,
    annotations: Sequence[Tuple[object, object]],
    *,
    adjust_passes: int,
    pad_pixels: float,
) -> None:
    """Move annotation text just enough to keep it inside its panel."""
    if not annotations:
        return

    pixels_per_point = fig.dpi / 72.0

    for _ in range(adjust_passes):
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        changed = False

        for ax, annotation in annotations:
            text_bbox = annotation.get_window_extent(renderer=renderer)
            axes_bbox = ax.get_window_extent(renderer=renderer)
            left = axes_bbox.x0 + pad_pixels
            right = axes_bbox.x1 - pad_pixels
            bottom = axes_bbox.y0 + pad_pixels
            top = axes_bbox.y1 - pad_pixels
            dx_pixels = 0.0
            dy_pixels = 0.0

            if text_bbox.x0 < left:
                dx_pixels = left - text_bbox.x0
            elif text_bbox.x1 > right:
                dx_pixels = right - text_bbox.x1

            if text_bbox.y0 < bottom:
                dy_pixels = bottom - text_bbox.y0
            elif text_bbox.y1 > top:
                dy_pixels = top - text_bbox.y1

            if dx_pixels == 0.0 and dy_pixels == 0.0:
                continue

            old_x, old_y = annotation.get_position()
            annotation.set_position((
                old_x + dx_pixels / pixels_per_point,
                old_y + dy_pixels / pixels_per_point,
            ))
            changed = True

        if not changed:
            break


def pareto_finalize_panel(
    ax: plt.Axes,
    coordinates: Sequence[Tuple[float, float]],
    *,
    x_margin: float,
    y_margin: float,
    visible_fraction_threshold: Mapping[str, float],
    outlier_region_fraction: Mapping[str, float],
    tick_decimals: int = 2,
    integer_tolerance: float = 1e-9,
) -> None:
    """Apply the shared grid, margins, ticks, and outlier handling."""
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_axisbelow('line')
    ax.margins(x=x_margin, y=y_margin)
    pareto_apply_xy_tick_format(
        ax,
        decimals=tick_decimals,
        integer_tolerance=integer_tolerance,
    )

    PV4NEW_apply_pareto_high_outlier_zone(
        ax,
        [coordinate[0] for coordinate in coordinates],
        axis='x',
        visible_fraction_threshold=visible_fraction_threshold['x'],
        outlier_region_fraction=outlier_region_fraction['x'],
        margin_fraction=x_margin,
    )
    PV4NEW_apply_pareto_high_outlier_zone(
        ax,
        [coordinate[1] for coordinate in coordinates],
        axis='y',
        visible_fraction_threshold=visible_fraction_threshold['y'],
        outlier_region_fraction=outlier_region_fraction['y'],
        margin_fraction=y_margin,
    )



PROJECT_ROOT = Path(os.environ.get('SPARS_PROJECT_ROOT', Path.cwd())).expanduser().resolve()
RESULT_SAVE_3_ROOT = PROJECT_ROOT / 'results'
OUTPUT_DIR = PROJECT_ROOT / 'new_output'
DATASET_DISPLAY_LABELS: Dict[str, str] = {
    "DAS2-fs1-0-3000": "DAS2 FS1",
    "DAS2-fs2-0-3000": "DAS2 FS2",
    "DAS2-fs3-0-3000": "DAS2 FS3",
    "DAS2-fs4-0-3000": "DAS2 FS4",
    "generated-markovian-3000": "Markovian 3000",
    "SDSC-BLUE-2000-4.2-cln-0-3000": "SDSC Blue",
}
FIGURE_CONFIG_RESULT_DIRS: Dict[Path, set[Path]] = {}

def dataset_display_label(dataset: str) -> str:
    return DATASET_DISPLAY_LABELS.get(str(dataset), str(dataset))

def register_source_file(path: Path, source_role: str) -> None:
    resolved = path.expanduser().resolve()
    key = (source_role, str(resolved))
    if key in SOURCE_DIR_SEEN:
        return
    SOURCE_DIR_SEEN.add(key)
    SOURCE_DIR_RECORDS.append({'source_role': source_role, 'source_root': source_root_label(resolved), 'source_dir': str(resolved.parent), 'source_file': resolved.name, 'source_path': str(resolved)})

def source_dir_rows(*, figure_name: str, chart_name: str, comparisons: Dict[str, Sequence[Path]]) -> List[Dict[str, str]]:
    rows: List[Dict[str, str]] = []
    seen: set[Tuple[str, str, str]] = set()
    for comparison_name, result_dirs in comparisons.items():
        for result_dir in result_dirs:
            resolved = result_dir.expanduser().resolve()
            key = (chart_name, comparison_name, str(resolved))
            if key in seen:
                continue
            seen.add(key)
            rows.append({'figure': figure_name, 'chart': chart_name, 'comparison': comparison_name, 'source_root': source_root_label(resolved), 'source_dir': str(resolved)})
    rows.sort(key=lambda row: (row['figure'], row['chart'], row['comparison'], row['source_root'], row['source_dir']))
    return rows

def write_chart_source_dirs(chart_path: Path, comparisons: Dict[str, Sequence[Path]]) -> Path:
    figure_dir = chart_path.parent.expanduser().resolve()
    output_path = chart_path.with_name(f'{chart_path.stem}_source_dirs.csv')
    frame = pd.DataFrame(source_dir_rows(figure_name=figure_dir.name, chart_name=chart_path.name, comparisons=comparisons), columns=('figure', 'chart', 'comparison', 'source_root', 'source_dir'))
    frame.to_csv(output_path, index=False)
    print(f'Wrote {output_path}')
    return output_path
SSC_ALGO_CONFIG_VARIANT_PATTERN = re.compile('^(?:transition-duration-|power-sweep-)?snf-ssc(?:-nf|-ng|-ngnf|-nfng)?(?:_|$)')

def config_output_has_ssc_variant_marker(result_dir: Path, flattened: Dict[str, object]) -> bool:
    """Return True when this result belongs to an SNF-SSC/ICON variant.

    `paths.output` is intentionally ignored in the config comparison reports,
    so callers may pass a flattened config where that key has already been
    removed.  Fall back to the actual result directory path instead of raising,
    because the directory layout also carries the variant name
    (`snf-ssc_64`, `snf-ssc_alpha-...`, `transition-duration-snf-ssc_...`, etc.).
    """
    paths_to_check: List[object] = [result_dir]
    output_value = flattened.get('paths.output')
    if output_value is not None:
        paths_to_check.insert(0, output_value)
    for path_value in paths_to_check:
        for part in Path(str(path_value)).parts:
            if SSC_ALGO_CONFIG_VARIANT_PATTERN.match(part):
                return True
    return False

def config_uses_algo_config(result_dir: Path, flattened: Dict[str, object] | None=None) -> bool:
    if flattened is None:
        flattened = read_simulator_config_flat(result_dir)
    return flattened.get('run.algorithm') == 'snf_icon' and config_output_has_ssc_variant_marker(result_dir, flattened)

def config_key_applies_to_result_dir(key: str, result_dir: Path, flattened: Dict[str, object] | None=None) -> bool:
    if not key.startswith('run.algo_config.'):
        return True
    return config_uses_algo_config(result_dir, flattened)

def config_value_for_audit(flattened: Dict[str, object], key: str) -> object:
    return flattened.get(key, '')

def config_algo_value_for_audit(result_dir: Path, flattened: Dict[str, object], key: str) -> object:
    if not config_key_applies_to_result_dir(key, result_dir, flattened):
        return ''
    return config_value_for_audit(flattened, key)

def write_chart_parameter_audit(chart_path: Path, comparisons: Dict[str, Sequence[Path]]) -> Path:
    figure_dir = chart_path.parent.expanduser().resolve()
    output_path = chart_path.with_name(f'{chart_path.stem}_parameter_audit.csv')
    rows: List[Dict[str, object]] = []
    seen: set[Tuple[str, str]] = set()
    for comparison_name, result_dirs in comparisons.items():
        for result_dir in result_dirs:
            resolved = result_dir.expanduser().resolve()
            key = (comparison_name, str(resolved))
            if key in seen:
                continue
            seen.add(key)
            flattened = read_simulator_config_flat(resolved)
            rows.append({'figure': figure_dir.name, 'chart': chart_path.name, 'comparison': comparison_name, 'source_root': source_root_label(resolved), 'source_dir': str(resolved), 'paths.platform': config_path_stem_value(resolved, 'paths.platform'), 'paths.workload': config_path_stem_value(resolved, 'paths.workload'), 'run.algorithm': config_value_for_audit(flattened, 'run.algorithm'), 'run.algo_config.timeout': config_value_for_audit(flattened, 'run.algo_config.timeout'), 'uses_algo_config': config_uses_algo_config(resolved, flattened), 'run.algo_config.alpha': config_algo_value_for_audit(resolved, flattened, 'run.algo_config.alpha'), 'run.algo_config.beta': config_algo_value_for_audit(resolved, flattened, 'run.algo_config.beta'), 'run.algo_config.markov_window_seconds': config_algo_value_for_audit(resolved, flattened, MARKOV_WINDOW_CONFIG_KEY)})
    rows.sort(key=lambda row: (str(row['figure']), str(row['chart']), str(row['comparison']), str(row['source_root']), str(row['source_dir'])))
    pd.DataFrame(rows, columns=('figure', 'chart', 'comparison', 'source_root', 'source_dir', 'paths.platform', 'paths.workload', 'run.algorithm', 'run.algo_config.timeout', 'uses_algo_config', 'run.algo_config.alpha', 'run.algo_config.beta', 'run.algo_config.markov_window_seconds')).to_csv(output_path, index=False)
    print(f'Wrote {output_path}')
    return output_path

def write_figure_source_dirs(figure_dir: Path) -> Path:
    figure_dir = figure_dir.expanduser().resolve()
    if figure_dir not in FIGURE_CONFIG_RESULT_DIRS:
        raise KeyError(f'No registered result directories for figure directory: {figure_dir}')
    output_path = figure_dir / f'CustomVisPaper_{figure_dir.name}_source_dirs.csv'
    comparisons = {figure_dir.name: sorted(FIGURE_CONFIG_RESULT_DIRS[figure_dir], key=lambda path: str(path))}
    frame = pd.DataFrame(source_dir_rows(figure_name=figure_dir.name, chart_name='ALL_CHARTS_IN_FIGURE', comparisons=comparisons), columns=('figure', 'chart', 'comparison', 'source_root', 'source_dir'))
    frame.to_csv(output_path, index=False)
    print(f'Wrote {output_path}')
    return output_path

def apply_plot_style() -> None:
    plt.style.use(['science', 'no-latex', 'grid'])
    plt.rcParams.update({
        'font.size': 12,
        'axes.titlesize': 12,
        'axes.labelsize': 12,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'figure.titlesize': 12,
        'figure.labelsize': 12,
        'legend.fontsize': 10,
        'figure.dpi': 140,
        'savefig.dpi': 300,
    })
    
    
def save_figure(
    fig: object,
    png_path: Path,
    pdf_path: Path,
    *,
    png_kwargs: Mapping[str, object] | None = None,
    pdf_kwargs: Mapping[str, object] | None = None,
) -> Tuple[Path, Path]:
    """Save one title-free PNG and PDF."""
    png_path = Path(png_path)
    pdf_path = Path(pdf_path)
    png_path.parent.mkdir(parents=True, exist_ok=True)
    pdf_path.parent.mkdir(parents=True, exist_ok=True)

    suptitle = getattr(fig, '_suptitle', None)
    if suptitle is not None:
        suptitle.remove()

    fig.savefig(png_path, **dict(png_kwargs or {}))
    fig.savefig(pdf_path, **dict(pdf_kwargs or {}))
    return png_path, pdf_path


def require_columns(frame: pd.DataFrame, required: Sequence[str], path: Path) -> None:
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise KeyError(f'{path} is missing required columns: {missing}')
SIMULATOR_CONFIG_FILENAME = 'simulator_config_used.yaml'
MISSING_CONFIG_VALUE = '<missing>'
SWEEP_CONFIG_IGNORED_KEYS = {'paths.output', 'logging.file', 'run.algo_config.decision_log_path'}
MARKOV_WINDOW_CONFIG_KEY = 'run.algo_config.markov_window_seconds'
MARKOV_WINDOW_SYMBOL = 'T_M'

def flatten_simulator_config(value: object, prefix: str='') -> Dict[str, object]:
    flattened: Dict[str, object] = {}
    if isinstance(value, dict):
        for key, child in value.items():
            child_key = f'{prefix}.{key}' if prefix else str(key)
            flattened.update(flatten_simulator_config(child, child_key))
    else:
        flattened[prefix] = value
    return flattened

def read_simulator_config_flat(result_dir: Path) -> Dict[str, object]:
    config_path = result_dir / SIMULATOR_CONFIG_FILENAME
    if not config_path.is_file():
        raise FileNotFoundError(f'Missing simulator config: {config_path}')
    register_source_file(config_path, 'simulator_config')
    with config_path.open('r', encoding='utf-8') as handle:
        config = yaml.safe_load(handle)
    if not isinstance(config, dict):
        raise ValueError(f'Simulator config must contain a YAML mapping: {config_path}')
    return {key: value for key, value in flatten_simulator_config(config).items() if key not in SWEEP_CONFIG_IGNORED_KEYS}

def config_numeric_value(result_dir: Path, key: str) -> float:
    flattened = read_simulator_config_flat(result_dir)
    if key not in flattened:
        raise KeyError(f'Missing simulator config key {key!r} in {result_dir / SIMULATOR_CONFIG_FILENAME}')
    value = flattened[key]
    try:
        numeric = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f'Simulator config key {key!r} must be numeric in {result_dir / SIMULATOR_CONFIG_FILENAME}: {value!r}') from exc
    if not math.isfinite(numeric):
        raise ValueError(f'Simulator config key {key!r} must be finite in {result_dir / SIMULATOR_CONFIG_FILENAME}: {value!r}')
    return numeric

def validate_config_numeric_matches(*, result_dir: Path, value_name: str, config_value: float, source_value: float, source_description: str) -> None:
    tolerance = 1e-09 * max(1.0, abs(config_value), abs(source_value))
    if abs(config_value - source_value) <= tolerance:
        return
    raise RuntimeError(f'{value_name} mismatch for {result_dir}: {source_description} gives {source_value:g}, but simulator_config_used.yaml gives {config_value:g}')
SCHEDULER_TIMEOUT_CONFIG_KEY = 'run.algo_config.timeout'

def config_scheduler_timeout_seconds(result_dir: Path) -> float:
    return config_numeric_value(
        result_dir,
        SCHEDULER_TIMEOUT_CONFIG_KEY,
    )

def validated_config_timeout_seconds(result_dir: Path, expected_timeout_seconds: int, source_description: str) -> int:
    timeout_seconds = config_scheduler_timeout_seconds(result_dir)
    validate_config_numeric_matches(result_dir=result_dir, value_name='timeout', config_value=timeout_seconds, source_value=float(expected_timeout_seconds), source_description=source_description)
    rounded = round(timeout_seconds)
    if abs(timeout_seconds - rounded) > 1e-09:
        raise ValueError(f'Timeout must be an integer number of seconds in {result_dir / SIMULATOR_CONFIG_FILENAME}: {timeout_seconds!r}')
    return int(rounded)

def config_path_stem_value(result_dir: Path, key: str) -> str:
    flattened = read_simulator_config_flat(result_dir)
    if key not in flattened:
        raise KeyError(f'Missing simulator config key {key!r} in {result_dir / SIMULATOR_CONFIG_FILENAME}')
    value = flattened[key]
    if value is None:
        raise ValueError(f'Simulator config key {key!r} must not be null in {result_dir / SIMULATOR_CONFIG_FILENAME}')
    return Path(str(value)).stem

def config_string_value(result_dir: Path, key: str) -> str:
    flattened = read_simulator_config_flat(result_dir)
    if key not in flattened:
        raise KeyError(
            f'Missing simulator config key {key!r} in '
            f'{result_dir / SIMULATOR_CONFIG_FILENAME}'
        )
    value = flattened[key]
    if value is None:
        raise ValueError(
            f'Simulator config key {key!r} must not be null in '
            f'{result_dir / SIMULATOR_CONFIG_FILENAME}'
        )
    return str(value)

def validate_result_config_identity(
    result_dir: Path,
    *,
    expected_platform: str,
    expected_workload: str,
    expected_algorithms: Sequence[str] | None = None,
) -> Dict[str, object]:
    flattened = read_simulator_config_flat(result_dir)

    for key, expected in (
        ('paths.platform', expected_platform),
        ('paths.workload', expected_workload),
    ):
        if key not in flattened:
            raise KeyError(
                f'Missing simulator config key {key!r} in '
                f'{result_dir / SIMULATOR_CONFIG_FILENAME}'
            )

        actual = Path(str(flattened[key])).stem
        if actual != expected:
            raise RuntimeError(
                f'{key} mismatch for {result_dir}: directory selection '
                f'gives {expected!r}, but simulator_config_used.yaml '
                f'gives {actual!r}'
            )

    if expected_algorithms is not None:
        if 'run.algorithm' not in flattened:
            raise KeyError(
                f'Missing simulator config key "run.algorithm" in '
                f'{result_dir / SIMULATOR_CONFIG_FILENAME}'
            )

        actual_algorithm = str(flattened['run.algorithm'])
        allowed = tuple(str(value) for value in expected_algorithms)
        if actual_algorithm not in allowed:
            raise RuntimeError(
                f'run.algorithm mismatch for {result_dir}: expected one of '
                f'{allowed!r}, but simulator_config_used.yaml gives '
                f'{actual_algorithm!r}'
            )

    return flattened

def scheduler_timeout_title_label(
    result_dirs: Sequence[Path],
) -> str:
    values: List[float] = []

    for result_dir in result_dirs:
        value = float(config_scheduler_timeout_seconds(result_dir))
        if not any(
            abs(value - existing)
            <= 1e-09 * max(1.0, abs(value), abs(existing))
            for existing in values
        ):
            values.append(value)

    if not values:
        raise KeyError(
            'No scheduler timeout value found for title generation'
        )

    values.sort()
    hours = [value / 3600.0 for value in values]

    if len(hours) == 1:
        return f'$\\Delta_t$={hours[0]:g} h'

    return (
        f'$\\Delta_t\\in['
        f'{hours[0]:g}, {hours[-1]:g}'
        f']$ h'
    )

def unique_config_path_stem_values(result_dirs: Sequence[Path], key: str) -> List[str]:
    values: List[str] = []
    for result_dir in result_dirs:
        value = config_path_stem_value(result_dir, key)
        if value not in values:
            values.append(value)
    values.sort()
    return values

def config_path_title_label(result_dirs: Sequence[Path], key: str, label: str) -> str:
    values = unique_config_path_stem_values(result_dirs, key)
    if not values:
        raise KeyError(f'No {key} value found for title generation')
    if len(values) == 1:
        return values[0]
    return label + '$\\in\\{$' + ', '.join(values) + '$\\}$'

def platform_title_label(result_dirs: Sequence[Path]) -> str:
    return config_path_title_label(result_dirs, 'paths.platform', 'platform')

def workload_title_label(result_dirs: Sequence[Path]) -> str:
    values = unique_config_path_stem_values(result_dirs, 'paths.workload')
    if not values:
        raise KeyError('No paths.workload value found for title generation')
    labels = [dataset_display_label(value) for value in values]
    if len(labels) == 1:
        return labels[0]
    return 'workload' + '$\\in\\{$' + ', '.join(labels) + '$\\}$'

def compact_unique_values(values: Sequence[object]) -> List[object]:
    unique: List[object] = []
    for value in values:
        if not any((value == existing for existing in unique)):
            unique.append(value)
    if all((isinstance(value, (int, float)) and (not isinstance(value, bool)) for value in unique)):
        return sorted(unique)
    if all((isinstance(value, str) for value in unique)):
        return sorted(unique)
    return unique

def format_markov_window_common_value(value: object) -> str:
    try:
        seconds = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f'Invalid Markov-window value for {MARKOV_WINDOW_CONFIG_KEY}: {value!r}') from exc
    if not math.isfinite(seconds):
        raise ValueError(f'Non-finite Markov-window value for {MARKOV_WINDOW_CONFIG_KEY}: {value!r}')
    return f'{seconds / 3600.0:g} h'

def config_title_symbol(key: str) -> str | None:
    normalized = key.lower().replace('-', '_')
    if normalized == 'run.algo_config.alpha' or normalized.endswith('.alpha'):
        return '\\alpha'
    if normalized == 'run.algo_config.beta' or normalized.endswith('.beta'):
        return '\\beta'
    if 'markov_window' in normalized:
        return MARKOV_WINDOW_SYMBOL
    return None

def format_title_config_value(value: object, key: str) -> str:
    normalized = key.lower().replace('-', '_')
    if value == MISSING_CONFIG_VALUE:
        return str(value)
    if 'markov_window' in normalized:
        return format_markov_window_common_value(value)
    if isinstance(value, (int, float)) and (not isinstance(value, bool)):
        numeric = float(value)
        if not math.isfinite(numeric):
            raise ValueError(f'Non-finite title config value for {key}: {value!r}')
        return f'{numeric:g}'
    return str(value)

def format_title_config_range(values: Sequence[object], key: str) -> str:
    if not values:
        raise ValueError(f'No swept values for {key}')
    if all((isinstance(value, (int, float)) and (not isinstance(value, bool)) for value in values)):
        numeric_values = [float(value) for value in values]
        if not all((math.isfinite(value) for value in numeric_values)):
            raise ValueError(f'Non-finite swept values for {key}: {values!r}')
        low = min(numeric_values)
        high = max(numeric_values)
        return f'{format_title_config_value(low, key)}, {format_title_config_value(high, key)}'
    ordered_values = sorted((str(value) for value in values))
    return f'{ordered_values[0]}, {ordered_values[-1]}'

def config_title_assignment(key: str, common_config: Dict[str, object], swept_config: Dict[str, List[object]]) -> str | None:
    symbol = config_title_symbol(key)
    if symbol is None:
        return None
    if key in swept_config:
        value_range = format_title_config_range(swept_config[key], key)
        return f'${symbol}\\in[{value_range}]$'
    if key in common_config:
        value = format_title_config_value(common_config[key], key)
        return f'${symbol}={value}$'
    return None

def config_values_title_label(result_dirs: Sequence[Path], keys: Sequence[str]) -> str:
    common_config = build_common_config_report(result_dirs)
    swept_config = build_swept_config_report(result_dirs)
    parts: List[str] = []
    for key in keys:
        assignment = config_title_assignment(key, common_config, swept_config)
        if assignment is not None:
            parts.append(assignment)
    return ' / '.join(parts)

def build_swept_config_report(result_dirs: Sequence[Path]) -> Dict[str, List[object]]:
    unique_result_dirs: List[Path] = []
    seen_dirs = set()
    for result_dir in result_dirs:
        resolved = result_dir.expanduser().resolve()
        if resolved in seen_dirs:
            continue
        seen_dirs.add(resolved)
        unique_result_dirs.append(resolved)
    unique_result_dirs.sort(key=lambda path: str(path))
    if not unique_result_dirs:
        raise ValueError('No result directories supplied for config comparison')
    flattened_by_dir: List[Tuple[Path, Dict[str, object]]] = []
    for result_dir in unique_result_dirs:
        config_path = result_dir / SIMULATOR_CONFIG_FILENAME
        if not config_path.is_file():
            raise FileNotFoundError(f'Missing simulator config: {config_path}')
        register_source_file(config_path, 'simulator_config')
        with config_path.open('r', encoding='utf-8') as handle:
            config = yaml.safe_load(handle)
        if not isinstance(config, dict):
            raise ValueError(f'Simulator config must contain a YAML mapping: {config_path}')
        flattened_by_dir.append((result_dir, {key: value for key, value in flatten_simulator_config(config).items() if key not in SWEEP_CONFIG_IGNORED_KEYS}))
    all_keys = sorted(set().union(*(set(config.keys()) for _, config in flattened_by_dir)))
    swept_config: Dict[str, List[object]] = {}
    for key in all_keys:
        applicable_configs = [config for result_dir, config in flattened_by_dir if config_key_applies_to_result_dir(key, result_dir, config)]
        if not applicable_configs:
            continue
        values = compact_unique_values([config.get(key, MISSING_CONFIG_VALUE) for config in applicable_configs])
        if len(values) > 1:
            swept_config[key] = values
    return swept_config

def write_chart_swept_config(chart_path: Path, comparisons: Dict[str, Sequence[Path]]) -> Path:
    figure_dir = chart_path.parent.expanduser().resolve()
    figure_result_dirs = FIGURE_CONFIG_RESULT_DIRS.setdefault(figure_dir, set())
    for result_dirs in comparisons.values():
        for result_dir in result_dirs:
            figure_result_dirs.add(result_dir.expanduser().resolve())
    report = {'chart': chart_path.name, 'sweeps': {comparison_name: build_swept_config_report(result_dirs) for comparison_name, result_dirs in comparisons.items()}}
    output_path = chart_path.with_name(f'{chart_path.stem}_swept_config.yaml')
    with output_path.open('w', encoding='utf-8') as handle:
        yaml.safe_dump(report, handle, sort_keys=False, allow_unicode=True)
    print(f'Wrote {output_path}')
    write_chart_source_dirs(chart_path, comparisons)
    write_chart_parameter_audit(chart_path, comparisons)
    write_figure_common_config(figure_dir)
    write_figure_source_dirs(figure_dir)
    return output_path

def build_common_config_report(result_dirs: Sequence[Path]) -> Dict[str, object]:
    unique_result_dirs = sorted({result_dir.expanduser().resolve() for result_dir in result_dirs}, key=lambda path: str(path))
    if not unique_result_dirs:
        raise ValueError('No result directories collected for common-config comparison')
    flattened_by_dir: List[Tuple[Path, Dict[str, object]]] = []
    for result_dir in unique_result_dirs:
        config_path = result_dir / SIMULATOR_CONFIG_FILENAME
        if not config_path.is_file():
            raise FileNotFoundError(f'Missing simulator config: {config_path}')
        register_source_file(config_path, 'simulator_config')
        with config_path.open('r', encoding='utf-8') as handle:
            config = yaml.safe_load(handle)
        if not isinstance(config, dict):
            raise ValueError(f'Simulator config must contain a YAML mapping: {config_path}')
        flattened_by_dir.append((result_dir, {key: value for key, value in flatten_simulator_config(config).items() if key not in SWEEP_CONFIG_IGNORED_KEYS}))
    all_keys = sorted(set().union(*(set(config.keys()) for _, config in flattened_by_dir)))
    common_config: Dict[str, object] = {}
    for key in all_keys:
        applicable_configs = [config for result_dir, config in flattened_by_dir if config_key_applies_to_result_dir(key, result_dir, config)]
        if not applicable_configs:
            continue
        if any((key not in config for config in applicable_configs)):
            continue
        values = compact_unique_values([config[key] for config in applicable_configs])
        if len(values) == 1:
            common_config[key] = values[0]
    if MARKOV_WINDOW_CONFIG_KEY in common_config:
        common_config[f'${MARKOV_WINDOW_SYMBOL}$'] = format_markov_window_common_value(common_config[MARKOV_WINDOW_CONFIG_KEY])
    return common_config

def write_figure_common_config(figure_dir: Path) -> Path:
    figure_dir = figure_dir.expanduser().resolve()
    if figure_dir not in FIGURE_CONFIG_RESULT_DIRS:
        raise KeyError(f'No registered result directories for figure directory: {figure_dir}')
    figure_dir.mkdir(parents=True, exist_ok=True)
    output_path = figure_dir / f'CustomVisPaper_{figure_dir.name}_common_config.yaml'
    report = {'figure': figure_dir.name, 'common_config': build_common_config_report(list(FIGURE_CONFIG_RESULT_DIRS[figure_dir]))}
    with output_path.open('w', encoding='utf-8') as handle:
        yaml.safe_dump(report, handle, sort_keys=False, allow_unicode=True)
    print(f'Wrote {output_path}')
    return output_path

# =============================================================================
# FCFS/B+IPM COMPARISON TABLE HELPERS
# Positive percentages mean the plotted method is better than FCFS/B+IPM.
# =============================================================================
COMPARISON_BASELINE_DISPLAY_LABEL = 'FCFS/B+IPM'


def comparison_improvement_percent(
    value: float,
    baseline: float,
    *,
    higher_is_better: bool = False,
) -> float:
    value = float(value)
    baseline = float(baseline)
    if not math.isfinite(value) or not math.isfinite(baseline):
        return float('nan')
    if abs(baseline) <= 1e-12:
        return 0.0 if abs(value) <= 1e-12 else float('nan')
    if higher_is_better:
        return 100.0 * (value - baseline) / abs(baseline)
    return 100.0 * (baseline - value) / abs(baseline)


def display_comparison_table(frame: pd.DataFrame, title: str) -> pd.DataFrame:
    exact_frame = frame.reset_index(drop=True).copy()
    shown = exact_frame.copy()
    for column in shown.select_dtypes(include=[np.number]).columns:
        decimals = 2 if 'percent' in str(column).lower() else 4
        shown[column] = shown[column].round(decimals)

    with pd.option_context(
        'display.max_rows', None,
        'display.max_columns', None,
        'display.width', 2400,
        'display.max_colwidth', None,
    ):
        display(shown)
    return exact_frame


def comparison_result_text(percent: float, *, is_baseline: bool=False) -> str:
    if is_baseline:
        return 'baseline'
    percent = float(percent)
    if not math.isfinite(percent):
        return 'n/a'
    if abs(percent) < 0.005:
        return 'same as baseline'
    if percent > 0.0:
        return f'{percent:.2f}% better'
    return f'{abs(percent):.2f}% worse'


def add_explicit_fcfs_b_ipm_comparisons(
    frame: pd.DataFrame,
    *,
    group_columns: Sequence[str],
    metric_specs: Sequence[Tuple[str, str, str, str, str, bool]],
    algorithm_column: str='algorithm',
    baseline_label: str=COMPARISON_BASELINE_DISPLAY_LABEL,
) -> pd.DataFrame:
    """Add explicit FCFS/B+IPM values, differences, percentages, and verdicts.

    Each metric spec is:
      (value_column, baseline_column, difference_column,
       improvement_percent_column, comparison_text_column, higher_is_better)
    """
    result = frame.reset_index(drop=True).copy()

    # Remove older percentage-only columns so the displayed table is not duplicated
    # or pushed off-screen by long legacy names.
    legacy_columns = [
        column for column in result.columns
        if str(column).lower().endswith('_improvement_vs_fcfs_b_ipm_percent')
    ]
    if legacy_columns:
        result = result.drop(columns=legacy_columns)

    missing = [
        column for column in [*group_columns, algorithm_column]
        if column not in result.columns
    ]
    if missing:
        raise KeyError(f'Missing comparison-table columns: {missing}')

    baseline_rows = result.loc[result[algorithm_column] == baseline_label].copy()
    if baseline_rows.empty:
        raise RuntimeError(f'No {baseline_label} row exists in the comparison table.')
    if baseline_rows.duplicated(list(group_columns)).any():
        duplicate_groups = baseline_rows.loc[
            baseline_rows.duplicated(list(group_columns), keep=False),
            list(group_columns),
        ]
        raise RuntimeError(
            f'Multiple {baseline_label} rows exist for comparison groups: '
            f'{duplicate_groups.to_dict(orient="records")}'
        )

    baseline_value_columns = [spec[0] for spec in metric_specs]
    baseline_rename = {spec[0]: spec[1] for spec in metric_specs}
    baseline_frame = baseline_rows[
        list(group_columns) + baseline_value_columns
    ].rename(columns=baseline_rename)
    result = result.merge(
        baseline_frame,
        on=list(group_columns),
        how='left',
        validate='many_to_one',
    )

    original_columns = [
        column for column in frame.columns
        if column not in legacy_columns
    ]
    expanded_columns: List[str] = []
    spec_by_value = {spec[0]: spec for spec in metric_specs}

    for value_column, baseline_column, difference_column, percent_column, text_column, higher_is_better in metric_specs:
        if value_column not in result.columns:
            raise KeyError(f'Missing metric column: {value_column}')
        if result[baseline_column].isna().any():
            raise RuntimeError(
                f'Missing {baseline_label} value for metric {value_column}.'
            )
        result[difference_column] = (
            result[value_column].astype(float)
            - result[baseline_column].astype(float)
        )
        result[percent_column] = [
            comparison_improvement_percent(
                value,
                baseline,
                higher_is_better=higher_is_better,
            )
            for value, baseline in zip(
                result[value_column],
                result[baseline_column],
            )
        ]
        result[text_column] = [
            comparison_result_text(
                percent,
                is_baseline=(algorithm == baseline_label),
            )
            for algorithm, percent in zip(
                result[algorithm_column],
                result[percent_column],
            )
        ]

    for column in original_columns:
        if column in spec_by_value:
            value_column, baseline_column, difference_column, percent_column, text_column, _ = spec_by_value[column]
            expanded_columns.extend([
                value_column,
                baseline_column,
                difference_column,
                percent_column,
                text_column,
            ])
        else:
            expanded_columns.append(column)

    # Keep any additional columns that were not present in the original frame.
    expanded_columns.extend([
        column for column in result.columns
        if column not in expanded_columns
    ])
    return result.loc[:, expanded_columns]

# =============================================================================
# SHARED PARETO OUTLIER-ZONE IMPLEMENTATION
# One implementation is used by Figures 4, 6, 7, 8, and 9.
# =============================================================================
PV4NEW_OUTLIER_ZONE_ENABLED = True
PV4NEW_OUTLIER_BLANK_GAP_FRACTION = 0.0
PV4NEW_OUTLIER_LABEL_GAP_PIXELS = 1.5
PV4NEW_OUTLIER_AUTO_ROTATE_X_LABELS = True
PV4NEW_OUTLIER_MIN_LABEL_FONTSIZE = 6.0
PV4NEW_OUTLIER_MIN_UNIQUE_VALUES = 3
PV4NEW_OUTLIER_MIN_MAIN_VALUES = 2
PV4NEW_OUTLIER_MAX_FRACTION = 0.40
PV4NEW_OUTLIER_MIN_NORMAL_MAJOR_TICKS = 3
PV4NEW_OUTLIER_ZONE_AXIS_COLOR = '#333333'
PV4NEW_OUTLIER_ZONE_AXIS_LINEWIDTH = 2.0
PV4NEW_OUTLIER_ORDINARY_SPINE_LINEWIDTH = 0.8

# Accessible, low-contrast background for compressed outlier regions.
# The light neutral blue-gray remains distinguishable in grayscale and does
# not compete visually with Pareto/non-Pareto marker colors.
PV4NEW_OUTLIER_ZONE_FACE_COLOR = '#D9DEE3'
PV4NEW_OUTLIER_ZONE_FACE_ALPHA = 1.0
PV4NEW_OUTLIER_ZONE_FACE_ZORDER = 0.25

def PV4NEW_clean_preserved_tick_label(label: str) -> str:
    """Remove only redundant decimal zeroes from an existing tick label."""
    text = str(label)

    # Matplotlib may wrap scalar tick labels in mathtext.
    mathtext_prefix = '$\\mathdefault{'
    mathtext_suffix = '}$'
    if text.startswith(mathtext_prefix) and text.endswith(mathtext_suffix):
        inner = text[len(mathtext_prefix):-len(mathtext_suffix)]
        return (
            mathtext_prefix
            + PV4NEW_clean_preserved_tick_label(inner)
            + mathtext_suffix
        )

    stripped = text.strip()
    leading = text[:len(text) - len(text.lstrip())]
    trailing = text[len(text.rstrip()):]

    # Preserve a scientific-notation exponent while cleaning its mantissa.
    exponent_index = -1
    for marker in ('e', 'E'):
        candidate = stripped.find(marker, 1)
        if candidate >= 0:
            exponent_index = candidate
            break

    if exponent_index >= 0:
        mantissa = stripped[:exponent_index]
        exponent = stripped[exponent_index:]
    else:
        mantissa = stripped
        exponent = ''

    ascii_mantissa = mantissa.replace('−', '-')
    unsigned = ascii_mantissa.lstrip('+-')

    if unsigned.count('.') == 1 and unsigned.replace('.', '', 1).isdigit():
        cleaned = ascii_mantissa.rstrip('0').rstrip('.')
        if cleaned in {'-0', '+0', ''}:
            cleaned = '0'
        if mantissa.startswith('−') and cleaned.startswith('-'):
            cleaned = '−' + cleaned[1:]
        return leading + cleaned + exponent + trailing

    return text

def PV4NEW_clean_outlier_tick_label(value: float) -> str:
    """Format an outlier with at most two decimals and no trailing zeroes."""
    numeric_value = float(value)
    rounded_value = round(numeric_value, 2)
    tolerance = 1e-12 * max(1.0, abs(numeric_value))

    if abs(rounded_value) <= tolerance:
        rounded_value = 0.0

    return f'{rounded_value:.2f}'.rstrip('0').rstrip('.')

class PV4NEW_PreservedTickFormatter(Formatter):
    """Keep the original normal-region labels and cleanly label outliers."""

    def __init__(
        self,
        tick_locations: Sequence[float],
        tick_labels: Sequence[str],
        offset_text: str = '',
    ) -> None:
        self._tick_locations = np.asarray(tick_locations, dtype=float)
        self._tick_labels = list(tick_labels)
        self._offset_text = offset_text

        if len(self._tick_locations) != len(self._tick_labels):
            raise ValueError(
                'tick_locations and tick_labels must have the same length.'
            )

    def __call__(self, value, position=None):
        if len(self._tick_locations) == 0:
            return ''

        distances = np.abs(self._tick_locations - float(value))
        nearest_index = int(np.argmin(distances))
        nearest_value = float(self._tick_locations[nearest_index])
        tolerance = 1e-9 * max(1.0, abs(float(value)), abs(nearest_value))

        if distances[nearest_index] <= tolerance:
            return self._tick_labels[nearest_index]
        return ''

    def get_offset(self):
        return self._offset_text

def PV4NEW_validate_outlier_selection_parameters(
    *,
    visible_fraction_threshold: float,
    margin_fraction: float,
    max_outlier_fraction: float,
) -> None:
    if not 0.0 < float(visible_fraction_threshold) < 1.0:
        raise ValueError(
            'visible_fraction_threshold must be between 0 and 1.'
        )
    if float(margin_fraction) < 0.0:
        raise ValueError('margin_fraction must be non-negative.')
    if not 0.0 < float(max_outlier_fraction) < 1.0:
        raise ValueError('max_outlier_fraction must be between 0 and 1.')


def PV4NEW_find_high_outlier_candidate(
    ordered_values: Sequence[float],
    *,
    visible_fraction_threshold: float,
    margin_fraction: float,
    max_outlier_fraction: float = PV4NEW_OUTLIER_MAX_FRACTION,
) -> Tuple[int, float] | None:
    """Return ``(start_index, visible_fraction)`` for high outliers."""
    values = np.asarray(ordered_values, dtype=float)
    PV4NEW_validate_outlier_selection_parameters(
        visible_fraction_threshold=visible_fraction_threshold,
        margin_fraction=margin_fraction,
        max_outlier_fraction=max_outlier_fraction,
    )

    if len(values) < PV4NEW_OUTLIER_MIN_UNIQUE_VALUES:
        return None

    data_min = float(values[0])
    total_count = len(values)

    # Ignore the first two unique values, then test candidates low-to-high.
    for candidate_index in range(PV4NEW_OUTLIER_MIN_MAIN_VALUES, total_count):
        candidate = float(values[candidate_index])
        previous_max = float(values[candidate_index - 1])
        candidate_span = candidate - data_min
        if candidate_span <= 0.0:
            continue

        outlier_count = total_count - candidate_index
        if outlier_count / total_count > float(max_outlier_fraction):
            continue

        padding = float(margin_fraction) * candidate_span
        prospective_lower = data_min - padding
        prospective_upper = candidate + padding
        prospective_span = prospective_upper - prospective_lower
        if prospective_span <= 0.0:
            continue

        previous_visible_fraction = (
            previous_max - prospective_lower
        ) / prospective_span

        if previous_visible_fraction <= float(visible_fraction_threshold):
            return candidate_index, float(previous_visible_fraction)

    return None


def PV4NEW_find_low_outlier_candidate(
    ordered_values: Sequence[float],
    *,
    visible_fraction_threshold: float,
    margin_fraction: float,
    max_outlier_fraction: float = PV4NEW_OUTLIER_MAX_FRACTION,
) -> Tuple[int, float] | None:
    """Return ``(end_index, visible_fraction)`` for low outliers.

    ``end_index`` is exclusive. The calculation mirrors the high-side test:
    candidates are checked from high to low, and a low value starts the
    outlier tier when adding it would compress all higher values into no more
    than the configured fraction of the prospective padded axis.
    """
    values = np.asarray(ordered_values, dtype=float)
    PV4NEW_validate_outlier_selection_parameters(
        visible_fraction_threshold=visible_fraction_threshold,
        margin_fraction=margin_fraction,
        max_outlier_fraction=max_outlier_fraction,
    )

    if len(values) < PV4NEW_OUTLIER_MIN_UNIQUE_VALUES:
        return None

    data_max = float(values[-1])
    total_count = len(values)
    first_candidate_index = total_count - PV4NEW_OUTLIER_MIN_MAIN_VALUES - 1

    # Ignore the final two unique values, then test candidates high-to-low.
    for candidate_index in range(first_candidate_index, -1, -1):
        candidate = float(values[candidate_index])
        next_min = float(values[candidate_index + 1])
        candidate_span = data_max - candidate
        if candidate_span <= 0.0:
            continue

        outlier_count = candidate_index + 1
        if outlier_count / total_count > float(max_outlier_fraction):
            continue

        padding = float(margin_fraction) * candidate_span
        prospective_lower = candidate - padding
        prospective_upper = data_max + padding
        prospective_span = prospective_upper - prospective_lower
        if prospective_span <= 0.0:
            continue

        higher_visible_fraction = (
            prospective_upper - next_min
        ) / prospective_span

        if higher_visible_fraction <= float(visible_fraction_threshold):
            return candidate_index + 1, float(higher_visible_fraction)

    return None


def PV4NEW_find_outlier_bounds(
    ordered_values: Sequence[float],
    *,
    visible_fraction_threshold: float,
    margin_fraction: float,
    max_outlier_fraction: float = PV4NEW_OUTLIER_MAX_FRACTION,
) -> Tuple[int, int]:
    """Return ``(main_start, main_end)`` after symmetric outlier detection."""
    values = np.asarray(ordered_values, dtype=float)
    total_count = len(values)

    low_candidate = PV4NEW_find_low_outlier_candidate(
        values,
        visible_fraction_threshold=visible_fraction_threshold,
        margin_fraction=margin_fraction,
        max_outlier_fraction=max_outlier_fraction,
    )
    high_candidate = PV4NEW_find_high_outlier_candidate(
        values,
        visible_fraction_threshold=visible_fraction_threshold,
        margin_fraction=margin_fraction,
        max_outlier_fraction=max_outlier_fraction,
    )

    main_start = 0 if low_candidate is None else low_candidate[0]
    main_end = total_count if high_candidate is None else high_candidate[0]

    low_count = main_start
    high_count = total_count - main_end
    combined_outlier_fraction = (
        (low_count + high_count) / total_count
        if total_count
        else 0.0
    )

    # If independently valid tails would together leave too little main data
    # or exceed the existing total outlier-fraction safeguard, keep only the
    # more visually severe side (the smaller visible fraction).
    if (
        low_candidate is not None
        and high_candidate is not None
        and (
            main_end - main_start < PV4NEW_OUTLIER_MIN_MAIN_VALUES
            or combined_outlier_fraction > float(max_outlier_fraction)
        )
    ):
        low_strength = low_candidate[1]
        high_strength = high_candidate[1]
        if low_strength <= high_strength:
            main_end = total_count
        else:
            main_start = 0

    return main_start, main_end


def PV4NEW_shade_outlier_regions(
    ax: plt.Axes,
    *,
    axis: str,
    normal_start: float,
    normal_end: float,
    has_low_outliers: bool,
    has_high_outliers: bool,
) -> None:
    """Shade compressed outlier regions using axes coordinates.

    A solid, very light background is used so overlapping x- and y-outlier
    regions do not become darker. The patch is drawn behind the grid, Pareto
    line, and data markers.
    """
    if axis not in {'x', 'y'}:
        raise ValueError("axis must be either 'x' or 'y'.")

    regions: List[Tuple[float, float, float, float]] = []

    if axis == 'x':
        if has_low_outliers and normal_start > 0.0:
            regions.append((0.0, 0.0, normal_start, 1.0))
        if has_high_outliers and normal_end < 1.0:
            regions.append((
                normal_end,
                0.0,
                1.0 - normal_end,
                1.0,
            ))
    else:
        if has_low_outliers and normal_start > 0.0:
            regions.append((0.0, 0.0, 1.0, normal_start))
        if has_high_outliers and normal_end < 1.0:
            regions.append((
                0.0,
                normal_end,
                1.0,
                1.0 - normal_end,
            ))

    for x, y, width, height in regions:
        ax.add_patch(
            Rectangle(
                (x, y),
                width,
                height,
                transform=ax.transAxes,
                facecolor=PV4NEW_OUTLIER_ZONE_FACE_COLOR,
                edgecolor='none',
                linewidth=0.0,
                alpha=PV4NEW_OUTLIER_ZONE_FACE_ALPHA,
                zorder=PV4NEW_OUTLIER_ZONE_FACE_ZORDER,
                clip_on=True,
            )
        )


def PV4NEW_apply_pareto_high_outlier_zone(
    ax: plt.Axes,
    values: Sequence[float],
    *,
    axis: str,
    visible_fraction_threshold: float,
    outlier_region_fraction: float,
    margin_fraction: float,
    max_outlier_fraction: float = PV4NEW_OUTLIER_MAX_FRACTION,
) -> bool:
    """Apply the margin-aware sequential outlier zone on either axis tail.

    The function name is retained so all existing figure calls remain
    untouched. Detection is now symmetric: unusually low values, unusually
    high values, or both can be assigned to compressed outlier zones. The
    normal-region tick-generation and formatting path is unchanged.
    """
    if axis not in {'x', 'y'}:
        raise ValueError("axis must be either 'x' or 'y'.")

    if not PV4NEW_OUTLIER_ZONE_ENABLED:
        return False

    finite_values = np.asarray(
        [float(value) for value in values if np.isfinite(float(value))],
        dtype=float,
    )
    unique_values = np.unique(finite_values)

    if len(unique_values) < PV4NEW_OUTLIER_MIN_UNIQUE_VALUES:
        return False

    main_start, main_end = PV4NEW_find_outlier_bounds(
        unique_values,
        visible_fraction_threshold=visible_fraction_threshold,
        margin_fraction=margin_fraction,
        max_outlier_fraction=max_outlier_fraction,
    )
    if main_start == 0 and main_end == len(unique_values):
        return False

    low_outlier_values = unique_values[:main_start]
    main_values = unique_values[main_start:main_end]
    high_outlier_values = unique_values[main_end:]
    outlier_values = np.concatenate((
        low_outlier_values,
        high_outlier_values,
    ))

    if len(main_values) < PV4NEW_OUTLIER_MIN_MAIN_VALUES:
        return False

    has_low_outliers = len(low_outlier_values) > 0
    has_high_outliers = len(high_outlier_values) > 0

    main_min = float(main_values[0])
    main_max = float(main_values[-1])
    main_span = max(main_max - main_min, np.finfo(float).eps)

    # The detection threshold and displayed outlier-region width are
    # independent controls. The region never expands beyond this value.
    reserved_fraction = float(outlier_region_fraction)
    blank_fraction = float(PV4NEW_OUTLIER_BLANK_GAP_FRACTION)
    label_gap_pixels = float(PV4NEW_OUTLIER_LABEL_GAP_PIXELS)

    if not 0.0 < reserved_fraction < 1.0:
        raise ValueError(
            'outlier_region_fraction must be between 0 and 1.'
        )
    if blank_fraction != 0.0:
        raise ValueError(
            'PV4NEW_OUTLIER_BLANK_GAP_FRACTION must be 0.0 because the '
            'blank outlier interval has been removed.'
        )
    if label_gap_pixels < 0.0:
        raise ValueError(
            'PV4NEW_OUTLIER_LABEL_GAP_PIXELS must be non-negative.'
        )

    margin_fraction = float(margin_fraction)
    if margin_fraction < 0.0:
        raise ValueError('margin_fraction must be non-negative.')
    padding = margin_fraction * main_span
    main_lower_limit = main_min - padding
    main_upper_limit = main_max + padding

    positive_main_gaps = np.diff(main_values)
    positive_main_gaps = positive_main_gaps[positive_main_gaps > 0.0]
    reference_gap = (
        float(np.median(positive_main_gaps))
        if len(positive_main_gaps)
        else main_span
    )

    if has_low_outliers:
        lower_padding = max(
            reference_gap,
            np.finfo(float).eps
            * max(1.0, abs(float(low_outlier_values[0]))),
        )
        lower_limit = float(low_outlier_values[0]) - lower_padding
        if main_lower_limit <= float(low_outlier_values[-1]):
            main_lower_limit = (
                float(low_outlier_values[-1]) + main_min
            ) / 2.0
    else:
        lower_limit = main_lower_limit

    if has_high_outliers:
        upper_padding = max(
            reference_gap,
            np.finfo(float).eps
            * max(1.0, abs(float(high_outlier_values[-1]))),
        )
        upper_limit = float(high_outlier_values[-1]) + upper_padding
        if main_upper_limit >= float(high_outlier_values[0]):
            main_upper_limit = (
                main_max + float(high_outlier_values[0])
            ) / 2.0
    else:
        upper_limit = main_upper_limit

    # Run the current locator/formatter on the ordinary range first. If that
    # full-range locator yields fewer than three visible ordinary ticks, replace
    # only the ordinary-region locator with a fresh MaxNLocator.
    if axis == 'x':
        ax.set_xlim(main_lower_limit, main_upper_limit)
        matplotlib_axis = ax.xaxis
    else:
        ax.set_ylim(main_lower_limit, main_upper_limit)
        matplotlib_axis = ax.yaxis

    tolerance = 1e-10 * max(
        1.0,
        abs(main_lower_limit),
        abs(main_upper_limit),
    )

    def PV4NEW_capture_normal_ticks():
        ax.figure.canvas.draw()

        generated_ticks = np.asarray(
            matplotlib_axis.get_majorticklocs(),
            dtype=float,
        )
        generated_tick_objects = matplotlib_axis.get_major_ticks(
            len(generated_ticks)
        )
        generated_labels = [
            PV4NEW_clean_preserved_tick_label(tick.label1.get_text())
            for tick in generated_tick_objects
        ]
        generated_minor = np.asarray(
            matplotlib_axis.get_minorticklocs(),
            dtype=float,
        )
        offset_text = matplotlib_axis.get_offset_text().get_text()

        major_ticks_local: List[float] = []
        major_labels_local: List[str] = []
        minor_ticks_local: List[float] = []

        for tick, label in zip(generated_ticks, generated_labels):
            if not np.isfinite(tick):
                continue
            if (
                main_lower_limit - tolerance
                <= float(tick)
                <= main_upper_limit + tolerance
            ):
                major_ticks_local.append(float(tick))
                major_labels_local.append(label)

        for tick in generated_minor:
            if not np.isfinite(tick):
                continue
            if (
                main_lower_limit - tolerance
                <= float(tick)
                <= main_upper_limit + tolerance
            ):
                minor_ticks_local.append(float(tick))

        return (
            major_ticks_local,
            major_labels_local,
            minor_ticks_local,
            generated_tick_objects,
            offset_text,
        )

    (
        captured_main_ticks,
        captured_main_labels,
        captured_minor_ticks,
        generated_main_tick_objects,
        normal_offset_text,
    ) = PV4NEW_capture_normal_ticks()

    if len(captured_main_ticks) < PV4NEW_OUTLIER_MIN_NORMAL_MAJOR_TICKS:
        matplotlib_axis.set_major_locator(
            matplotlib.ticker.MaxNLocator(
                nbins=3,
                min_n_ticks=PV4NEW_OUTLIER_MIN_NORMAL_MAJOR_TICKS,
                steps=[1, 2, 2.5, 5, 10],
            )
        )
        (
            captured_main_ticks,
            captured_main_labels,
            captured_minor_ticks,
            generated_main_tick_objects,
            normal_offset_text,
        ) = PV4NEW_capture_normal_ticks()

    # Extremely defensive final fallback for unusual custom formatters/locators.
    if len(captured_main_ticks) < PV4NEW_OUTLIER_MIN_NORMAL_MAJOR_TICKS:
        fallback_ticks = np.linspace(
            main_lower_limit,
            main_upper_limit,
            PV4NEW_OUTLIER_MIN_NORMAL_MAJOR_TICKS,
        )
        formatter = matplotlib_axis.get_major_formatter()
        fallback_labels = [
            PV4NEW_clean_preserved_tick_label(formatter(float(tick), index))
            for index, tick in enumerate(fallback_ticks)
        ]
        captured_main_ticks = [float(tick) for tick in fallback_ticks]
        captured_main_labels = [
            label if label else PV4NEW_clean_outlier_tick_label(tick)
            for tick, label in zip(fallback_ticks, fallback_labels)
        ]

    low_outlier_labels = [
        PV4NEW_clean_outlier_tick_label(float(value))
        for value in low_outlier_values
    ]
    high_outlier_labels = [
        PV4NEW_clean_outlier_tick_label(float(value))
        for value in high_outlier_values
    ]

    combined_entries = [
        (float(value), label, True)
        for value, label in zip(low_outlier_values, low_outlier_labels)
    ]
    combined_entries.extend(
        (tick, label, False)
        for tick, label in zip(captured_main_ticks, captured_main_labels)
    )
    combined_entries.extend(
        (float(value), label, True)
        for value, label in zip(high_outlier_values, high_outlier_labels)
    )
    combined_entries.sort(key=lambda entry: entry[0])

    major_ticks = [entry[0] for entry in combined_entries]
    major_labels = [entry[1] for entry in combined_entries]
    outlier_tick_indices = {
        index
        for index, entry in enumerate(combined_entries)
        if entry[2]
    }
    normal_tick_indices = set(range(len(major_ticks))) - outlier_tick_indices

    source_parts: List[np.ndarray] = []
    if has_low_outliers:
        source_parts.extend((
            np.asarray([lower_limit], dtype=float),
            np.asarray(low_outlier_values, dtype=float),
            np.asarray([main_lower_limit], dtype=float),
        ))
    else:
        source_parts.append(np.asarray([main_lower_limit], dtype=float))

    source_parts.append(np.asarray([main_upper_limit], dtype=float))

    if has_high_outliers:
        source_parts.extend((
            np.asarray(high_outlier_values, dtype=float),
            np.asarray([upper_limit], dtype=float),
        ))

    source_knots = np.concatenate(source_parts)

    def interpolate_with_extrapolation(interpolation_values, source, target):
        array = np.asarray(interpolation_values, dtype=float)
        flat = array.reshape(-1)
        transformed = np.interp(flat, source, target)

        left_mask = flat < source[0]
        if np.any(left_mask):
            left_slope = (target[1] - target[0]) / (source[1] - source[0])
            transformed[left_mask] = (
                target[0] + (flat[left_mask] - source[0]) * left_slope
            )

        right_mask = flat > source[-1]
        if np.any(right_mask):
            right_slope = (
                (target[-1] - target[-2]) / (source[-1] - source[-2])
            )
            transformed[right_mask] = (
                target[-1] + (flat[right_mask] - source[-1]) * right_slope
            )

        return transformed.reshape(array.shape)

    base_tick_pad = next(
        (
            float(tick.get_pad())
            for tick in generated_main_tick_objects
            if tick.label1.get_text()
        ),
        float(
            plt.rcParams.get(
                'xtick.major.pad' if axis == 'x' else 'ytick.major.pad',
                3.5,
            )
        ),
    )

    def apply_outlier_layout(
        reserved_fraction: float,
        outlier_rotation: float,
        outlier_font_size: float | None,
    ) -> Tuple[float, float, List[object]]:
        total_outlier_count = len(outlier_values)
        low_reserved_fraction = (
            reserved_fraction * len(low_outlier_values) / total_outlier_count
        )
        high_reserved_fraction = (
            reserved_fraction * len(high_outlier_values) / total_outlier_count
        )
        normal_start = low_reserved_fraction
        normal_end = 1.0 - high_reserved_fraction

        display_parts: List[np.ndarray] = []
        if has_low_outliers:
            low_positions = (
                low_reserved_fraction
                * (np.arange(len(low_outlier_values), dtype=float) + 0.5)
                / len(low_outlier_values)
            )
            display_parts.extend((
                np.asarray([0.0], dtype=float),
                low_positions,
                np.asarray([normal_start], dtype=float),
            ))
        else:
            display_parts.append(np.asarray([normal_start], dtype=float))

        display_parts.append(np.asarray([normal_end], dtype=float))

        if has_high_outliers:
            high_positions = (
                normal_end
                + high_reserved_fraction
                * (np.arange(len(high_outlier_values), dtype=float) + 0.5)
                / len(high_outlier_values)
            )
            display_parts.extend((
                high_positions,
                np.asarray([1.0], dtype=float),
            ))

        display_knots = np.concatenate(display_parts)

        def forward(axis_values):
            return interpolate_with_extrapolation(
                axis_values,
                source_knots,
                display_knots,
            )

        def inverse(display_values):
            return interpolate_with_extrapolation(
                display_values,
                display_knots,
                source_knots,
            )

        if axis == 'x':
            ax.set_xscale('function', functions=(forward, inverse))
            ax.set_xlim(lower_limit, upper_limit)
            current_axis = ax.xaxis
        else:
            ax.set_yscale('function', functions=(forward, inverse))
            ax.set_ylim(lower_limit, upper_limit)
            current_axis = ax.yaxis

        current_axis.set_major_locator(FixedLocator(major_ticks))
        current_axis.set_major_formatter(
            PV4NEW_PreservedTickFormatter(
                major_ticks,
                major_labels,
                normal_offset_text,
            )
        )
        current_axis.set_minor_locator(FixedLocator(captured_minor_ticks))

        ax.figure.canvas.draw()
        tick_objects = current_axis.get_major_ticks(len(major_ticks))

        for index, tick_object in enumerate(tick_objects):
            if index not in outlier_tick_indices:
                continue

            if outlier_font_size is not None:
                tick_object.label1.set_fontsize(outlier_font_size)
            tick_object.label1.set_rotation(outlier_rotation)

            if axis == 'x' and outlier_rotation != 0.0:
                tick_object.label1.set_rotation_mode('anchor')
                tick_object.label1.set_horizontalalignment('right')
                tick_object.label1.set_verticalalignment('center')
                tick_object.set_pad(base_tick_pad)
            elif axis == 'x':
                tick_object.label1.set_rotation_mode('default')
                tick_object.label1.set_horizontalalignment('center')
                tick_object.label1.set_verticalalignment('top')
                tick_object.set_pad(base_tick_pad)

        ax.figure.canvas.draw()
        return normal_start, normal_end, tick_objects

    def labels_overlap(
        tick_objects: Sequence[object],
        *,
        include_normal_pairs: bool,
    ) -> bool:
        renderer = ax.figure.canvas.get_renderer()
        label_entries = []

        for index, tick_object in enumerate(tick_objects):
            label = tick_object.label1
            if not label.get_visible() or not label.get_text():
                continue
            label_entries.append((
                index,
                label.get_window_extent(renderer=renderer),
            ))

        for left_position, (left_index, left_bbox) in enumerate(label_entries):
            for right_index, right_bbox in label_entries[left_position + 1:]:
                pair_has_outlier = (
                    left_index in outlier_tick_indices
                    or right_index in outlier_tick_indices
                )
                if not pair_has_outlier:
                    if not include_normal_pairs:
                        continue
                    if (
                        left_index not in normal_tick_indices
                        or right_index not in normal_tick_indices
                    ):
                        continue

                separated = (
                    left_bbox.x1 + label_gap_pixels <= right_bbox.x0
                    or right_bbox.x1 + label_gap_pixels <= left_bbox.x0
                    or left_bbox.y1 + label_gap_pixels <= right_bbox.y0
                    or right_bbox.y1 + label_gap_pixels <= left_bbox.y0
                )
                if not separated:
                    return True

        return False

    def normal_labels_overlap(tick_objects: Sequence[object]) -> bool:
        renderer = ax.figure.canvas.get_renderer()
        normal_bboxes = []

        for index, tick_object in enumerate(tick_objects):
            if index not in normal_tick_indices:
                continue
            label = tick_object.label1
            if not label.get_visible() or not label.get_text():
                continue
            normal_bboxes.append(
                label.get_window_extent(renderer=renderer)
            )

        for left_position, left_bbox in enumerate(normal_bboxes):
            for right_bbox in normal_bboxes[left_position + 1:]:
                separated = (
                    left_bbox.x1 + label_gap_pixels <= right_bbox.x0
                    or right_bbox.x1 + label_gap_pixels <= left_bbox.x0
                    or left_bbox.y1 + label_gap_pixels <= right_bbox.y0
                    or right_bbox.y1 + label_gap_pixels <= left_bbox.y0
                )
                if not separated:
                    return True

        return False

    outlier_rotation = 0.0
    outlier_font_size: float | None = None
    minimum_outlier_font_size: float | None = None
    normal_start, normal_end, major_tick_objects = apply_outlier_layout(
        reserved_fraction,
        outlier_rotation,
        outlier_font_size,
    )

    if (
        axis == 'x'
        and PV4NEW_OUTLIER_AUTO_ROTATE_X_LABELS
        and labels_overlap(
            major_tick_objects,
            include_normal_pairs=False,
        )
    ):
        outlier_rotation = 90.0
        normal_start, normal_end, major_tick_objects = apply_outlier_layout(
            reserved_fraction,
            outlier_rotation,
            outlier_font_size,
        )

    # The configured region is fixed. After x-label rotation, reduce only
    # the outlier-label font size if rendered labels still overlap.
    while labels_overlap(
        major_tick_objects,
        include_normal_pairs=False,
    ):
        if outlier_font_size is None:
            outlier_font_size = next(
                (
                    float(tick_object.label1.get_fontsize())
                    for index, tick_object in enumerate(major_tick_objects)
                    if index in outlier_tick_indices
                    and tick_object.label1.get_visible()
                    and tick_object.label1.get_text()
                ),
                float(PV4NEW_OUTLIER_MIN_LABEL_FONTSIZE),
            )
            minimum_outlier_font_size = min(
                outlier_font_size,
                float(PV4NEW_OUTLIER_MIN_LABEL_FONTSIZE),
            )

        if (
            minimum_outlier_font_size is None
            or outlier_font_size <= minimum_outlier_font_size
        ):
            break

        outlier_font_size = max(
            minimum_outlier_font_size,
            outlier_font_size - 0.5,
        )
        normal_start, normal_end, major_tick_objects = apply_outlier_layout(
            reserved_fraction,
            outlier_rotation,
            outlier_font_size,
        )

    PV4NEW_shade_outlier_regions(
        ax,
        axis=axis,
        normal_start=normal_start,
        normal_end=normal_end,
        has_low_outliers=has_low_outliers,
        has_high_outliers=has_high_outliers,
    )

    if axis == 'x':
        ax.grid(False, axis='x', which='minor')
        if has_low_outliers:
            ax.plot(
                [0.0, normal_start],
                [0.0, 0.0],
                transform=ax.transAxes,
                color=PV4NEW_OUTLIER_ZONE_AXIS_COLOR,
                linewidth=PV4NEW_OUTLIER_ZONE_AXIS_LINEWIDTH,
                solid_capstyle='butt',
                clip_on=False,
                zorder=10,
            )
        if has_high_outliers:
            ax.plot(
                [normal_end, 1.0],
                [0.0, 0.0],
                transform=ax.transAxes,
                color=PV4NEW_OUTLIER_ZONE_AXIS_COLOR,
                linewidth=PV4NEW_OUTLIER_ZONE_AXIS_LINEWIDTH,
                solid_capstyle='butt',
                clip_on=False,
                zorder=10,
            )
    else:
        ax.grid(False, axis='y', which='minor')
        if has_low_outliers:
            ax.plot(
                [0.0, 0.0],
                [0.0, normal_start],
                transform=ax.transAxes,
                color=PV4NEW_OUTLIER_ZONE_AXIS_COLOR,
                linewidth=PV4NEW_OUTLIER_ZONE_AXIS_LINEWIDTH,
                solid_capstyle='butt',
                clip_on=False,
                zorder=10,
            )
        if has_high_outliers:
            ax.plot(
                [0.0, 0.0],
                [normal_end, 1.0],
                transform=ax.transAxes,
                color=PV4NEW_OUTLIER_ZONE_AXIS_COLOR,
                linewidth=PV4NEW_OUTLIER_ZONE_AXIS_LINEWIDTH,
                solid_capstyle='butt',
                clip_on=False,
                zorder=10,
            )

    ordinary_spine_width = PV4NEW_OUTLIER_ORDINARY_SPINE_LINEWIDTH
    ordinary_spine_color = plt.rcParams.get('axes.edgecolor', 'black')

    for spine_name in ('top', 'right'):
        ax.spines[spine_name].set_linewidth(ordinary_spine_width)
        ax.spines[spine_name].set_edgecolor(ordinary_spine_color)
        ax.spines[spine_name].set_zorder(2.5)

    return True


## Figure 1

Grouped comparison.


In [ ]:
# Figure 1
# Run the common setup cell first. This cell contains Figure 1-specific code and settings.

from __future__ import annotations

HEURISTIC_IPM_ROOT = RESULT_SAVE_3_ROOT / 'Heuristic-IPM'

HEURISTIC_ORACLE_ROOT = HEURISTIC_IPM_ROOT / 'Experiment-64'

HEURISTIC_ROOT = RESULT_SAVE_3_ROOT / 'Heuristic-ICON'

JOULES_PER_MWH = 3600000000.0

SECONDS_PER_MINUTE = 60.0

ORACLE_LABEL = 'SNF Oracle'

STRUCTURAL_DIRS = {'generated_configs', 'metrics_comparison', 'plots', 'analysis_figures', 'publication_figures', '__pycache__'}

SWEEP_VARIANTS: Dict[str, Dict[str, str]] = {'snf-ssc': {'label': 'SNF-ICON', 'marker': 'o', 'color': '#377eb8'}, 'snf-ssc-nf': {'label': 'SNF-ICON-NF', 'marker': 's', 'color': '#e41a1c'}, 'snf-ssc-ng': {'label': 'SNF-ICON-NG', 'marker': '^', 'color': '#4daf4a'}, 'snf-ssc-ngnf': {'label': 'SNF-ICON-NGNF', 'marker': 'D', 'color': '#984ea3'}}

RESULT_SAVE_3_RL_RUN = 'EASY_PSUS_Budiarjo_w10.5_w20.5'

RESULT_SAVE_3_RL_LABEL = 'RL Budiarjo'

SOURCE_DIR_RECORDS: List[Dict[str, str]] = []

SOURCE_DIR_SEEN: set[Tuple[str, str]] = set()

def source_root_label(path: Path) -> str:
    resolved = path.expanduser().resolve()
    roots = (('Heuristic-ICON', HEURISTIC_ROOT), ('Heuristic-IPM', HEURISTIC_IPM_ROOT), ('RL-Budiarjo', RESULT_SAVE_3_ROOT / 'RL-Budiarjo'))
    for label, root in roots:
        try:
            resolved.relative_to(root.expanduser().resolve())
            return label
        except ValueError:
            continue
    return 'Other'

def platform_node_count(platform: str) -> int:
    match = re.search('-(\\d+)(?:-|$)', platform)
    if match is None:
        raise ValueError(f'Could not infer node count from platform name: {platform}')
    return int(match.group(1))

def extract_timeout(run_name: str) -> int | None:
    match = re.search('(?:^|_)timeout-(\\d+)(?:_|$)', run_name, flags=re.IGNORECASE)
    return int(match.group(1)) if match else None

def algorithm_label(run_name: str) -> str:
    lower = run_name.lower()
    normalized = lower.replace('_', '-')
    if lower.startswith('snf_oracle') or normalized.startswith('snf-oracle'):
        return ORACLE_LABEL
    if normalized.startswith('snf-ssc-nfng') or normalized.startswith('snf-ssc-ngnf'):
        return 'SNF-ICON-NGNF'
    if normalized.startswith('snf-ssc-ng'):
        return 'SNF-ICON-NG'
    if normalized.startswith('snf-ssc-nf'):
        return 'SNF-ICON-NF'
    if normalized.startswith('snf-ssc'):
        return 'SNF-ICON'
    if lower.startswith('snf_psas') or lower.startswith('snf_ipm'):
        return 'SNF+IPM'
    if lower.startswith('easy_psas') or lower.startswith('easy_ipm'):
        return 'FCFS/B+IPM'
    raise ValueError(f'Unknown run directory name: {run_name}')

def strict_numeric(series: pd.Series, column: str, path: Path) -> pd.Series:
    if series.isna().any():
        rows = series.index[series.isna()].tolist()
        raise ValueError(f'Null value in {path}, column {column!r}, rows {rows}')
    try:
        numeric = pd.to_numeric(series, errors='raise')
    except (TypeError, ValueError) as exc:
        raise ValueError(f'Non-numeric value in {path}, column {column!r}') from exc
    values = numeric.to_numpy(dtype=float)
    invalid = ~np.isfinite(values)
    if invalid.any():
        rows = numeric.index[invalid].tolist()
        raise ValueError(f'Non-finite value in {path}, column {column!r}, rows {rows}')
    return numeric.astype(float)

def read_pareto_metrics(metrics_path: Path) -> Tuple[float, float]:
    if not metrics_path.is_file():
        raise FileNotFoundError(f'Missing metrics file: {metrics_path}')
    register_source_file(metrics_path, 'pareto_metrics')
    frame = pd.read_csv(metrics_path)
    if len(frame) != 1:
        raise ValueError(f'{metrics_path} must contain exactly one data row; found {len(frame)}')
    required = ('mean_waiting_time', 'total_energy_waste')
    require_columns(frame, required, metrics_path)
    waiting_seconds = float(strict_numeric(frame['mean_waiting_time'], 'mean_waiting_time', metrics_path).iloc[0])
    wasted_joules = float(strict_numeric(frame['total_energy_waste'], 'total_energy_waste', metrics_path).iloc[0])
    if waiting_seconds < 0.0:
        raise ValueError(f'Negative mean_waiting_time in {metrics_path}')
    if wasted_joules < 0.0:
        raise ValueError(f'Negative total_energy_waste in {metrics_path}')
    return (waiting_seconds / SECONDS_PER_MINUTE, wasted_joules / JOULES_PER_MWH)

RESULT_SAVE_3_TITLE_CONFIG_KEYS: Tuple[str, ...] = (
    "run.algo_config.alpha",
    "run.algo_config.beta",
    "run.algo_config.markov_window_seconds",
)

def result_save_3_flatten_config(value: object, prefix: str = "") -> Dict[str, object]:
    flat: Dict[str, object] = {}
    if isinstance(value, dict):
        for key, child in value.items():
            child_key = f"{prefix}.{key}" if prefix else str(key)
            flat.update(result_save_3_flatten_config(child, child_key))
    else:
        flat[prefix] = value
    return flat

def result_save_3_title_config_label(
    records: Sequence["ResultSave3BarRecord"],
) -> str:
    """Read title values from each selected SNF-ICON run config."""
    result_dirs = sorted(
        {
            record.metrics_path.parent.resolve()
            for record in records
            if record.algorithm == "SNF-ICON"
        },
        key=str,
    )
    if not result_dirs:
        raise RuntimeError(
            'No SNF-ICON result directories were supplied for '
            'title configuration'
        )

    for result_dir in result_dirs:
        for key in RESULT_SAVE_3_TITLE_CONFIG_KEYS:
            config_numeric_value(result_dir, key)

    return config_values_title_label(
        result_dirs,
        RESULT_SAVE_3_TITLE_CONFIG_KEYS,
    )


def result_save_3_figure_title(
    records: Sequence["ResultSave3BarRecord"],
    *,
    timeout: int,
    platform_separator: str = " / ",
) -> str:
    if not records:
        raise ValueError('No records supplied for title generation')

    result_dirs = sorted(
        {
            record.metrics_path.parent.resolve()
            for record in records
        },
        key=str,
    )
    config_platforms = unique_config_path_stem_values(
        result_dirs,
        'paths.platform',
    )

    parsed_platforms: List[Tuple[str, int]] = []
    unparsed_platforms: List[str] = []

    for platform in config_platforms:
        match = re.fullmatch(
            r'(?P<family>.+)-(?P<size>\d+)',
            platform,
        )
        if match is None:
            unparsed_platforms.append(platform)
            continue
        parsed_platforms.append(
            (
                match.group('family'),
                int(match.group('size')),
            )
        )

    family_order: List[str] = []
    for family, _ in parsed_platforms:
        if family not in family_order:
            family_order.append(family)

    platform_label = (
        family_order[0]
        if len(family_order) == 1 and not unparsed_platforms
        else platform_separator.join(
            family_order + unparsed_platforms
        )
    )

    heuristic_result_dirs = sorted(
        {
            record.metrics_path.parent.resolve()
            for record in records
            if record.algorithm != RESULT_SAVE_3_RL_ALGORITHM_LABEL
        },
        key=str,
    )
    timeout_label = scheduler_timeout_title_label(
        heuristic_result_dirs
    )

    config_timeouts = {
        config_scheduler_timeout_seconds(result_dir)
        for result_dir in heuristic_result_dirs
    }
    if any(
        not math.isclose(
            value,
            float(timeout),
            rel_tol=1e-9,
            abs_tol=1e-9,
        )
        for value in config_timeouts
    ):
        raise RuntimeError(
            f'Configured timeout does not match requested timeout '
            f'{timeout:g}s: {sorted(config_timeouts)}'
        )

    parts = [platform_label, timeout_label]
    config_label = result_save_3_title_config_label(records)
    if config_label:
        parts.append(config_label)
    return " / ".join(parts)

RESULT_SAVE_3_ICON_DIRNAME = 'Heuristic-ICON'

RESULT_SAVE_3_IPM_DIRNAME = 'Heuristic-IPM'

RESULT_SAVE_3_RL_DIRNAME = 'RL-Budiarjo'

RESULT_SAVE_3_OUTPUT_DIR = OUTPUT_DIR / 'Figure1_Barplot'

RESULT_SAVE_3_SELECTED_TIMEOUT = 7200

RESULT_SAVE_3_RL_OPTIONAL_PLATFORM_SIZES: Tuple[int, ...] = (
    1152,
)

RESULT_SAVE_3_RL_ALGORITHM_LABEL = RESULT_SAVE_3_RL_LABEL

@dataclass(frozen=True)
class ResultSave3BarRecord:
    source_family: str
    platform: str
    dataset: str
    timeout: int | None
    algorithm: str
    waiting_seconds: float
    wasted_energy_mwh: float
    metrics_path: Path

def result_save_3_icon_algorithm_label(variant: str) -> str:
    """Convert an ICON experiment variant into a plot-friendly label."""
    normalized = variant.lower().replace('_', '-')
    if normalized == 'snf-ssc-nfng':
        normalized = 'snf-ssc-ngnf'
    if normalized in SWEEP_VARIANTS:
        return str(SWEEP_VARIANTS[normalized]['label'])
    try:
        return algorithm_label(normalized)
    except ValueError:
        return variant.replace('_', '-').upper()

def result_save_3_run_algorithm_label(run_name: str) -> str:
    """Infer a barplot label from an IPM run-directory name."""
    try:
        label = algorithm_label(run_name)
    except ValueError:
        without_timeout = re.sub('(?:^|_)timeout-\\d+(?:_|$)', '_', run_name, flags=re.IGNORECASE).strip('_-')
        return without_timeout.replace('_', '-')
    if label == 'FCFS/B+IPM':
        return 'FCFS/B+IPM'
    return label

def result_save_3_rl_required(platform: str) -> bool:
    return (
        platform_node_count(platform)
        not in RESULT_SAVE_3_RL_OPTIONAL_PLATFORM_SIZES
    )


def result_save_3_expected_config_algorithms(
    source_family: str,
    algorithm: str,
) -> Tuple[str, ...] | None:
    if source_family == RESULT_SAVE_3_ICON_DIRNAME:
        return ('snf_icon',)
    if source_family == RESULT_SAVE_3_IPM_DIRNAME:
        if algorithm == 'FCFS/B+IPM':
            return ('easy_psas', 'easy_ipm')
        if algorithm == 'SNF+IPM':
            return ('snf_psas', 'snf_ipm')
    return None


def result_save_3_add_record(selected: Dict[Tuple[str, int | None, str, str], ResultSave3BarRecord], *, source_family: str, platform: str, dataset: str, timeout: int | None, algorithm: str, metrics_path: Path) -> None:
    key = (platform, timeout, dataset, algorithm)
    if key in selected:
        previous = selected[key].metrics_path
        raise RuntimeError(f'Multiple results metrics files map to the same platform/timeout/dataset/algorithm tuple {key!r}:\n- {previous}\n- {metrics_path}')
    waiting_minutes, wasted_energy_mwh = read_pareto_metrics(metrics_path)
    waiting_seconds = waiting_minutes * SECONDS_PER_MINUTE
    selected[key] = ResultSave3BarRecord(source_family=source_family, platform=platform, dataset=dataset, timeout=timeout, algorithm=algorithm, waiting_seconds=waiting_seconds, wasted_energy_mwh=wasted_energy_mwh, metrics_path=metrics_path)

def result_save_3_collect_platform_runs(platform_dir: Path, *, source_family: str, algorithm_from_run: bool, fixed_algorithm: str | None, include_oracle: bool, selected_timeout: int, selected: Dict[Tuple[str, int | None, str, str], ResultSave3BarRecord]) -> None:
    if not platform_dir.is_dir():
        return
    platform = platform_dir.name
    for dataset_dir in sorted(platform_dir.iterdir(), key=lambda path: path.name.lower()):
        if not dataset_dir.is_dir():
            continue
        if dataset_dir.name.startswith('.') or dataset_dir.name in STRUCTURAL_DIRS:
            continue
        for run_dir in sorted(dataset_dir.iterdir(), key=lambda path: path.name.lower()):
            if not run_dir.is_dir():
                continue
            if run_dir.name.startswith('.') or run_dir.name in STRUCTURAL_DIRS:
                continue
            metrics_path = run_dir / 'metrics.csv'
            if not metrics_path.is_file():
                continue
            if algorithm_from_run:
                algorithm = result_save_3_run_algorithm_label(run_dir.name)
            elif fixed_algorithm is not None:
                algorithm = fixed_algorithm
            else:
                raise ValueError('A fixed algorithm label is required')
            if algorithm == ORACLE_LABEL and (not include_oracle):
                continue
            timeout = extract_timeout(run_dir.name)
            if algorithm == ORACLE_LABEL and timeout is None:
                pass
            elif timeout != selected_timeout:
                continue

            expected_algorithms = (
                result_save_3_expected_config_algorithms(
                    source_family,
                    algorithm,
                )
            )
            validate_result_config_identity(
                run_dir,
                expected_platform=platform,
                expected_workload=dataset_dir.name,
                expected_algorithms=expected_algorithms,
            )
            if algorithm != ORACLE_LABEL:
                validated_config_timeout_seconds(
                    run_dir,
                    selected_timeout,
                    'selected results timeout',
                )

            result_save_3_add_record(selected, source_family=source_family, platform=platform, dataset=dataset_dir.name, timeout=timeout, algorithm=algorithm, metrics_path=metrics_path)

def result_save_3_collect_rl_budiarjo_runs(root: Path, *, selected: Dict[Tuple[str, int | None, str, str], ResultSave3BarRecord]) -> None:
    """Collect curriculum-trained RL Budiarjo test results.

    Expected hierarchy:
        RL-Budiarjo/Budiarjo-curriculum-*/<platform>/<dataset>/
            easy_psus_Budiarjo_w10.5_w20.5

    RL Budiarjo does not use a scheduler timeout. Its records therefore use
    ``timeout=None`` and are matched to the selected 7200-second heuristic
    results only by platform and dataset.
    """
    rl_root = root / RESULT_SAVE_3_RL_DIRNAME
    if not rl_root.is_dir():
        raise FileNotFoundError(f'Missing RL-Budiarjo directory: {rl_root}')
    found = 0
    for curriculum_dir in sorted(rl_root.iterdir(), key=lambda path: path.name.lower()):
        if not curriculum_dir.is_dir():
            continue
        if not curriculum_dir.name.lower().startswith('budiarjo-curriculum-'):
            continue
        for platform_dir in sorted(curriculum_dir.iterdir(), key=lambda path: path.name.lower()):
            if not platform_dir.is_dir():
                continue
            if platform_dir.name.startswith('.') or platform_dir.name in STRUCTURAL_DIRS:
                continue
            for dataset_dir in sorted(platform_dir.iterdir(), key=lambda path: path.name.lower()):
                if not dataset_dir.is_dir():
                    continue
                if dataset_dir.name.startswith('.') or dataset_dir.name in STRUCTURAL_DIRS:
                    continue
                for run_dir in sorted(dataset_dir.iterdir(), key=lambda path: path.name.lower()):
                    if not run_dir.is_dir():
                        continue
                    if run_dir.name.lower() != RESULT_SAVE_3_RL_RUN.lower():
                        continue
                    metrics_path = run_dir / 'metrics.csv'
                    if not metrics_path.is_file():
                        continue

                    validate_result_config_identity(
                        run_dir,
                        expected_platform=platform_dir.name,
                        expected_workload=dataset_dir.name,
                    )

                    result_save_3_add_record(selected, source_family=RESULT_SAVE_3_RL_DIRNAME, platform=platform_dir.name, dataset=dataset_dir.name, timeout=None, algorithm=RESULT_SAVE_3_RL_ALGORITHM_LABEL, metrics_path=metrics_path)
                    found += 1
    if found == 0:
        raise FileNotFoundError(f'No RL-Budiarjo metrics.csv files were found below {rl_root}.')

def result_save_3_discover_barplot_records(
    root: Path = RESULT_SAVE_3_ROOT,
    *,
    include_oracle: bool,
    selected_timeout: int = RESULT_SAVE_3_SELECTED_TIMEOUT,
    include_icon_variants: bool = False,
    include_rl: bool = True,
) -> List[ResultSave3BarRecord]:
    """Read the selected-timeout ICON, IPM, and RL-Budiarjo results.

    Expected ICON hierarchy:
        Heuristic-ICON/<variant>_<platform-size>/<platform>/<dataset>/<run>

    Expected IPM hierarchy:
        Heuristic-IPM/Experiment-<platform-size>/<platform>/<dataset>/<run>

    Expected RL hierarchy:
        RL-Budiarjo/Budiarjo-curriculum-*/<platform>/<dataset>/<run>
    """
    root = root.expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f'results directory does not exist: {root}')
    selected: Dict[Tuple[str, int | None, str, str], ResultSave3BarRecord] = {}
    icon_root = root / RESULT_SAVE_3_ICON_DIRNAME
    if not icon_root.is_dir():
        raise FileNotFoundError(f'Missing Heuristic-ICON directory: {icon_root}')
    icon_experiment_pattern = re.compile('^(?P<variant>.+)_(?P<platform_size>\\d+)$', flags=re.IGNORECASE)
    for experiment_dir in sorted(icon_root.iterdir(), key=lambda path: path.name.lower()):
        if not experiment_dir.is_dir():
            continue
        match = icon_experiment_pattern.match(experiment_dir.name)
        if match is None:
            continue
        variant = match.group('variant')
        normalized_variant = variant.lower().replace('_', '-')
        if normalized_variant not in SWEEP_VARIANTS:
            continue
        if (
            not include_icon_variants
            and normalized_variant != 'snf-ssc'
        ):
            continue
        expected_platform_size = int(match.group('platform_size'))
        for platform_dir in sorted(experiment_dir.iterdir(), key=lambda path: path.name.lower()):
            if not platform_dir.is_dir():
                continue
            if platform_dir.name.startswith('.') or platform_dir.name in STRUCTURAL_DIRS:
                continue
            try:
                actual_platform_size = platform_node_count(platform_dir.name)
            except ValueError:
                print('Skipping ICON platform with an unrecognized name:', platform_dir)
                continue
            if actual_platform_size != expected_platform_size:
                print('Skipping ICON platform-size mismatch:', platform_dir)
                continue
            result_save_3_collect_platform_runs(platform_dir, source_family=RESULT_SAVE_3_ICON_DIRNAME, algorithm_from_run=False, fixed_algorithm=result_save_3_icon_algorithm_label(variant), include_oracle=include_oracle, selected_timeout=selected_timeout, selected=selected)
    ipm_root = root / RESULT_SAVE_3_IPM_DIRNAME
    if not ipm_root.is_dir():
        raise FileNotFoundError(f'Missing Heuristic-IPM directory: {ipm_root}')
    ipm_experiment_pattern = re.compile('^Experiment-(?P<platform_size>\\d+)$', flags=re.IGNORECASE)
    for experiment_dir in sorted(ipm_root.iterdir(), key=lambda path: path.name.lower()):
        if not experiment_dir.is_dir():
            continue
        match = ipm_experiment_pattern.match(experiment_dir.name)
        if match is None:
            continue
        expected_platform_size = int(match.group('platform_size'))
        for platform_dir in sorted(experiment_dir.iterdir(), key=lambda path: path.name.lower()):
            if not platform_dir.is_dir():
                continue
            if platform_dir.name.startswith('.') or platform_dir.name in STRUCTURAL_DIRS:
                continue
            try:
                actual_platform_size = platform_node_count(platform_dir.name)
            except ValueError:
                print('Skipping IPM platform with an unrecognized name:', platform_dir)
                continue
            if actual_platform_size != expected_platform_size:
                print('Skipping IPM platform-size mismatch:', platform_dir)
                continue
            result_save_3_collect_platform_runs(platform_dir, source_family=RESULT_SAVE_3_IPM_DIRNAME, algorithm_from_run=True, fixed_algorithm=None, include_oracle=include_oracle, selected_timeout=selected_timeout, selected=selected)
    if include_rl:
        result_save_3_collect_rl_budiarjo_runs(
            root,
            selected=selected,
        )
    if not selected:
        raise FileNotFoundError(
            f'No results metrics.csv files were found below '
            f'{icon_root} or {ipm_root}.'
        )
    return sorted(selected.values(), key=lambda record: (record.platform.lower(), record.timeout is None, float('inf') if record.timeout is None else record.timeout, record.dataset.lower(), record.algorithm.lower()))

def result_save_3_algorithm_order(records: Sequence[ResultSave3BarRecord]) -> List[str]:
    discovered = {record.algorithm for record in records}
    preferred_order = ('FCFS/B+IPM', 'SNF+IPM', 'SNF-ICON', RESULT_SAVE_3_RL_ALGORITHM_LABEL)
    preferred = [algorithm for algorithm in preferred_order if algorithm in discovered]
    extras = sorted(discovered.difference(preferred), key=str.lower)
    return preferred + extras

def result_save_3_style_map(algorithms: Sequence[str]) -> Dict[str, Tuple[str, str]]:
    """Assign PlotMetrics.py colors and hatches in algorithm order."""
    return {algorithm: ((('#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#999999'))[index % len((('#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#999999')))], (('///', '\\\\', 'xxx', '---', '+++', '...', '**', 'oo'))[index % len((('///', '\\\\', 'xxx', '---', '+++', '...', '**', 'oo')))]) for index, algorithm in enumerate(algorithms)}

def result_save_3_format_short_number(value: float) -> str:
    if not math.isfinite(value):
        return ''
    absolute_value = abs(value)
    if absolute_value >= 1000000000000:
        return f'{value / 1000000000000:.1f}T'
    if absolute_value >= 1000000000:
        return f'{value / 1000000000:.1f}B'
    if absolute_value >= 1000000:
        return f'{value / 1000000:.1f}M'
    if absolute_value >= 1000:
        return f'{value / 1000:.1f}K'
    if absolute_value >= 100:
        return f'{value:.0f}'
    if absolute_value >= 10:
        return f'{value:.1f}'
    if absolute_value >= 1:
        return f'{value:.2f}'
    if absolute_value == 0:
        return '0'
    return f'{value:.3g}'

def result_save_3_annotate_bars(ax: plt.Axes, bars: object) -> None:
    """Use the compact value labels from PlotMetrics.py."""

def result_save_3_target_dataset_keys(
    records: Sequence[ResultSave3BarRecord],
    *,
    heuristic_algorithms: Sequence[str],
    timeout: int,
    include_rl: bool = True,
) -> Tuple[
    set[Tuple[str, str]],
    set[Tuple[str, str]],
]:
    def is_figure1_platform(platform: str) -> bool:
        return platform.startswith('AOBA-')

    rl_dataset_keys = {
        (record.platform, record.dataset)
        for record in records
        if (
            is_figure1_platform(record.platform)
            and record.source_family == RESULT_SAVE_3_RL_DIRNAME
            and record.algorithm
            == RESULT_SAVE_3_RL_ALGORITHM_LABEL
        )
    }

    rl_optional_dataset_keys = {
        (record.platform, record.dataset)
        for record in records
        if (
            is_figure1_platform(record.platform)
            and not result_save_3_rl_required(record.platform)
            and record.algorithm in heuristic_algorithms
            and record.timeout == timeout
        )
    }

    if include_rl:
        target_dataset_keys = (
            rl_dataset_keys
            | rl_optional_dataset_keys
        )
    else:
        # With RL disabled, keep only datasets that have the complete
        # selected-timeout heuristic comparison set.
        algorithms_by_dataset: Dict[Tuple[str, str], set[str]] = {}
        for record in records:
            if (
                is_figure1_platform(record.platform)
                and record.algorithm in heuristic_algorithms
                and record.timeout == timeout
            ):
                key = (record.platform, record.dataset)
                algorithms_by_dataset.setdefault(key, set()).add(
                    record.algorithm
                )

        required_heuristics = set(heuristic_algorithms)
        target_dataset_keys = {
            key
            for key, discovered_algorithms in algorithms_by_dataset.items()
            if required_heuristics.issubset(discovered_algorithms)
        }
        rl_dataset_keys = set()

    if not target_dataset_keys:
        raise RuntimeError(
            'No Figure 1 comparison datasets were found'
        )

    return target_dataset_keys, rl_dataset_keys


def result_save_3_single_comparison_records(
    records: Sequence[ResultSave3BarRecord],
    *,
    timeout: int = RESULT_SAVE_3_SELECTED_TIMEOUT,
    include_rl: bool = True,
) -> List[ResultSave3BarRecord]:
    """Select complete Figure 1 comparisons.

    FCFS/B+IPM, SNF+IPM, and SNF-ICON are required for every panel.
    RL Budiarjo is required except on configured RL-optional platform
    sizes, currently the 1152-node platforms.
    """
    heuristic_algorithms = (
        'FCFS/B+IPM',
        'SNF+IPM',
        'SNF-ICON',
    )
    global_order = (
        *heuristic_algorithms,
        *((RESULT_SAVE_3_RL_ALGORITHM_LABEL,) if include_rl else ()),
    )

    target_dataset_keys, rl_dataset_keys = (
        result_save_3_target_dataset_keys(
            records,
            heuristic_algorithms=heuristic_algorithms,
            timeout=timeout,
            include_rl=include_rl,
        )
    )

    selected_records = [
        record
        for record in records
        if (
            (record.platform, record.dataset)
            in target_dataset_keys
            and (
                (
                    record.algorithm in heuristic_algorithms
                    and record.timeout == timeout
                )
                or (
                    include_rl
                    and record.algorithm
                    == RESULT_SAVE_3_RL_ALGORITHM_LABEL
                    and (
                        record.platform,
                        record.dataset,
                    ) in rl_dataset_keys
                )
            )
        )
    ]

    for platform, dataset in sorted(
        target_dataset_keys,
        key=lambda item: (
            item[0].lower(),
            item[1].lower(),
        ),
    ):
        expected = list(heuristic_algorithms)
        if include_rl and result_save_3_rl_required(platform):
            expected.append(
                RESULT_SAVE_3_RL_ALGORITHM_LABEL
            )

        discovered = {
            record.algorithm
            for record in selected_records
            if (
                record.platform == platform
                and record.dataset == dataset
            )
        }
        missing = [
            algorithm
            for algorithm in expected
            if algorithm not in discovered
        ]
        if missing:
            raise RuntimeError(
                f'The Figure 1 comparison for '
                f'{platform}/{dataset} is missing '
                f'{", ".join(missing)}. RL Budiarjo is '
                f'optional only for platform sizes '
                f'{RESULT_SAVE_3_RL_OPTIONAL_PLATFORM_SIZES}.'
            )

    return sorted(
        selected_records,
        key=lambda record: (
            record.platform.lower(),
            record.dataset.lower(),
            global_order.index(record.algorithm),
        ),
    )


def result_save_3_variant_comparison_records(
    records: Sequence[ResultSave3BarRecord],
    *,
    timeout: int = RESULT_SAVE_3_SELECTED_TIMEOUT,
    include_rl: bool = True,
) -> List[ResultSave3BarRecord]:
    """Select complete comparisons including every ICON variant."""
    heuristic_algorithms = (
        'FCFS/B+IPM',
        'SNF+IPM',
        'SNF-ICON',
        'SNF-ICON-NF',
        'SNF-ICON-NG',
        'SNF-ICON-NGNF',
    )
    global_order = (
        *heuristic_algorithms,
        *((RESULT_SAVE_3_RL_ALGORITHM_LABEL,) if include_rl else ()),
    )

    target_dataset_keys, rl_dataset_keys = (
        result_save_3_target_dataset_keys(
            records,
            heuristic_algorithms=heuristic_algorithms,
            timeout=timeout,
            include_rl=include_rl,
        )
    )

    selected_records = [
        record
        for record in records
        if (
            (record.platform, record.dataset)
            in target_dataset_keys
            and (
                (
                    record.algorithm in heuristic_algorithms
                    and record.timeout == timeout
                )
                or (
                    include_rl
                    and record.algorithm
                    == RESULT_SAVE_3_RL_ALGORITHM_LABEL
                    and (
                        record.platform,
                        record.dataset,
                    ) in rl_dataset_keys
                )
            )
        )
    ]

    for platform, dataset in sorted(
        target_dataset_keys,
        key=lambda item: (
            item[0].lower(),
            item[1].lower(),
        ),
    ):
        expected = list(heuristic_algorithms)
        if include_rl and result_save_3_rl_required(platform):
            expected.append(
                RESULT_SAVE_3_RL_ALGORITHM_LABEL
            )

        discovered = {
            record.algorithm
            for record in selected_records
            if (
                record.platform == platform
                and record.dataset == dataset
            )
        }
        missing = [
            algorithm
            for algorithm in expected
            if algorithm not in discovered
        ]
        if missing:
            raise RuntimeError(
                f'The variant comparison for '
                f'{platform}/{dataset} is missing '
                f'{", ".join(missing)}. RL Budiarjo is '
                f'optional only for platform sizes '
                f'{RESULT_SAVE_3_RL_OPTIONAL_PLATFORM_SIZES}.'
            )

    return sorted(
        selected_records,
        key=lambda record: (
            record.platform.lower(),
            record.dataset.lower(),
            global_order.index(record.algorithm),
        ),
    )


def result_save_3_make_barplot(records: Sequence[ResultSave3BarRecord], *, timeout: int=RESULT_SAVE_3_SELECTED_TIMEOUT, output_dir: Path=RESULT_SAVE_3_OUTPUT_DIR) -> Path:
    """Create one PNG with energy as bars and waiting time as lines."""
    if not records:
        raise ValueError('No results records supplied for bar plotting')
    dataset_keys = sorted(
        {(record.platform, record.dataset) for record in records},
        key=lambda item: (
            item[1] == 'SDSC-BLUE-2000-4.2-cln-0-3000',
            item[0].lower(),
            item[1].lower(),
        ),
    )
    platforms = sorted({platform for platform, _ in dataset_keys}, key=str.lower)
    dataset_labels = [
        dataset_display_label(dataset)
        if len(platforms) == 1
        else f"{platform}\n{dataset_display_label(dataset)}"
        for platform, dataset in dataset_keys
    ]
    algorithms = result_save_3_algorithm_order(records)
    styles = result_save_3_style_map(algorithms)
    lookup = {(record.platform, record.dataset, record.algorithm): record for record in records}

    # Normalize each metric independently within each dataset.
    # 1.0 = worst (largest value), smaller values are better.
    worst_energy_by_dataset = {}
    worst_waiting_by_dataset = {}
    for platform, dataset in dataset_keys:
        dataset_records = [
            record
            for record in records
            if record.platform == platform and record.dataset == dataset
        ]
        worst_energy_by_dataset[(platform, dataset)] = max(
            record.wasted_energy_mwh for record in dataset_records
        )
        worst_waiting_by_dataset[(platform, dataset)] = max(
            record.waiting_seconds for record in dataset_records
        )

    fig, ax_left = plt.subplots(
        nrows=1,
        ncols=1,
        figsize=((7.16, 1.0)),
    )
    ax_right = ax_left.twinx()


    x_positions = np.arange(len(dataset_keys), dtype=float) * (1.3)
    bar_width = (1.1) / max(1, len(algorithms))

    algorithm_offsets: Dict[str, float] = {}

    for algorithm_index, algorithm in enumerate(algorithms):
        offset = (
            algorithm_index
            - (len(algorithms) - 1) / 2.0
        ) * bar_width
        algorithm_offsets[algorithm] = offset

        color, hatch = styles[algorithm]
        energy_values = [
            (
                lookup[
                    platform,
                    dataset,
                    algorithm,
                ].wasted_energy_mwh
                / worst_energy_by_dataset[(platform, dataset)]
                if worst_energy_by_dataset[(platform, dataset)] > 0.0
                else 0.0
            )
            if (
                platform,
                dataset,
                algorithm,
            ) in lookup
            else np.nan
            for platform, dataset in dataset_keys
        ]

        energy_bars = ax_left.bar(
            x_positions + offset,
            energy_values,
            width=bar_width,
            label=algorithm,
            color='white',
            hatch=hatch,
            edgecolor=color,
            linewidth=(1.2),
            alpha=1.0,
            zorder=2,
        )
        result_save_3_annotate_bars(
            ax_left,
            energy_bars,
        )

    # One waiting-time line per dataset.
    # The line connects algorithms inside that dataset only.
    for dataset_index, (platform, dataset) in enumerate(dataset_keys):
        waiting_x: List[float] = []
        waiting_y: List[float] = []
        marker_colors: List[str] = []

        for algorithm in algorithms:
            key = (
                platform,
                dataset,
                algorithm,
            )
            if key not in lookup:
                continue

            waiting_value = lookup[key].waiting_seconds
            worst_waiting = worst_waiting_by_dataset[(platform, dataset)]
            waiting_value = (
                waiting_value / worst_waiting
                if worst_waiting > 0.0
                else 0.0
            )
            if not math.isfinite(waiting_value):
                continue

            waiting_x.append(
                x_positions[dataset_index]
                + algorithm_offsets[algorithm]
            )
            waiting_y.append(waiting_value)
            marker_colors.append(
                styles[algorithm][0]
            )

        if len(waiting_x) >= 2:
            ax_right.plot(
                waiting_x,
                waiting_y,
                color=('black'),
                linewidth=(1.8),
                zorder=3,
            )

        ax_right.scatter(
            waiting_x,
            waiting_y,
            c=marker_colors,
            marker=('o'),
            s=(4.5) ** 2,
            edgecolors=('black'),
            linewidths=(0.8),
            zorder=4,
        )

    ax_left.set_ylabel('EW/max EW (bars)')
    ax_right.set_ylabel('AW/max AW (lines)')

    ax_left.set_xticks(x_positions)
    ax_left.set_xticklabels(
        dataset_labels,
        rotation=(0),
        ha=('center'),
    )

    ax_left.grid(False)
    ax_right.grid(False)
    ax_left.grid(
        axis='y',
        linestyle=('--'),
        alpha=(0.5),
    )
    ax_left.set_axisbelow(True)

    ax_left.set_xlim(
        -(0.6),
        x_positions[-1] + (0.6),
    )
    ax_left.set_ylim(0.0, 1.05)
    ax_right.set_ylim(0.0, 1.05)

    fig.supxlabel('Platform & Workload', y=(-0.5))

    legend_handles = [
        Patch(
            facecolor='white',
            edgecolor=styles[algorithm][0],
            hatch=styles[algorithm][1],
            label=algorithm,
            linewidth=(1.2),
            alpha=1.0,
        )
        for algorithm in algorithms
    ]
    fig.legend(
        handles=legend_handles,
        loc=('upper center'),
        bbox_to_anchor=((0.5, 1.35)),
        ncol=(4),
        frameon=(True),
        handlelength=(1.8),
        columnspacing=(1.0),
        borderpad=(0.25),
    )

    fig.subplots_adjust(
        left=(0.0),
        right=(1.0),
        top=(1.0),
        bottom=(0.0),
        wspace=(0.1),
        hspace=(0.0),
    )

    output_dir = output_dir.expanduser().resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    output_stem = f'CustomVisPaper_Figure1_Barplot_timeout-{timeout}'
    png_path = output_dir / f'{output_stem}.png'
    pdf_path = output_dir / f'{output_stem}.pdf'
    current_outputs = {png_path, pdf_path}
    for stale_path in output_dir.glob('CustomVisPaper_Figure1_Barplot_*'):
        if stale_path.is_file() and stale_path not in current_outputs:
            stale_path.unlink()
    save_figure(
        fig,
        png_path,
        pdf_path,
        png_kwargs={},
        pdf_kwargs={},
    )
    plt.close(fig)
    print(f'Wrote {png_path}')
    print(f'Wrote {pdf_path}')
    return png_path

# =============================================================================
# FIGURE 1 PLOT SETTINGS
# Edit these values, then rerun this cell.
# =============================================================================

# Include RL Budiarjo in discovery, completeness checks, and plotting.
# Set to True to include RL again.
FIGURE1_INCLUDE_RL = True

# Generate and display Figure 1
apply_plot_style()
figure1_all_records = result_save_3_discover_barplot_records(
    include_oracle=False,
    selected_timeout=RESULT_SAVE_3_SELECTED_TIMEOUT,
    include_icon_variants=False,
    include_rl=FIGURE1_INCLUDE_RL,
)
figure1_comparison_records = result_save_3_single_comparison_records(
    figure1_all_records,
    timeout=RESULT_SAVE_3_SELECTED_TIMEOUT,
    include_rl=FIGURE1_INCLUDE_RL,
)
png_path = result_save_3_make_barplot(
    figure1_comparison_records,
    timeout=RESULT_SAVE_3_SELECTED_TIMEOUT,
)
display(Image(filename=str(png_path)))
png_path

# Figure 1 comparison table (FCFS/B+IPM baseline)
def figure1_build_comparison_table(
    records: Sequence[ResultSave3BarRecord],
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    algorithm_order = result_save_3_algorithm_order(records)
    group_keys = sorted(
        {(record.platform, record.dataset) for record in records},
        key=lambda key: (key[0].lower(), key[1].lower()),
    )
    for platform, dataset in group_keys:
        group = [
            record for record in records
            if record.platform == platform and record.dataset == dataset
        ]
        baseline_matches = [r for r in group if r.algorithm == 'FCFS/B+IPM']
        if len(baseline_matches) != 1:
            raise RuntimeError(
                f'Expected one FCFS/B+IPM baseline for {platform} / {dataset}; '
                f'found {len(baseline_matches)}.'
            )
        baseline = baseline_matches[0]
        for record in sorted(group, key=lambda r: algorithm_order.index(r.algorithm)):
            rows.append({
                'platform': platform,
                'workload': dataset_display_label(dataset),
                'algorithm': (
                    COMPARISON_BASELINE_DISPLAY_LABEL
                    if record.algorithm == 'FCFS/B+IPM'
                    else record.algorithm
                ),
                'waiting_time_seconds': record.waiting_seconds,
                'energy_waste_mwh': record.wasted_energy_mwh,
                'waiting_time_improvement_vs_fcfs_b_ipm_percent': (
                    comparison_improvement_percent(
                        record.waiting_seconds,
                        baseline.waiting_seconds,
                    )
                ),
                'energy_waste_improvement_vs_fcfs_b_ipm_percent': (
                    comparison_improvement_percent(
                        record.wasted_energy_mwh,
                        baseline.wasted_energy_mwh,
                    )
                ),
            })
    return add_explicit_fcfs_b_ipm_comparisons(
        pd.DataFrame(rows),
        group_columns=['platform', 'workload'],
        metric_specs=[
            (
                'waiting_time_seconds',
                'fcfs_b_ipm_waiting_time_seconds',
                'waiting_time_difference_seconds',
                'waiting_time_improvement_percent',
                'waiting_time_comparison',
                False,
            ),
            (
                'energy_waste_mwh',
                'fcfs_b_ipm_energy_waste_mwh',
                'energy_waste_difference_mwh',
                'energy_waste_improvement_percent',
                'energy_waste_comparison',
                False,
            ),
        ],
    )


figure1_comparison_table = display_comparison_table(
    figure1_build_comparison_table(figure1_comparison_records),
    'Figure 1 comparison table',
)


## Figure 2

Kiviat comparison.


In [ ]:
# Figure 2
# Run the common setup cell and Figure 1 first.
# This cell uses the exact same selected run records as Figure 1.

from __future__ import annotations

EPS = 1e-12

FIGURE2_ALGORITHM_ORDER: Tuple[str, ...] = (
    'FCFS/B+IPM',
    'SNF+IPM',
    'SNF-ICON',
    RESULT_SAVE_3_RL_ALGORITHM_LABEL,
)

FIGURE2_KIVIAT_AXES: Tuple[
    Tuple[str, str, bool, bool],
    ...,
] = (
    ('avg_wait', 'AW', True, False),
    ('max_wait', 'MW', True, False),
    ('avg_slowdown', 'AS', True, False),
    ('avg_response', 'AR', True, False),
    ('utilization', 'SU', False, True),
    ('wasted_energy_mwh', 'EE', True, False),
)


def figure2_barplot_records() -> List[ResultSave3BarRecord]:
    """Return the exact records selected and plotted by Figure 1."""
    if 'figure1_comparison_records' not in globals():
        raise RuntimeError(
            'Run Figure 1 before Figure 2. Figure 2 '
            'intentionally reuses the exact Figure 1 '
            'barplot records.'
        )

    records = list(figure1_comparison_records)
    if not records:
        raise RuntimeError(
            'Figure 1 did not produce any comparison records.'
        )

    return records


def figure2_read_jobs(
    record: ResultSave3BarRecord,
) -> pd.DataFrame:
    run_dir = record.metrics_path.parent
    raw_path = run_dir / 'raw_job_log.csv'
    wait_path = run_dir / 'waiting_time_log.csv'

    if not raw_path.is_file():
        raise FileNotFoundError(
            f'Missing job log: {raw_path}'
        )
    if not wait_path.is_file():
        raise FileNotFoundError(
            f'Missing waiting-time log: {wait_path}'
        )

    register_source_file(raw_path, 'raw_job_log')
    register_source_file(wait_path, 'waiting_time_log')

    raw = pd.read_csv(raw_path)
    wait = pd.read_csv(wait_path)

    if raw.empty:
        raise ValueError(f'Empty job log: {raw_path}')
    if wait.empty:
        raise ValueError(
            f'Empty waiting-time log: {wait_path}'
        )

    require_columns(
        raw,
        ('job_id', 'subtime', 'finish_time', 'runtime'),
        raw_path,
    )
    require_columns(
        wait,
        ('job_id', 'waiting_time'),
        wait_path,
    )

    if raw['job_id'].isna().any():
        raise ValueError(f'Null job_id in {raw_path}')
    if wait['job_id'].isna().any():
        raise ValueError(f'Null job_id in {wait_path}')

    if raw['job_id'].duplicated().any():
        duplicates = raw.loc[
            raw['job_id'].duplicated(keep=False),
            'job_id',
        ].tolist()
        raise ValueError(
            f'Duplicate job_id in {raw_path}: {duplicates}'
        )

    if wait['job_id'].duplicated().any():
        duplicates = wait.loc[
            wait['job_id'].duplicated(keep=False),
            'job_id',
        ].tolist()
        raise ValueError(
            f'Duplicate job_id in {wait_path}: {duplicates}'
        )

    raw_ids = set(raw['job_id'].tolist())
    wait_ids = set(wait['job_id'].tolist())
    if raw_ids != wait_ids:
        missing_wait = sorted(raw_ids - wait_ids, key=str)
        missing_raw = sorted(wait_ids - raw_ids, key=str)
        raise ValueError(
            'Job IDs do not match between logs. '
            f'Missing in waiting log: {missing_wait}; '
            f'missing in raw log: {missing_raw}'
        )

    jobs = raw[
        ['job_id', 'subtime', 'finish_time', 'runtime']
    ].merge(
        wait[['job_id', 'waiting_time']],
        on='job_id',
        how='left',
        sort=False,
        validate='one_to_one',
    )

    for column, source_path in (
        ('subtime', raw_path),
        ('finish_time', raw_path),
        ('runtime', raw_path),
        ('waiting_time', wait_path),
    ):
        jobs[column] = strict_numeric(
            jobs[column],
            column,
            source_path,
        )

    if (jobs['runtime'] < 0.0).any():
        bad = jobs.loc[
            jobs['runtime'] < 0.0,
            ['job_id', 'runtime'],
        ]
        raise ValueError(
            'runtime must be >= 0 for every job. '
            f'Bad rows:\n{bad.to_string(index=False)}'
        )

    if (jobs['waiting_time'] < 0.0).any():
        bad = jobs.loc[
            jobs['waiting_time'] < 0.0,
            ['job_id', 'waiting_time'],
        ]
        raise ValueError(
            'waiting_time must be >= 0 for every job. '
            f'Bad rows:\n{bad.to_string(index=False)}'
        )

    jobs['response_time'] = (
        jobs['finish_time'] - jobs['subtime']
    )
    response_values = jobs['response_time'].to_numpy(
        dtype=float
    )
    if not np.isfinite(response_values).all():
        raise ValueError(
            f'Non-finite response time in {run_dir}'
        )

    if (jobs['response_time'] < 0.0).any():
        bad = jobs.loc[
            jobs['response_time'] < 0.0,
            [
                'job_id',
                'subtime',
                'finish_time',
                'response_time',
            ],
        ]
        raise ValueError(
            'response_time must be >= 0 for every job. '
            f'Bad rows:\n{bad.to_string(index=False)}'
        )

    return jobs


def figure2_metrics_totals(
    record: ResultSave3BarRecord,
) -> Tuple[float, float]:
    path = record.metrics_path
    frame = pd.read_csv(path)

    if len(frame) != 1:
        raise ValueError(
            f'{path} must contain exactly one data row; '
            f'found {len(frame)}'
        )

    required = (
        'total_active_compute',
        'total_time_all_states',
    )
    require_columns(frame, required, path)

    active_compute = float(
        strict_numeric(
            frame['total_active_compute'],
            'total_active_compute',
            path,
        ).iloc[0]
    )
    time_all_states = float(
        strict_numeric(
            frame['total_time_all_states'],
            'total_time_all_states',
            path,
        ).iloc[0]
    )

    if time_all_states <= 0.0:
        raise ValueError(
            f'total_time_all_states must be > 0 in {path}; '
            f'found {time_all_states}'
        )

    return active_compute, time_all_states


def figure2_derive_metrics(
    record: ResultSave3BarRecord,
) -> Dict[str, float]:
    jobs = figure2_read_jobs(record)

    slowdown_jobs = jobs.loc[
        jobs['runtime'] > 0.0,
        ['response_time', 'runtime'],
    ]
    if slowdown_jobs.empty:
        raise ValueError(
            f'No jobs with runtime > 0 in '
            f'{record.metrics_path.parent}'
        )

    slowdown = (
        slowdown_jobs['response_time']
        / slowdown_jobs['runtime']
    )
    if not np.isfinite(
        slowdown.to_numpy(dtype=float)
    ).all():
        raise ValueError(
            f'Non-finite slowdown in '
            f'{record.metrics_path.parent}'
        )

    active_compute, time_all_states = (
        figure2_metrics_totals(record)
    )
    utilization = active_compute / time_all_states

    values = {
        # Exact mean-waiting-time value used by Figure 1.
        'avg_wait': float(record.waiting_seconds),
        'max_wait': float(jobs['waiting_time'].max()),
        'avg_response': float(
            jobs['response_time'].mean()
        ),
        'avg_slowdown': float(slowdown.mean()),
        'utilization': float(utilization),
        # Exact energy value used by Figure 1.
        'wasted_energy_mwh': float(
            record.wasted_energy_mwh
        ),
    }

    for name, value in values.items():
        if not math.isfinite(value):
            raise ValueError(
                f'Invalid metric {name!r} for '
                f'{record.metrics_path.parent}: {value}'
            )

    for name in (
        'avg_wait',
        'max_wait',
        'avg_response',
        'avg_slowdown',
    ):
        if values[name] < 0.0:
            raise ValueError(
                f'{name} must be >= 0 in '
                f'{record.metrics_path.parent}; '
                f'found {values[name]}'
            )

    if not 0.0 <= values['utilization'] <= 1.0 + EPS:
        raise ValueError(
            'utilization must be in [0, 1] for '
            f'{record.metrics_path.parent}; '
            f"found {values['utilization']}"
        )

    return values


def figure2_normalize_metric(
    values: pd.Series,
    *,
    higher_is_better: bool,
) -> pd.Series:
    numeric = pd.to_numeric(
        values,
        errors='raise',
    ).astype(float)

    if not np.isfinite(
        numeric.to_numpy(dtype=float)
    ).all():
        raise ValueError(
            'Non-finite value passed to Kiviat normalization'
        )

    low = float(numeric.min())
    high = float(numeric.max())

    if abs(high - low) <= EPS:
        return pd.Series(
            1.0,
            index=numeric.index,
            dtype=float,
        )

    scaled = (numeric - low) / (high - low)
    return (
        scaled
        if higher_is_better
        else 1.0 - scaled
    )


def figure2_build_dataset_frame(
    records: Sequence[ResultSave3BarRecord],
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []

    for record in records:
        metrics = figure2_derive_metrics(record)
        rows.append(
            {
                'algorithm': record.algorithm,
                'run_dir': str(record.metrics_path.parent),
                **metrics,
            }
        )

    frame = pd.DataFrame(rows)
    if len(frame) < 2:
        raise RuntimeError(
            'Each Kiviat panel requires at least two algorithms'
        )

    if frame['algorithm'].duplicated().any():
        duplicates = frame.loc[
            frame['algorithm'].duplicated(keep=False),
            'algorithm',
        ].tolist()
        raise ValueError(
            f'Duplicate algorithms in one Kiviat panel: '
            f'{duplicates}'
        )

    platforms = {
        record.platform
        for record in records
    }
    if len(platforms) != 1:
        raise RuntimeError(
            f'Each Figure 2 panel must contain exactly one '
            f'platform; found {sorted(platforms)}'
        )
    platform = next(iter(platforms))

    # RL is optional for every individual dataset panel.
    # The three non-RL baselines remain mandatory.
    required_algorithms = [
        algorithm
        for algorithm in FIGURE2_ALGORITHM_ORDER
        if algorithm != RESULT_SAVE_3_RL_ALGORITHM_LABEL
    ]
    present_algorithms = set(frame['algorithm'])
    missing = [
        algorithm
        for algorithm in required_algorithms
        if algorithm not in present_algorithms
    ]
    if missing:
        raise RuntimeError(
            f'Figure 2 is missing {", ".join(missing)} '
            f'for {platform}.'
        )

    algorithm_order = [
        algorithm
        for algorithm in FIGURE2_ALGORITHM_ORDER
        if algorithm in present_algorithms
    ]

    frame['algorithm_order'] = frame['algorithm'].map(
        algorithm_order.index
    )
    frame = (
        frame.sort_values('algorithm_order')
        .drop(columns='algorithm_order')
        .reset_index(drop=True)
    )

    plotted = frame[['algorithm', 'run_dir']].copy()
    for (
        metric,
        _,
        should_normalize,
        higher_is_better,
    ) in FIGURE2_KIVIAT_AXES:
        if should_normalize:
            plotted[metric] = figure2_normalize_metric(
                frame[metric],
                higher_is_better=higher_is_better,
            )
        else:
            plotted[metric] = frame[metric]

    return plotted


def figure2_make_kiviat(
    records: Sequence[ResultSave3BarRecord],
) -> Tuple[Path, Path]:
    dataset_keys = sorted(
        {
            (record.platform, record.dataset)
            for record in records
        },
        key=lambda item: (
            item[0].lower(),
            item[1].lower(),
        ),
    )
    if not dataset_keys:
        raise ValueError('No Figure 2 records supplied')

    styles_from_barplot = result_save_3_style_map(
        FIGURE2_ALGORITHM_ORDER
    )
    styles = {
        algorithm: {
            'color': styles_from_barplot[algorithm][0],
            'marker': (('o', 's', '^', 'D'))[index],
            'linestyle': (('-', '--', '-.', ':'))[index],
        }
        for index, algorithm
        in enumerate(FIGURE2_ALGORITHM_ORDER)
    }

    angles = np.linspace(
        0.0,
        2.0 * np.pi,
        len(FIGURE2_KIVIAT_AXES),
        endpoint=False,
    )
    closed_angles = np.concatenate(
        [angles, angles[:1]]
    )

    panel_count = len(dataset_keys)
    panel_ncols = min((2), panel_count)
    panel_nrows = math.ceil(panel_count / panel_ncols)

    fig, axes = plt.subplots(
        nrows=panel_nrows,
        ncols=panel_ncols,
        figsize=((4.0, 8.0)),
        subplot_kw={'projection': 'polar'},
        squeeze=False,
    )
    panel_axes = axes.ravel()

    for unused_ax in panel_axes[panel_count:]:
        unused_ax.set_visible(False)

    for panel_index, (
        ax,
        (platform, dataset),
    ) in enumerate(zip(panel_axes, dataset_keys)):
        dataset_records = [
            record
            for record in records
            if (
                record.platform == platform
                and record.dataset == dataset
            )
        ]
        frame = figure2_build_dataset_frame(
            dataset_records
        )

        for _, row in frame.iterrows():
            algorithm = str(row['algorithm'])
            values = [
                float(row[metric])
                for metric, _, _, _
                in FIGURE2_KIVIAT_AXES
            ]
            closed_values = values + values[:1]
            style = styles[algorithm]

            ax.plot(
                closed_angles,
                closed_values,
                color=style['color'],
                marker=style['marker'],
                linestyle=style['linestyle'],
                linewidth=(1.5),
                markersize=(5.0),
                label=algorithm,
            )
            ax.fill(
                closed_angles,
                closed_values,
                color=style['color'],
                alpha=(0.025),
            )

        ax.set_theta_offset(np.pi / 2.0)
        ax.set_theta_direction(-1)
        ax.set_xticks(angles)
        ax.set_xticklabels(
            ['' for _ in FIGURE2_KIVIAT_AXES]
        )

        axis_labels = [
            label
            for _, label, _, _
            in FIGURE2_KIVIAT_AXES
        ]
        if (
            len(([1.06, 1.2, 1.2, 1.06, 1.2, 1.2]))
            != len(axis_labels)
        ):
            raise RuntimeError(
                'FIGURE2_XTICK_LABEL_RADII must have '
                'one value per Kiviat axis'
            )

        for label_index, (
            angle,
            label,
            label_radius,
        ) in enumerate(
            zip(
                angles,
                axis_labels,
                ([1.06, 1.2, 1.2, 1.06, 1.2, 1.2]),
            )
        ):
            if label_index == 0:
                vertical_alignment = 'bottom'
            elif label_index == len(axis_labels) // 2:
                vertical_alignment = 'top'
            else:
                vertical_alignment = 'center'

            ax.text(
                angle,
                label_radius,
                label,
                ha='center',
                va=vertical_alignment,
                linespacing=1.15,
                clip_on=False,
            )

        ax.set_ylim(0.0, 1.0)

        ax.set_rlabel_position(22)
        ax.set_yticklabels([])
        ax.set_title(
            f"({chr(ord('a') + panel_index)}) "
            f"{platform} / "
            f"{dataset_display_label(dataset)}",
            y=(1.2),
            pad=0,
        )

    present_legend_algorithms = [
        algorithm
        for algorithm in FIGURE2_ALGORITHM_ORDER
        if any(
            record.algorithm == algorithm
            for record in records
        )
    ]
    legend_handles = [
        Line2D(
            [0],
            [0],
            color=styles[algorithm]['color'],
            marker=styles[algorithm]['marker'],
            linestyle=styles[algorithm]['linestyle'],
            linewidth=(1.5),
            markersize=(6.0),
            label=algorithm,
        )
        for algorithm in present_legend_algorithms
    ]

    fig.legend(
        handles=legend_handles,
        labels=present_legend_algorithms,
        loc='lower center',
        bbox_to_anchor=((0.5, -0.03)),
        ncol=len(present_legend_algorithms),
        frameon=True,
        columnspacing=(1.0),
        handlelength=(2.7),
    )
    fig.subplots_adjust(
        left=(0.0),
        right=(1.0),
        top=(1.0),
        bottom=(0.0),
        wspace=(0.6),
        hspace=(0.0),
    )

    for ax in panel_axes:
        position = ax.get_position()
        scale = (1.2)
        width = position.width * scale
        height = position.height * scale
        center_x = position.x0 + position.width / 2.0
        center_y = position.y0 + position.height / 2.0
        ax.set_position(
            [
                center_x - width / 2.0,
                center_y - height / 2.0,
                width,
                height,
            ]
        )

    output_dir = OUTPUT_DIR / 'Figure2_Kiviat'
    output_dir.mkdir(parents=True, exist_ok=True)
    png_path = (
        output_dir
        / 'CustomVisPaper_Figure2_Kiviat.png'
    )
    pdf_path = (
        output_dir
        / 'CustomVisPaper_Figure2_Kiviat.pdf'
    )

    save_figure(
        fig,
        png_path,
        pdf_path,
        png_kwargs={},
        pdf_kwargs={},
    )
    plt.close(fig)

    write_chart_swept_config(
        png_path,
        {
            f'{platform} / {dataset}': [
                record.metrics_path.parent
                for record in records
                if (
                    record.platform == platform
                    and record.dataset == dataset
                )
            ]
            for platform, dataset in dataset_keys
        },
    )

    return png_path, pdf_path


apply_plot_style()
figure2_records = figure2_barplot_records()
png_path, pdf_path = figure2_make_kiviat(
    figure2_records
)
display(Image(filename=str(png_path)))
(png_path, pdf_path)

# Figure 2 comparison table (FCFS/B+IPM baseline)
def figure2_build_comparison_table(
    records: Sequence[ResultSave3BarRecord],
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    group_keys = sorted(
        {(record.platform, record.dataset) for record in records},
        key=lambda key: (key[0].lower(), key[1].lower()),
    )
    for platform, dataset in group_keys:
        group = [r for r in records if r.platform == platform and r.dataset == dataset]
        metric_rows = [{'record': r, **figure2_derive_metrics(r)} for r in group]
        baseline_matches = [row for row in metric_rows if row['record'].algorithm == 'FCFS/B+IPM']
        if len(baseline_matches) != 1:
            raise RuntimeError(
                f'Expected one FCFS/B+IPM baseline for {platform} / {dataset}; '
                f'found {len(baseline_matches)}.'
            )
        baseline = baseline_matches[0]
        for row in metric_rows:
            record = row['record']
            output: Dict[str, object] = {
                'platform': platform,
                'workload': dataset_display_label(dataset),
                'algorithm': (
                    COMPARISON_BASELINE_DISPLAY_LABEL
                    if record.algorithm == 'FCFS/B+IPM'
                    else record.algorithm
                ),
            }
            for metric, _, _, higher_is_better in FIGURE2_KIVIAT_AXES:
                output[metric] = row[metric]
                output[f'{metric}_improvement_vs_fcfs_b_ipm_percent'] = (
                    comparison_improvement_percent(
                        row[metric],
                        baseline[metric],
                        higher_is_better=higher_is_better,
                    )
                )
            rows.append(output)
    return add_explicit_fcfs_b_ipm_comparisons(
        pd.DataFrame(rows),
        group_columns=['platform', 'workload'],
        metric_specs=[
            (
                metric,
                f'fcfs_b_ipm_{metric}',
                f'{metric}_difference',
                f'{metric}_improvement_percent',
                f'{metric}_comparison',
                higher_is_better,
            )
            for metric, _, _, higher_is_better in FIGURE2_KIVIAT_AXES
        ],
    )


figure2_comparison_table = display_comparison_table(
    figure2_build_comparison_table(figure2_records),
    'Figure 2 comparison table',
)

## Figure 3

Markov diagnostics.


In [ ]:
# Figure 3
# Run the common setup cell first. This cell contains Figure 3-specific code and settings.

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Mapping, Sequence, Tuple
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import ConnectionPatch, Rectangle
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from IPython.display import display, Image
from matplotlib.ticker import FormatStrFormatter
from matplotlib.lines import Line2D


HEURISTIC_ROOT = RESULT_SAVE_3_ROOT / 'Heuristic-ICON'
FIGURE3_PLATFORM = 'AOBA-64'
FIGURE3_TIMEOUT = 7200
FIGURE3_ALPHA: float | None = None
FIGURE3_BETA: float | None = None
FIGURE3_VARIANT = 'snf-ssc'
FIGURE3_WORKLOADS: Tuple[str, ...] = (
    'generated-markovian-3000',
    'DAS2-fs1-0-3000',
    'DAS2-fs2-0-3000',
    'DAS2-fs3-0-3000',
    'DAS2-fs4-0-3000',
)
FIGURE3_CONDITIONS_WORKLOADS: Tuple[str, ...] = (
    'generated-markovian-3000',
    'DAS2-fs3-0-3000',
)
FIGURE3_CONDITIONS_COLUMN_TITLES: Dict[str, str] = {
    'generated-markovian-3000': (
        'Generated Markovian'
    ),
    'DAS2-fs3-0-3000': (
        'DAS2 FS3'
    ),
}
FIGURE3_CV_TOLERANCE = 0.45
FIGURE3_LAG1_LIMIT = 0.65
FIGURE3_KS_PARAM = 3.0
SWEEP_RUN_DIR_PREFIX = 'snf_icon'
FIGURE3_RUN_DIR_PREFIX = SWEEP_RUN_DIR_PREFIX


@dataclass(frozen=True)
class Figure3RunSelection:
    workload: str
    run_dir: Path
    scheduler_log_path: Path
    alpha: float
    beta: float
    timeout: int
    markov_window_seconds: float


SOURCE_DIR_RECORDS: List[Dict[str, str]] = []
SOURCE_DIR_SEEN: set[Tuple[str, str]] = set()


def source_root_label(path: Path) -> str:
    resolved = path.expanduser().resolve()
    roots = (('Heuristic-ICON', HEURISTIC_ROOT),)

    for label, root in roots:
        try:
            resolved.relative_to(root.expanduser().resolve())
            return label
        except ValueError:
            continue

    return 'Other'


def platform_node_count(platform: str) -> int:
    match = re.search(r'-(\d+)(?:-|$)', platform)

    if match is None:
        raise ValueError(
            f'Could not infer node count from platform name: {platform}'
        )

    return int(match.group(1))


IMPORTANT_ALGO_CONFIG_KEYS: Tuple[str, ...] = (
    'run.algo_config.alpha',
    'run.algo_config.beta',
    MARKOV_WINDOW_CONFIG_KEY,
)
FIGURE_CONFIG_RESULT_DIRS: Dict[Path, set[Path]] = {}


def unique_config_numeric_values(
    result_dirs: Sequence[Path],
    key: str,
    *,
    skip_missing: bool = False,
) -> List[float]:
    values: List[float] = []

    for result_dir in result_dirs:
        try:
            value = config_numeric_value(result_dir, key)
        except KeyError:
            if skip_missing:
                continue
            raise

        if not any(
            abs(value - existing)
            <= 1e-09 * max(1.0, abs(value), abs(existing))
            for existing in values
        ):
            values.append(value)

    values.sort()
    return values


FIGURE3_TIMEOUT_CONFIG_KEY = 'run.algo_config.timeout'


def figure3_config_timeout_seconds(result_dir: Path) -> int:
    return int(
        round(
            config_numeric_value(
                result_dir,
                FIGURE3_TIMEOUT_CONFIG_KEY,
            )
        )
    )


def validated_config_timeout_seconds(
    result_dir: Path,
    expected_timeout: int,
    expected_source: str,
) -> int:
    config_timeout = figure3_config_timeout_seconds(result_dir)

    if config_timeout != int(expected_timeout):
        raise RuntimeError(
            f'Timeout mismatch for {result_dir}: '
            f'{expected_source} gives {int(expected_timeout)}, '
            f'but simulator_config_used.yaml gives {config_timeout}'
        )

    return config_timeout


def timeout_title_label(result_dirs: Sequence[Path]) -> str:
    values: List[float] = []

    for result_dir in result_dirs:
        value = float(figure3_config_timeout_seconds(result_dir))

        if not any(
            abs(value - existing)
            <= 1e-09 * max(1.0, abs(value), abs(existing))
            for existing in values
        ):
            values.append(value)

    if not values:
        raise KeyError(
            'No scheduler timeout value found for title generation'
        )

    values.sort()
    hours = [value / 3600.0 for value in values]

    if len(hours) == 1:
        return f'$\\Delta_t$={hours[0]:g} h'

    return (
        f'$\\Delta_t\\in['
        f'{hours[0]:g}, {hours[-1]:g}'
        f']$ h'
    )


def figure3_bool_series(
    frame: pd.DataFrame,
    column: str,
) -> pd.Series:
    if column not in frame.columns:
        raise KeyError(f'Missing column {column!r}')

    series = frame[column]

    if series.dtype == bool:
        return series.astype(bool)

    mapped = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                'true': True,
                'false': False,
                '1': True,
                '0': False,
            }
        )
    )

    if mapped.isna().any():
        bad = (
            series[mapped.isna()]
            .astype(str)
            .unique()
            .tolist()
        )
        raise ValueError(
            f'Invalid boolean values in {column!r}: {bad}'
        )

    return mapped.astype(bool)


def figure3_read_scheduler_log(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(
            f'Missing scheduler decision log: {path}'
        )

    register_source_file(path, 'scheduler_decision_log')
    frame = pd.read_csv(path, low_memory=False)

    if frame.empty:
        raise ValueError(
            f'Empty scheduler decision log: {path}'
        )

    required = (
        'current_time',
        'arrival_coefficient_of_variation',
        'arrival_lag1_autocorrelation',
        'arrival_ks_distance',
        'arrival_sample_count',
        'fallback',
        'gate',
        'spare_suppressed',
        'waiting_jobs',
        'idle_nodes',
        'spare_target',
    )
    require_columns(frame, required, path)

    numeric_columns = (
        'current_time',
        'arrival_coefficient_of_variation',
        'arrival_lag1_autocorrelation',
        'arrival_ks_distance',
        'arrival_sample_count',
        'waiting_jobs',
        'idle_nodes',
        'spare_target',
    )

    for column in numeric_columns:
        frame[column] = pd.to_numeric(
            frame[column],
            errors='coerce',
        )

    for column in (
        'fallback',
        'gate',
        'spare_suppressed',
    ):
        frame[column] = figure3_bool_series(
            frame,
            column,
        )
    frame = (
        frame.sort_values('current_time')
        .reset_index(drop=True)
    )
    frame['time_hours'] = (
        frame['current_time'] / 3600.0
    )

    return frame


def figure3_workload_styles(
    workloads: Sequence[str],
) -> Dict[str, Dict[str, object]]:
    styles: Dict[str, Dict[str, object]] = {}

    for index, workload in enumerate(workloads):
        styles[workload] = {
            'color': COLORS[index % len(COLORS)],
            'marker': MARKERS[index % len(MARKERS)],
            'label': workload,
        }

    return styles


def figure3_heuristic_run_dir(
    *,
    variant: str,
    platform: str,
    workload: str,
    timeout: int,
) -> Path:
    return (
        HEURISTIC_ROOT
        / f'{variant}_{platform_node_count(platform)}'
        / platform
        / workload
        / f'{FIGURE3_RUN_DIR_PREFIX}_timeout-{timeout}'
    )


def figure3_discover_run_selections(
    root: Path = HEURISTIC_ROOT,
    platform: str = FIGURE3_PLATFORM,
    workloads: Sequence[str] = FIGURE3_WORKLOADS,
    timeout: int = FIGURE3_TIMEOUT,
    alpha: float | None = FIGURE3_ALPHA,
    beta: float | None = FIGURE3_BETA,
    variant: str = FIGURE3_VARIANT,
) -> Dict[str, Figure3RunSelection]:
    root = root.expanduser().resolve()

    if root != HEURISTIC_ROOT.expanduser().resolve():
        raise ValueError(
            f'Figure 3 must read from HEURISTIC_ROOT. '
            f'Received root={root}'
        )

    if not root.is_dir():
        raise FileNotFoundError(
            f'Figure 3 Heuristic root does not exist: {root}'
        )

    selections: Dict[str, Figure3RunSelection] = {}

    for workload in workloads:
        run_dir = figure3_heuristic_run_dir(
            variant=variant,
            platform=platform,
            workload=workload,
            timeout=timeout,
        )
        scheduler_log_path = (
            run_dir / 'scheduler_decision_log.csv'
        )

        if not scheduler_log_path.is_file():
            raise FileNotFoundError(
                f'Missing Figure 3 scheduler decision log: '
                f'{scheduler_log_path}'
            )

        config_timeout = validated_config_timeout_seconds(
            run_dir,
            timeout,
            'Figure 3 Heuristic run directory name',
        )
        config_alpha = config_numeric_value(
            run_dir,
            'run.algo_config.alpha',
        )
        config_beta = config_numeric_value(
            run_dir,
            'run.algo_config.beta',
        )

        if (
            alpha is not None
            and abs(config_alpha - alpha)
            > 1e-09
            * max(1.0, abs(config_alpha), abs(alpha))
        ):
            raise RuntimeError(
                f'Figure 3 alpha filter does not match '
                f'Heuristic config for {run_dir}: '
                f'filter gives {alpha:g}, but '
                f'simulator_config_used.yaml gives '
                f'{config_alpha:g}'
            )

        if (
            beta is not None
            and abs(config_beta - beta)
            > 1e-09
            * max(1.0, abs(config_beta), abs(beta))
        ):
            raise RuntimeError(
                f'Figure 3 beta filter does not match '
                f'Heuristic config for {run_dir}: '
                f'filter gives {beta:g}, but '
                f'simulator_config_used.yaml gives '
                f'{config_beta:g}'
            )

        selections[workload] = Figure3RunSelection(
            workload=workload,
            run_dir=run_dir,
            scheduler_log_path=scheduler_log_path,
            alpha=config_alpha,
            beta=config_beta,
            timeout=config_timeout,
            markov_window_seconds=config_numeric_value(
                run_dir,
                MARKOV_WINDOW_CONFIG_KEY,
            ),
        )

    return selections


def figure3_ks_limit(
    sample_count: pd.Series,
) -> pd.Series:
    count = pd.to_numeric(
        sample_count,
        errors='coerce',
    )
    count = count.where(count > 0.0)
    return FIGURE3_KS_PARAM / np.sqrt(count)


def figure3_mode_series(
    frame: pd.DataFrame,
) -> pd.Series:
    # Existing convention:
    # fallback=True  -> displayed M(t)=1
    # fallback=False -> displayed M(t)=2
    #
    # Internal values:
    # 0 -> displayed label 1
    # 1 -> displayed label 2
    return (~frame['fallback']).astype(int)


def figure3_mode_duration_hours(
    frame: pd.DataFrame,
) -> Tuple[float, float]:
    """
    Calculate time in each mode between consecutive scheduler
    decisions. The final row contributes zero duration because
    no later decision timestamp is available.
    """
    mode_values = (
        figure3_mode_series(frame)
        .to_numpy(dtype=int)
    )
    times = (
        pd.to_numeric(
            frame['current_time'],
            errors='coerce',
        )
        .to_numpy(dtype=float)
    )

    if len(times) <= 1:
        return 0.0, 0.0

    durations = np.diff(times)
    interval_modes = mode_values[:-1]

    valid = (
        np.isfinite(durations)
        & (durations >= 0.0)
    )
    durations = durations[valid]
    interval_modes = interval_modes[valid]

    mode1_hours = (
        durations[interval_modes == 0].sum()
        / 3600.0
    )
    mode2_hours = (
        durations[interval_modes == 1].sum()
        / 3600.0
    )

    return (
        float(mode1_hours),
        float(mode2_hours),
    )


def figure3_last_completed_mode2_interval_indices(
    frame: pd.DataFrame,
) -> Tuple[int, int] | None:
    """
    Find the last completed interval that starts when M(t)
    flips from 1 to 2 and ends when it next flips from 2 to 1.

    The returned values are row indices for the two flip events.
    """
    mode_values = (
        figure3_mode_series(frame)
        .to_numpy(dtype=int)
    )

    if len(mode_values) < 3:
        return None

    active_start_index: int | None = None
    last_completed: Tuple[int, int] | None = None

    for index in range(1, len(mode_values)):
        previous_mode = mode_values[index - 1]
        current_mode = mode_values[index]

        if previous_mode == 0 and current_mode == 1:
            active_start_index = index

        elif (
            previous_mode == 1
            and current_mode == 0
            and active_start_index is not None
        ):
            last_completed = (
                active_start_index,
                index,
            )
            active_start_index = None

    return last_completed


def figure3_padded_inset_indices(
    frame: pd.DataFrame,
    start_index: int,
    end_index: int,
) -> Tuple[int, int]:
    padded_start = max(
        0,
        start_index - (5),
    )
    padded_end = min(
        len(frame) - 1,
        end_index + (5),
    )
    return padded_start, padded_end

def figure3_apply_axis_ticks(
    ax: plt.Axes,
    workload: str,
    tick_config: Mapping[
        str,
        Sequence[float] | None,
    ],
) -> None:
    ticks = tick_config.get(workload)

    if ticks is not None:
        ax.set_xticks(list(ticks))


def figure3_apply_main_xlim(
    ax: plt.Axes,
    workload: str,
) -> None:
    limits = ({'generated-markovian-3000': None, 'DAS2-fs3-0-3000': None, 'DAS2-fs4-0-3000': None}).get(workload)

    if limits is not None:
        ax.set_xlim(*limits)


def figure3_add_zoom_box_and_connectors(
    fig: plt.Figure,
    main_ax: plt.Axes,
    zoom_ax: plt.Axes,
    x0: float,
    x1: float,
) -> None:
    y0 = (-0.05)
    y1 = (1.1)

    selection_box = Rectangle(
        (x0, y0),
        x1 - x0,
        y1 - y0,
        fill=False,
        edgecolor=('#555555'),
        linewidth=(0.9),
        linestyle=('--'),
        zorder=6,
    )
    main_ax.add_patch(selection_box)

    upper_connector = ConnectionPatch(
        xyA=(x1, y1),
        coordsA=main_ax.transData,
        xyB=(0.0, 1.0),
        coordsB=zoom_ax.transAxes,
        color=('#555555'),
        linewidth=(0.8),
        linestyle=('-'),
        alpha=(0.8),
        clip_on=False,
        zorder=5,
    )
    lower_connector = ConnectionPatch(
        xyA=(x1, y0),
        coordsA=main_ax.transData,
        xyB=(0.0, 0.0),
        coordsB=zoom_ax.transAxes,
        color=('#555555'),
        linewidth=(0.8),
        linestyle=('-'),
        alpha=(0.8),
        clip_on=False,
        zorder=5,
    )

    fig.add_artist(upper_connector)
    fig.add_artist(lower_connector)


def figure3_plot_mode_main_axis(
    ax: plt.Axes,
    workload: str,
    frame: pd.DataFrame,
    style: Mapping[str, object],
    workload_label: str,
    panel_index: int,
) -> None:
    mode_values = figure3_mode_series(frame)

    ax.step(
        frame['time_hours'],
        mode_values,
        where='post',
        color=style['color'],
        linewidth=(1.2),
        label=workload_label,
        zorder=3,
    )
    ax.plot(
        frame['time_hours'],
        mode_values,
        linestyle='none',
        marker=style['marker'],
        color=style['color'],
        markersize=(2.5),
        markevery=max(
            1,
            len(frame)
            // (35),
        ),
        zorder=4,
    )

    ax.set_yticks([0, 1])
    ax.set_yticklabels(['1', '2'])
    ax.set_ylim(*((-0.3, 1.3)))
    ax.grid(
        True,
        linestyle='--',
        alpha=(0.5),
    )
    ax.margins(
        x=(0.01),
        y=(0.0),
    )
    figure3_apply_main_xlim(ax, workload)
    figure3_apply_axis_ticks(
        ax,
        workload,
        ({'generated-markovian-3000': None, 'DAS2-fs3-0-3000': None, 'DAS2-fs4-0-3000': None}),
    )

def figure3_add_mode_inset(
    fig: plt.Figure,
    main_ax: plt.Axes,
    workload: str,
    frame: pd.DataFrame,
    style: Mapping[str, object],
) -> None:
    interval = (
        figure3_last_completed_mode2_interval_indices(
            frame
        )
    )

    if interval is None:
        return

    flip_to_mode2_index, flip_to_mode1_index = interval
    padded_start, padded_end = (
        figure3_padded_inset_indices(
            frame,
            flip_to_mode2_index,
            flip_to_mode1_index,
        )
    )

    mode_values = figure3_mode_series(frame)
    inset_frame = frame.iloc[
        padded_start:padded_end + 1
    ]
    inset_y = mode_values.iloc[
        padded_start:padded_end + 1
    ]

    if len(inset_frame) < 2:
        return

    x0 = float(
        frame['time_hours'].iloc[padded_start]
    )
    x1 = float(
        frame['time_hours'].iloc[padded_end]
    )

    if x1 <= x0:
        return

    # This is a real inset axis attached to the main chart.
    # It is positioned outside the chart on the right.
    inset_ax = inset_axes(
        main_ax,
        width=('20%'),
        height=('72%'),
        loc='center left',
        bbox_to_anchor=(
            1.0 + (0.06),
            0.0,
            1.0,
            1.0,
        ),
        bbox_transform=main_ax.transAxes,
        borderpad=0.0,
    )

    inset_ax.step(
        inset_frame['time_hours'],
        inset_y,
        where='post',
        color=style['color'],
        linewidth=(1.2),
        zorder=3,
    )
    inset_ax.plot(
        inset_frame['time_hours'],
        inset_y,
        linestyle='none',
        marker=style['marker'],
        color=style['color'],
        markersize=(3.0),
        zorder=4,
    )

    inset_ax.set_xlim(x0, x1)
    inset_ax.set_xticks([x0, x1])
    inset_ax.xaxis.set_major_formatter(
        FormatStrFormatter('%.2f')
    )
    inset_ax.set_ylim(*((-0.3, 1.3)))
    inset_ax.set_yticks([0, 1])
    inset_ax.set_yticklabels(['1', '2'])
    inset_ax.tick_params(
        axis='both',
        labelsize=(8.5),
        pad=(6),
    )
    inset_ax.grid(
        True,
        linestyle='--',
        alpha=(0.5),
    )
    inset_ax.margins(x=0.0, y=0.0)

    mode1_hours, mode2_hours = (
        figure3_mode_duration_hours(frame)
    )

    # Place the accumulated durations immediately to the right
    # of the inset, aligned with modes 1 and 2.
    duration_transform = inset_ax.get_yaxis_transform()
    inset_ax.text(
        (1.06),
        (0.0),
        f'{mode1_hours:.{(2)}f} h',
        transform=duration_transform,
        ha='left',
        va='center',
        fontsize=(8.5),
        clip_on=False,
    )
    inset_ax.text(
        (1.06),
        (1.0),
        f'{mode2_hours:.{(2)}f} h',
        transform=duration_transform,
        ha='left',
        va='center',
        fontsize=(8.5),
        clip_on=False,
    )

    figure3_apply_axis_ticks(
        inset_ax,
        workload,
        ({'generated-markovian-3000': None, 'DAS2-fs3-0-3000': None, 'DAS2-fs4-0-3000': None}),
    )

    figure3_add_zoom_box_and_connectors(
        fig=fig,
        main_ax=main_ax,
        zoom_ax=inset_ax,
        x0=x0,
        x1=x1,
    )

def figure3_render_mode_figure(
    frames: Mapping[str, pd.DataFrame],
    styles: Mapping[str, Mapping[str, object]],
    workload_display_labels: Mapping[str, str],
    selections: Mapping[str, Figure3RunSelection],
    platform: str,
    timeout_label: str,
    png_path: Path,
    pdf_path: Path,
) -> None:
    workloads = list(frames.keys())

    fig = plt.figure(
        figsize=((4.0, 3.0)),
    )
    grid = GridSpec(
        nrows=len(workloads),
        ncols=1,
        figure=fig,
        hspace=(0.8),
    )

    for row_index, workload in enumerate(workloads):
        main_ax = fig.add_subplot(
            grid[row_index, 0]
        )

        frame = frames[workload]
        style = styles[workload]

        figure3_plot_mode_main_axis(
            ax=main_ax,
            workload=workload,
            frame=frame,
            style=style,
            workload_label=workload_display_labels[
                workload
            ],
            panel_index=row_index,
        )
        figure3_add_mode_inset(
            fig=fig,
            main_ax=main_ax,
            workload=workload,
            frame=frame,
            style=style,
        )

    mode_legend_handles = [
        Line2D(
            [0],
            [0],
            color=styles[workload]['color'],
            marker=styles[workload]['marker'],
            linewidth=(1.2),
            markersize=(2.5) + 2.0,
            label=workload_display_labels[workload],
        )
        for workload in workloads
    ]
    fig.legend(
        handles=mode_legend_handles,
        loc=('upper center'),
        bbox_to_anchor=(
            (0.5),
            (1.1),
        ),
        ncol=(3),
        frameon=(True),
    )

    combined_run_dirs = [
        selections[workload].run_dir
        for workload in workloads
    ]
    title_config_label = config_values_title_label(
        combined_run_dirs,
        IMPORTANT_ALGO_CONFIG_KEYS,
    )

    title_parts = [
        platform,
        f'$\\Delta_t$={timeout_label}',
    ]

    if title_config_label:
        title_parts.append(title_config_label)

    fig.subplots_adjust(
        left=(0.05),
        right=(0.75),
        top=(0.9),
        bottom=(0.075),
    )

    fig.supylabel(
        '$M(t)$',
        x=(-0.03),
        y=0.5,
    )

    fig.supxlabel(
        'Simulation time (hours)',
        x=(0.5),
        y=(-0.03),
        va='baseline',
    )

    save_figure(
        fig,
        png_path,
        pdf_path,
        pdf_kwargs={'dpi': 300},
    )
    plt.close(fig)


def figure3_render_conditions_figure(
    workloads: Sequence[str],
    frames: Mapping[str, pd.DataFrame],
    styles: Mapping[str, Mapping[str, object]],
    run_dirs: Mapping[str, Path],
    platform: str,
    timeout_label: str,
    column_titles: Mapping[str, str],
    png_path: Path,
    pdf_path: Path,
) -> None:
    """
    Render one combined 6x1 interarrival-analysis figure.

    The two workloads are stacked vertically. Each workload has
    three metric rows:
      1. Arrival coefficient of variation
      2. Arrival lag-1 autocorrelation
      3. Arrival KS distance
    """
    workloads = list(workloads)

    if len(workloads) != 2:
        raise ValueError(
            'The combined conditions figure requires exactly '
            'two workloads'
        )

    metric_specs = (
        (
            'arrival_coefficient_of_variation',
            'arrival_sample_count',
            '$\\mathrm{cv}_{A}$',
            'cv',
        ),
        (
            'arrival_lag1_autocorrelation',
            'arrival_sample_count',
            '$\\hat{\\rho}_{1,A}$',
            'lag',
        ),
        (
            'arrival_ks_distance',
            'arrival_sample_count',
            '$D_{n,A}$',
            'ks',
        ),
    )

    rows_per_workload = len(metric_specs)

    fig = plt.figure(
        figsize=((4.0, 4.0)),
    )

    # Use a nested layout so spacing is uniform within each
    # workload, while a separate larger gap is applied between
    # workload blocks.
    workload_grid = GridSpec(
        nrows=len(workloads),
        ncols=1,
        figure=fig,
        hspace=(0.15),
    )

    for workload_index, workload in enumerate(workloads):
        frame = frames[workload]
        style = styles[workload]

        metric_grid = workload_grid[
            workload_index, 0
        ].subgridspec(
            nrows=rows_per_workload,
            ncols=1,
            hspace=(0.2),
        )

        for metric_index, (
            value_column,
            sample_column,
            ylabel,
            metric_kind,
        ) in enumerate(metric_specs):
            ax = fig.add_subplot(
                metric_grid[metric_index, 0]
            )

            if metric_kind == 'cv':
                ax.axhspan(
                    1.0 - FIGURE3_CV_TOLERANCE,
                    1.0 + FIGURE3_CV_TOLERANCE,
                    color='#dfe8f3',
                    alpha=0.7,
                    zorder=0,
                )
                ax.axhline(
                    1.0,
                    color='#888888',
                    linewidth=0.8,
                    zorder=1,
                )

            elif metric_kind == 'lag':
                ax.axhspan(
                    -FIGURE3_LAG1_LIMIT,
                    FIGURE3_LAG1_LIMIT,
                    color='#dfe8f3',
                    alpha=0.7,
                    zorder=0,
                )
                ax.axhline(
                    FIGURE3_LAG1_LIMIT,
                    color='#888888',
                    linewidth=0.8,
                    linestyle=':',
                    zorder=1,
                )
                ax.axhline(
                    -FIGURE3_LAG1_LIMIT,
                    color='#888888',
                    linewidth=0.8,
                    linestyle=':',
                    zorder=1,
                )

            x = frame['time_hours']

            ax.plot(
                x,
                frame[value_column],
                color=style['color'],
                marker=style['marker'],
                linewidth=(1.0),
                markersize=(2.5),
                markevery=max(
                    1,
                    len(frame)
                    // (35),
                ),
                zorder=3,
            )

            if metric_kind == 'ks':
                ax.plot(
                    x,
                    figure3_ks_limit(
                        frame[sample_column]
                    ),
                    color=style['color'],
                    linestyle=':',
                    linewidth=(
                        (1.0)
                    ),
                    alpha=0.9,
                    zorder=2,
                )

            ax.set_ylabel(ylabel)
            ax.yaxis.set_label_coords(
                (-0.07),
                0.5,
            )
            ax.grid(
                True,
                linestyle='--',
                alpha=0.5,
            )
            ax.margins(
                x=0.01,
                y=0.08,
            )

            # Keep x tick labels only on the last metric of each
            # workload block. This preserves readable time axes
            # because the workloads may have different durations.
            if metric_index < rows_per_workload - 1:
                ax.tick_params(
                    axis='x',
                    labelbottom=False,
                )

    combined_run_dirs = [
        run_dirs[workload]
        for workload in workloads
    ]
    title_config_label = config_values_title_label(
        combined_run_dirs,
        IMPORTANT_ALGO_CONFIG_KEYS,
    )

    title_parts = [
        platform,
        f'$\\Delta_t$={timeout_label}',
    ]

    if title_config_label:
        title_parts.append(title_config_label)

    fig.subplots_adjust(
        left=(0.05),
        right=(0.98),
        top=(0.965),
        bottom=(0.065),
    )

    fig.supxlabel(
        'Simulation time (hours)',
        x=(0.5),
        y=(-0.03),
        va='baseline',
    )

    save_figure(
        fig,
        png_path,
        pdf_path,
        pdf_kwargs={'dpi': 300},
    )
    plt.close(fig)

def figure3_remove_old_figure_files(
    figure3_output_dir: Path,
) -> None:
    if not FIGURE3_REMOVE_OLD_FIGURES:
        return

    for pattern in (
        'CustomVisPaper_Figure3_*.png',
        'CustomVisPaper_Figure3_*.pdf',
    ):
        for old_path in figure3_output_dir.glob(pattern):
            old_path.unlink()


def figure3_make_markovianity_figures(
    root: Path = HEURISTIC_ROOT,
    output_dir: Path = OUTPUT_DIR,
    platform: str = FIGURE3_PLATFORM,
    workloads: Sequence[str] = FIGURE3_WORKLOADS,
    timeout: int = FIGURE3_TIMEOUT,
    alpha: float | None = FIGURE3_ALPHA,
    beta: float | None = FIGURE3_BETA,
    variant: str = FIGURE3_VARIANT,
) -> List[Tuple[Path, Path]]:
    selections = figure3_discover_run_selections(
        root=root,
        platform=platform,
        workloads=workloads,
        timeout=timeout,
        alpha=alpha,
        beta=beta,
        variant=variant,
    )

    logs = {
        workload: figure3_read_scheduler_log(
            selection.scheduler_log_path
        )
        for workload, selection
        in selections.items()
    }
    markov_windows = {
        workload: selection.markov_window_seconds
        for workload, selection
        in selections.items()
    }
    workload_display_labels = {
        workload: workload_title_label(
            [selection.run_dir]
        )
        for workload, selection
        in selections.items()
    }

    workload_styles = figure3_workload_styles(
        workloads
    )

    for workload, style in workload_styles.items():
        style['label'] = (
            workload_display_labels[workload]
        )

    timeout_label = timeout_title_label(
        [
            selection.run_dir
            for selection in selections.values()
        ]
    ).replace('$\\Delta_t$=', '')

    figure3_output_dir = (
        output_dir / 'Figure3_Markovianity'
    )
    figure3_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )
    figure3_remove_old_figure_files(
        figure3_output_dir
    )

    output_paths: List[Tuple[Path, Path]] = []

    # ----------------------------------------------------------------------
    # 1. M(t) figure (one chart per workload)
    # ----------------------------------------------------------------------
    mode_png = (
        figure3_output_dir
        / 'CustomVisPaper_Figure3_Mode_AllWorkloads.png'
    )
    mode_pdf = (
        figure3_output_dir
        / 'CustomVisPaper_Figure3_Mode_AllWorkloads.pdf'
    )

    figure3_render_mode_figure(
        frames={
            workload: logs[workload]
            for workload in workloads
        },
        styles=workload_styles,
        workload_display_labels=workload_display_labels,
        selections=selections,
        platform=platform_title_label(
            [
                selections[workload].run_dir
                for workload in workloads
            ]
        ),
        timeout_label=timeout_label,
        png_path=mode_png,
        pdf_path=mode_pdf,
    )
    write_chart_swept_config(
        mode_png,
        {
            workload: [
                selections[workload].run_dir
            ]
            for workload in workloads
        },
    )
    output_paths.append(
        (mode_png, mode_pdf)
    )

    # ----------------------------------------------------------------------
    # 2. Spare-target figure – REMOVED (per user request)
    # ----------------------------------------------------------------------
    # (The spare-target figure is no longer generated.)

    # ----------------------------------------------------------------------
    # 3. Combined interarrival-condition figure (6x1)
    # ----------------------------------------------------------------------
    condition_workloads = list(
        FIGURE3_CONDITIONS_WORKLOADS
    )

    missing_condition_workloads = [
        workload
        for workload in condition_workloads
        if workload not in logs
    ]

    if missing_condition_workloads:
        raise KeyError(
            'Missing condition workloads: '
            f'{missing_condition_workloads}'
        )

    conditions_png = (
        figure3_output_dir
        / (
            'CustomVisPaper_Figure3_Conditions_'
            'Interarrival_GeneratedMarkovian_DAS2_FS3.png'
        )
    )
    conditions_pdf = (
        figure3_output_dir
        / (
            'CustomVisPaper_Figure3_Conditions_'
            'Interarrival_GeneratedMarkovian_DAS2_FS3.pdf'
        )
    )

    figure3_render_conditions_figure(
        workloads=condition_workloads,
        frames={
            workload: logs[workload]
            for workload in condition_workloads
        },
        styles={
            workload: workload_styles[workload]
            for workload in condition_workloads
        },
        run_dirs={
            workload: selections[workload].run_dir
            for workload in condition_workloads
        },
        platform=platform_title_label(
            [
                selections[workload].run_dir
                for workload in condition_workloads
            ]
        ),
        timeout_label=timeout_label,
        column_titles=(
            FIGURE3_CONDITIONS_COLUMN_TITLES
        ),
        png_path=conditions_png,
        pdf_path=conditions_pdf,
    )
    write_chart_swept_config(
        conditions_png,
        {
            workload: [
                selections[workload].run_dir
            ]
            for workload in condition_workloads
        },
    )
    output_paths.append(
        (conditions_png, conditions_pdf)
    )

    return output_paths


# =============================================================================
# FIGURE 3 PLOT SETTINGS
# Edit these values, then rerun this cell.
# =============================================================================

# Dataset colors and markers.
# The same workload receives the same color in every figure.
COLORS = (
    '#377eb8',
    '#e41a1c',
    '#4daf4a',
    '#984ea3',
    '#ff7f00',
    '#a65628',
    '#009e73',
)
MARKERS = (
    'o',
    's',
    '^',
    'D',
    'v',
    'P',
    'H',
)

# Delete older Figure 3 PNG/PDF files before writing new outputs.
FIGURE3_REMOVE_OLD_FIGURES = True


apply_plot_style()

outputs = figure3_make_markovianity_figures(
    root=HEURISTIC_ROOT,
    output_dir=OUTPUT_DIR,
)

for png_path, _ in outputs:
    display(
        Image(filename=str(png_path))
    )

outputs

# Figure 3 plotted-value table.
# Figure 3 does not plot FCFS/B+IPM, so no baseline percentage is fabricated.
def figure3_build_plotted_values_table() -> pd.DataFrame:
    selections = figure3_discover_run_selections(
        root=HEURISTIC_ROOT,
        platform=FIGURE3_PLATFORM,
        workloads=FIGURE3_WORKLOADS,
        timeout=FIGURE3_TIMEOUT,
        alpha=FIGURE3_ALPHA,
        beta=FIGURE3_BETA,
        variant=FIGURE3_VARIANT,
    )
    rows: List[Dict[str, object]] = []
    for workload in FIGURE3_WORKLOADS:
        selection = selections[workload]
        frame = figure3_read_scheduler_log(selection.scheduler_log_path)
        mode1_hours, mode2_hours = figure3_mode_duration_hours(frame)
        total_mode_hours = mode1_hours + mode2_hours
        rows.append({
            'platform': FIGURE3_PLATFORM,
            'workload': dataset_display_label(workload),
            'algorithm': 'SNF-ICON',
            'alpha': selection.alpha,
            'beta': selection.beta,
            'timeout_seconds': selection.timeout,
            'markov_window_seconds': selection.markov_window_seconds,
            'scheduler_decisions': int(len(frame)),
            'mode_1_hours': mode1_hours,
            'mode_2_hours': mode2_hours,
            'mode_2_share_percent': (
                100.0 * mode2_hours / total_mode_hours
                if total_mode_hours > 0.0 else float('nan')
            ),
            'fallback_decisions': int(frame['fallback'].sum()),
            'fallback_share_percent': (
                100.0 * float(frame['fallback'].mean())
                if len(frame) else float('nan')
            ),
            'mean_waiting_jobs': float(frame['waiting_jobs'].mean()),
            'mean_idle_nodes': float(frame['idle_nodes'].mean()),
            'mean_spare_target_nodes': float(frame['spare_target'].mean()),
        })
    return pd.DataFrame(rows)


figure3_plotted_values_table = display_comparison_table(
    figure3_build_plotted_values_table(),
    'Figure 3 plotted values (FCFS/B+IPM is not plotted in Figure 3)',
)

## Figure 4

Pareto comparison.


In [ ]:
# Figure 4
# Run the common setup cell and Figure 1 first.
# This cell uses the Figure 1 datasets and timeout, then adds
# all SNF-ICON variants to the comparison.

# Shared Pareto mechanics are defined once in the common setup cell.

from __future__ import annotations

import matplotlib.patheffects as path_effects
from matplotlib.patches import Patch, Circle
from matplotlib.ticker import FormatStrFormatter, MultipleLocator, FuncFormatter, MaxNLocator

FIGURE4_ALGORITHM_ORDER: Tuple[str, ...] = (
    'FCFS/B+IPM',
    'SNF+IPM',
    'SNF-ICON',
    'SNF-ICON-NF',
    'SNF-ICON-NG',
    'SNF-ICON-NGNF',
    RESULT_SAVE_3_RL_ALGORITHM_LABEL,
)

def figure4_discover_records() -> List[ResultSave3BarRecord]:
    if 'result_save_3_discover_barplot_records' not in globals():
        raise RuntimeError(
            'Run Figure 1 before Figure 4. Figure 4 reuses '
            'the Figure 1 results discovery helpers.'
        )

    all_records = result_save_3_discover_barplot_records(
        include_oracle=False,
        selected_timeout=RESULT_SAVE_3_SELECTED_TIMEOUT,
        include_icon_variants=True,
    )
    return result_save_3_variant_comparison_records(
        all_records,
        timeout=RESULT_SAVE_3_SELECTED_TIMEOUT,
    )


def figure4_make_pareto(
    records: Sequence[ResultSave3BarRecord],
) -> Tuple[Path, Path]:
    if not records:
        raise ValueError(
            'No Figure 4 records supplied for Pareto plotting'
        )

    dataset_keys = sorted(
        {
            (record.platform, record.dataset)
            for record in records
        },
        key=lambda item: (
            item[1]
            == 'SDSC-BLUE-2000-4.2-cln-0-3000',
            item[0].lower(),
            item[1].lower(),
        ),
    )

    styles_from_barplot = result_save_3_style_map(
        FIGURE4_ALGORITHM_ORDER
    )
    styles = {
        algorithm: {
            'marker': (('X', '*', 'o', 's', '^', 'D', 'P'))[index],
            'size': ((60, 90, 45, 45, 45, 45, 70))[index],
        }
        for index, algorithm
        in enumerate(FIGURE4_ALGORITHM_ORDER)
    }

    panel_count = len(dataset_keys)

    panel_ncols = min(
        (2),
        panel_count,
    )
    panel_nrows = math.ceil(
        panel_count / panel_ncols
    )

    fig, axes = plt.subplots(
        nrows=panel_nrows,
        ncols=panel_ncols,
        figsize=((4.5, 6.0)),
        squeeze=False,
    )
    panel_axes = axes.ravel()

    for unused_ax in panel_axes[panel_count:]:
        unused_ax.set_visible(False)

    for panel_index, (
        ax,
        (platform, dataset),
    ) in enumerate(zip(panel_axes, dataset_keys)):
        dataset_records = [
            record
            for record in records
            if (
                record.platform == platform
                and record.dataset == dataset
            )
        ]
        lookup = {
            record.algorithm: record
            for record in dataset_records
        }

        required_algorithms = tuple(
            algorithm
            for algorithm in FIGURE4_ALGORITHM_ORDER
            if algorithm != RESULT_SAVE_3_RL_ALGORITHM_LABEL
        )
        has_rl = (
            RESULT_SAVE_3_RL_ALGORITHM_LABEL
            in lookup
        )

        missing = [
            algorithm
            for algorithm in required_algorithms
            if algorithm not in lookup
        ]
        if missing:
            raise RuntimeError(
                f'Figure 4 is missing {", ".join(missing)} '
                f'for {platform}/{dataset}'
            )

        coordinates = [
            (
                lookup[algorithm].waiting_seconds,
                lookup[algorithm].wasted_energy_mwh,
            )
            for algorithm in required_algorithms
        ]
        frontier = pareto_frontier(
            coordinates
        )
        frontier_coordinates = set(frontier)

        for algorithm in required_algorithms:
            record = lookup[algorithm]
            style = styles[algorithm]

            coordinate = (
                record.waiting_seconds,
                record.wasted_energy_mwh,
            )
            is_pareto = coordinate in frontier_coordinates

            ax.scatter(
                record.waiting_seconds,
                record.wasted_energy_mwh,
                s=style['size'],
                marker=style['marker'],
                facecolors=(
                    (plt.get_cmap('cividis')(0.0))
                    if is_pareto
                    else 'none'
                ),
                edgecolors=(
                    (plt.get_cmap('cividis')(0.0))
                    if is_pareto
                    else (plt.get_cmap('cividis')(0.8))
                ),
                hatch=(
                    None if is_pareto
                    else ('//////')
                ),
                label=algorithm,
                zorder=3,
            )

        plot_coordinates = list(coordinates)
        if has_rl:
            rl_record = lookup[RESULT_SAVE_3_RL_ALGORITHM_LABEL]
            rl_style = styles[RESULT_SAVE_3_RL_ALGORITHM_LABEL]
            plot_coordinates.append(
                (rl_record.waiting_seconds, rl_record.wasted_energy_mwh)
            )
            ax.scatter(
                rl_record.waiting_seconds,
                rl_record.wasted_energy_mwh,
                s=rl_style['size'],
                marker=rl_style['marker'],
                facecolors='none',
                edgecolors=(plt.get_cmap('cividis')(0.8)),
                hatch=('//////'),
                label=RESULT_SAVE_3_RL_ALGORITHM_LABEL,
                zorder=6,
            )

        if len(frontier) >= 2:
            ax.plot(
                [
                    coordinate[0]
                    for coordinate in frontier
                ],
                [
                    coordinate[1]
                    for coordinate in frontier
                ],
                color=(plt.get_cmap('cividis')(0.0)),
                linestyle=('--'),
                linewidth=(1.0),
                label='Pareto frontier',
                zorder=4,
            )

        ax.set_title(
            f"({chr(ord('a') + panel_index)}) "
            f"{platform}\n"
            f"{dataset_display_label(dataset)}",
            y=(1.02),
        )
        visible_fraction_control = (
            ({panel_index: {'x': 0.25, 'y': 0.25} for panel_index in range(16)})
            .setdefault(panel_index, {'x': 0.25, 'y': 0.25})
        )
        outlier_region_control = (
            ({0: {'x': 0.25, 'y': 0.25}, 1: {'x': 0.25, 'y': 0.25}, 2: {'x': 0.25, 'y': 0.25}, 3: {'x': 0.25, 'y': 0.25}, 4: {'x': 0.25, 'y': 0.25}, 5: {'x': 0.25, 'y': 0.25}, 6: {'x': 0.25, 'y': 0.25}, 7: {'x': 0.25, 'y': 0.25}, 8: {'x': 0.25, 'y': 0.25}, 9: {'x': 0.25, 'y': 0.25}, 10: {'x': 0.25, 'y': 0.25}, 11: {'x': 0.25, 'y': 0.25}, 12: {'x': 0.25, 'y': 0.25}, 13: {'x': 0.25, 'y': 0.25}, 14: {'x': 0.25, 'y': 0.25}, 15: {'x': 0.25, 'y': 0.25}})[panel_index]
        )
        pareto_finalize_panel(
            ax,
            plot_coordinates,
            x_margin=(0.15),
            y_margin=(0.15),
            visible_fraction_threshold=visible_fraction_control,
            outlier_region_fraction=outlier_region_control,
            tick_decimals=1,
            integer_tolerance=1e-6,
        )

    handles: List[object] = []
    labels: List[str] = []
    seen_labels = set()

    for ax in panel_axes[:panel_count]:
        axis_handles, axis_labels = ax.get_legend_handles_labels()
        for handle, label in zip(axis_handles, axis_labels):
            if label in seen_labels:
                continue
            seen_labels.add(label)
            handles.append(handle)
            labels.append(label)

    label_to_handle: Dict[str, object] = dict(zip(labels, handles))

    # ================ FIX: Use configurable legend marker sizes ================
    # Now using the global variable FIGURE4_LEGEND_MARKER_SIZE_COMPENSATION
    # ==========================================================================

    for algorithm in FIGURE4_ALGORITHM_ORDER:
        if algorithm not in label_to_handle:
            continue
        style = styles[algorithm]
        marker = style['marker']
        # Use compensation dict, fallback to base size if marker not found
        size = ({'X': 6.0, '*': 9.0, 'o': 5.5, 's': 5.5, '^': 5.5, 'D': 5.5, 'P': 7.0}).get(marker, (6.0))
        label_to_handle[algorithm] = Line2D(
            [0], [0],
            linestyle='none',
            marker=marker,
            markerfacecolor=('#333333'),
            markeredgecolor=('#333333'),
            markersize=size,
            label=algorithm,
        )

    # Both Pareto and Non-Pareto use Circle patches with same radius
    circle_radius = (6.0) / 20

    label_to_handle['Pareto'] = Circle(
        (0.5, 0.5),
        radius=circle_radius,
        facecolor=(plt.get_cmap('cividis')(0.0)),
        edgecolor=(plt.get_cmap('cividis')(0.0)),
        label='Pareto',
    )
    label_to_handle['Non-Pareto'] = Circle(
        (0.5, 0.5),
        radius=circle_radius,
        facecolor='none',
        edgecolor=(plt.get_cmap('cividis')(0.8)),
        hatch=('//////'),
        label='Non-Pareto',
    )

    blank_handle = Line2D([0], [0], linestyle='none', marker='', label='')

    column_order: List[List[str]] = [
        ['SNF-ICON', 'SNF-ICON-NF', 'SNF-ICON-NG', 'SNF-ICON-NGNF'],
        ['FCFS/B+IPM', 'SNF+IPM', RESULT_SAVE_3_RL_ALGORITHM_LABEL],
        ['Pareto frontier', 'Pareto', 'Non-Pareto'],
    ]

    rows_per_column = max(len(column) for column in column_order)

    legend_handles: List[object] = []
    legend_labels = []
    for column in column_order:
        for row_index in range(rows_per_column):
            if row_index < len(column):
                label = column[row_index]
                if label not in label_to_handle:
                    legend_handles.append(blank_handle)
                    legend_labels.append('')
                    continue
                legend_handles.append(label_to_handle[label])
                legend_labels.append(label)
            else:
                legend_handles.append(blank_handle)
                legend_labels.append('')

    fig.supxlabel(
        'Mean waiting time (seconds)',
        y=(-0.09),
    )
    fig.supylabel(
        'Energy waste (MWh)',
        x=(-0.03),
    )

    fig.legend(
        handles=legend_handles,
        labels=legend_labels,
        loc=('upper center'),
        bbox_to_anchor=((0.5, 1.25)),
        ncol=len(column_order),
        frameon=(True),
        columnspacing=(2.0),
        handlelength=(1.0),
        handleheight=(1.0),   # new: make square
        handletextpad=(0.5),
        labelspacing=(0.5),
    )

    fig.subplots_adjust(
        left=(0.1),
        right=(0.99),
        top=(1.0),
        bottom=(0.0),
        wspace=(0.25),
        hspace=(0.5),
    )

    output_dir = OUTPUT_DIR / 'Figure4_Pareto'
    output_dir.mkdir(parents=True, exist_ok=True)
    png_path = output_dir / 'CustomVisPaper_Figure4_Pareto.png'
    pdf_path = output_dir / 'CustomVisPaper_Figure4_Pareto.pdf'

    save_figure(
        fig,
        png_path,
        pdf_path,
        png_kwargs={},
        pdf_kwargs={},
    )
    plt.close(fig)

    write_chart_swept_config(
        png_path,
        {
            f'{platform} / {dataset}': [
                record.metrics_path.parent
                for record in records
                if (
                    record.platform == platform
                    and record.dataset == dataset
                )
            ]
            for platform, dataset in dataset_keys
        },
    )

    return png_path, pdf_path


apply_plot_style()
figure4_records = figure4_discover_records()
png_path, pdf_path = figure4_make_pareto(figure4_records)
display(Image(filename=str(png_path)))
(png_path, pdf_path)

# Figure 4 comparison table
def figure4_build_comparison_table(
    records: Sequence[ResultSave3BarRecord],
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    group_keys = sorted(
        {(record.platform, record.dataset) for record in records},
        key=lambda key: (key[0].lower(), key[1].lower()),
    )
    for platform, dataset in group_keys:
        group = [r for r in records if r.platform == platform and r.dataset == dataset]
        baseline_matches = [r for r in group if r.algorithm == 'FCFS/B+IPM']
        if len(baseline_matches) != 1:
            raise RuntimeError(
                f'Expected one FCFS/B+IPM baseline for {platform} / {dataset}; '
                f'found {len(baseline_matches)}.'
            )
        baseline = baseline_matches[0]
        frontier_coordinates = set(
            pareto_frontier([
                (record.waiting_seconds, record.wasted_energy_mwh)
                for record in group
            ])
        )
        for record in group:
            rows.append({
                'platform': platform,
                'workload': dataset_display_label(dataset),
                'algorithm': (
                    COMPARISON_BASELINE_DISPLAY_LABEL
                    if record.algorithm == 'FCFS/B+IPM'
                    else record.algorithm
                ),
                'waiting_time_seconds': record.waiting_seconds,
                'energy_waste_mwh': record.wasted_energy_mwh,
                'is_pareto': (
                    record.waiting_seconds,
                    record.wasted_energy_mwh,
                ) in frontier_coordinates,
                'waiting_time_improvement_vs_fcfs_b_ipm_percent': (
                    comparison_improvement_percent(
                        record.waiting_seconds,
                        baseline.waiting_seconds,
                    )
                ),
                'energy_waste_improvement_vs_fcfs_b_ipm_percent': (
                    comparison_improvement_percent(
                        record.wasted_energy_mwh,
                        baseline.wasted_energy_mwh,
                    )
                ),
            })
    return add_explicit_fcfs_b_ipm_comparisons(
        pd.DataFrame(rows),
        group_columns=['platform', 'workload'],
        metric_specs=[
            (
                'waiting_time_seconds',
                'fcfs_b_ipm_waiting_time_seconds',
                'waiting_time_difference_seconds',
                'waiting_time_improvement_percent',
                'waiting_time_comparison',
                False,
            ),
            (
                'energy_waste_mwh',
                'fcfs_b_ipm_energy_waste_mwh',
                'energy_waste_difference_mwh',
                'energy_waste_improvement_percent',
                'energy_waste_comparison',
                False,
            ),
        ],
    )

figure4_comparison_table = display_comparison_table(
    figure4_build_comparison_table(figure4_records),
    'Figure 4 comparison table',
)


## Figure 5

Waiting-time distributions.


In [ ]:
# Figure 5
# Run the common setup cell first. This cell contains Figure 5-specific code and settings.

from __future__ import annotations

HEURISTIC_ORACLE_ROOT = RESULT_SAVE_3_ROOT / 'Heuristic-IPM' / 'Experiment-64'
HEURISTIC_ROOT = RESULT_SAVE_3_ROOT / 'Heuristic-ICON'
SNF_SSC_HEURISTIC_LABEL = 'SNF-ICON'
SNF_SSC_HEURISTIC_RUN_PREFIX = 'snf_icon'

FIGURE5_ICON_VARIANTS: Dict[str, str] = {
    'SNF-ICON': 'snf-ssc',
    'SNF-ICON-NG': 'snf-ssc-ng',
    'SNF-ICON-NF': 'snf-ssc-nf',
    'SNF-ICON-NGNF': 'snf-ssc-ngnf',
}
STARVATION_OUTPUT_DIR = OUTPUT_DIR
SECONDS_PER_MINUTE = 60.0
ORACLE_LABEL = 'SNF Oracle'
ORACLE_RUN_DIR = 'snf_oracle_psas_no-timeout'
BASE_ALGORITHM_ORDER: Tuple[str, ...] = ('FCFS/B+IPM', 'SNF+IPM', 'SNF-ICON', 'SNF-ICON-NG', 'SNF-ICON-NF', 'SNF-ICON-NGNF')
ALGORITHM_ORDER: Tuple[str, ...] = BASE_ALGORITHM_ORDER + (ORACLE_LABEL,)
STARVATION_ZERO_WAIT_MINUTES = 1.0 / 60.0
STARVATION_SCENARIOS: Tuple[Tuple[str, str, int], ...] = (('AOBA-64', 'DAS2-fs1-0-3000', 7200), ('AOBA-64', 'generated-markovian-3000', 7200))
STRUCTURAL_DIRS = {'generated_configs', 'metrics_comparison', 'plots', 'analysis_figures', 'publication_figures', '__pycache__'}

@dataclass(frozen=True)
class StarvationRun:
    platform: str
    workload: str
    timeout: int
    algorithm: str
    run_dir: Path

SOURCE_DIR_RECORDS: List[Dict[str, str]] = []
SOURCE_DIR_SEEN: set[Tuple[str, str]] = set()

def source_root_label(path: Path) -> str:
    resolved = path.expanduser().resolve()
    roots = (('Heuristic-IPM', HEURISTIC_ORACLE_ROOT), ('Heuristic-ICON', HEURISTIC_ROOT))
    for label, root in roots:
        try:
            resolved.relative_to(root.expanduser().resolve())
            return label
        except ValueError:
            continue
    return 'Other'

def platform_node_count(platform: str) -> int:
    match = re.search('-(\\d+)(?:-|$)', platform)
    if match is None:
        raise ValueError(f'Could not infer node count from platform name: {platform}')
    return int(match.group(1))

def figure5_icon_run_dir(
    algorithm: str,
    platform: str,
    workload: str,
    timeout: int,
) -> Path:
    variant = FIGURE5_ICON_VARIANTS[algorithm]
    return (
        HEURISTIC_ROOT
        / f'{variant}_{platform_node_count(platform)}'
        / platform
        / workload
        / f'{SNF_SSC_HEURISTIC_RUN_PREFIX}_timeout-{timeout}'
    )

def active_algorithm_order(include_oracle: bool) -> Tuple[str, ...]:
    return ALGORITHM_ORDER if include_oracle else BASE_ALGORITHM_ORDER

def extract_timeout(run_name: str) -> int | None:
    match = re.search('(?:^|_)timeout-(\\d+)(?:_|$)', run_name, flags=re.IGNORECASE)
    return int(match.group(1)) if match else None

def algorithm_label(run_name: str) -> str:
    lower = run_name.lower()
    normalized = lower.replace('_', '-')
    if lower.startswith('snf_oracle') or normalized.startswith('snf-oracle'):
        return ORACLE_LABEL
    if normalized.startswith('snf-ssc-nfng') or normalized.startswith('snf-ssc-ngnf'):
        return 'SNF-ICON-NGNF'
    if normalized.startswith('snf-ssc-ng'):
        return 'SNF-ICON-NG'
    if normalized.startswith('snf-ssc-nf'):
        return 'SNF-ICON-NF'
    if normalized.startswith('snf-ssc'):
        return 'SNF-ICON'
    if lower.startswith('snf_psas') or lower.startswith('snf_ipm'):
        return 'SNF+IPM'
    if lower.startswith('easy_psas') or lower.startswith('easy_ipm'):
        return 'FCFS/B+IPM'
    raise ValueError(f'Unknown run directory name: {run_name}')

def strict_numeric(series: pd.Series, column: str, path: Path) -> pd.Series:
    if series.isna().any():
        rows = series.index[series.isna()].tolist()
        raise ValueError(f'Null value in {path}, column {column!r}, rows {rows}')
    try:
        numeric = pd.to_numeric(series, errors='raise')
    except (TypeError, ValueError) as exc:
        raise ValueError(f'Non-numeric value in {path}, column {column!r}') from exc
    values = numeric.to_numpy(dtype=float)
    invalid = ~np.isfinite(values)
    if invalid.any():
        rows = numeric.index[invalid].tolist()
        raise ValueError(f'Non-finite value in {path}, column {column!r}, rows {rows}')
    return numeric.astype(float)

IMPORTANT_ALGO_CONFIG_KEYS: Tuple[str, ...] = ('run.algo_config.alpha', 'run.algo_config.beta', MARKOV_WINDOW_CONFIG_KEY)
FIGURE_CONFIG_RESULT_DIRS: Dict[Path, set[Path]] = {}

def unique_config_numeric_values(result_dirs: Sequence[Path], key: str, *, skip_missing: bool=False) -> List[float]:
    values: List[float] = []
    for result_dir in result_dirs:
        try:
            value = config_numeric_value(result_dir, key)
        except KeyError:
            if skip_missing:
                continue
            raise
        if not any((abs(value - existing) <= 1e-09 * max(1.0, abs(value), abs(existing)) for existing in values)):
            values.append(value)
    values.sort()
    return values

def timeout_title_label(result_dirs: Sequence[Path]) -> str:
    values = unique_config_numeric_values(result_dirs, SCHEDULER_TIMEOUT_CONFIG_KEY, skip_missing=True)
    if not values:
        raise KeyError('No run.algo_config.timeout value found for title generation')
    hours = [value / 3600.0 for value in values]
    if len(hours) == 1:
        return f'$\\Delta_t$={hours[0]:g} h'
    return f'$\\Delta_t\\in[{hours[0]:g}, {hours[-1]:g}]$ h'

def figure5_assign_resource_bins(resources: pd.Series) -> Tuple[pd.Series, List[str]]:
    numeric = pd.to_numeric(resources, errors='raise').astype(float)
    values = numeric.to_numpy(dtype=float)
    if values.size == 0 or not np.isfinite(values).all():
        raise ValueError('Resource requests must be finite and non-empty')
    maximum = int(max(1, math.ceil(float(values.max()))))
    conditions: List[pd.Series] = [numeric == 1]
    labels: List[str] = ['1']
    if maximum >= 2:
        conditions.append(numeric == 2)
        labels.append('2')
    low = 3
    high = 4
    while low <= maximum:
        conditions.append((numeric >= low) & (numeric <= high))
        labels.append(f'{low}-{high}')
        low = high + 1
        high *= 2
    assigned = np.select(conditions, labels, default='Other')
    return (pd.Series(assigned, index=resources.index), labels)

def figure5_box_stats(values: Sequence[float], label: str) -> Dict[str, object] | None:
    array = np.asarray(list(values), dtype=float)
    array = array[np.isfinite(array)]
    if array.size == 0:
        return None
    q05, q25, q50, q75, q95 = np.percentile(array, [5, 25, 50, 75, 95])
    return {'label': label, 'whislo': float(q05), 'q1': float(q25), 'med': float(q50), 'q3': float(q75), 'whishi': float(q95), 'fliers': []}

def figure5_discover_runs(root: Path=HEURISTIC_ORACLE_ROOT, include_oracle: bool=True) -> List[StarvationRun]:
    root = root.expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f'Heuristic-Oracle root does not exist: {root}')
    discovered: Dict[Tuple[str, str, int, str], StarvationRun] = {}
    for platform_dir in sorted(root.iterdir(), key=lambda path: path.name.lower()):
        if not platform_dir.is_dir() or platform_dir.name.startswith('.') or platform_dir.name in STRUCTURAL_DIRS:
            continue
        for workload_dir in sorted(platform_dir.iterdir(), key=lambda path: path.name.lower()):
            if not workload_dir.is_dir() or workload_dir.name.startswith('.') or workload_dir.name in STRUCTURAL_DIRS:
                continue
            for run_dir in sorted(workload_dir.iterdir(), key=lambda path: path.name.lower()):
                if not run_dir.is_dir() or run_dir.name.startswith('.') or run_dir.name in STRUCTURAL_DIRS:
                    continue
                try:
                    algorithm = algorithm_label(run_dir.name)
                except ValueError:
                    continue
                if algorithm == ORACLE_LABEL:
                    continue
                if algorithm == SNF_SSC_HEURISTIC_LABEL:
                    continue
                timeout = extract_timeout(run_dir.name)
                if timeout is None:
                    continue
                timeout = validated_config_timeout_seconds(run_dir, timeout, 'Figure 5 run directory name')
                if algorithm not in active_algorithm_order(include_oracle):
                    continue
                key = (platform_dir.name, workload_dir.name, timeout, algorithm)
                if key in discovered:
                    raise RuntimeError(f'Duplicate starvation run for {platform_dir.name}/{workload_dir.name}/timeout-{timeout}/{algorithm}:\n- {discovered[key].run_dir}\n- {run_dir}')
                discovered[key] = StarvationRun(platform=platform_dir.name, workload=workload_dir.name, timeout=timeout, algorithm=algorithm, run_dir=run_dir)
    for platform, workload, timeout in STARVATION_SCENARIOS:
        for algorithm in FIGURE5_ICON_VARIANTS:
            icon_run_dir = figure5_icon_run_dir(
                algorithm,
                platform,
                workload,
                timeout,
            )
            validated_config_timeout_seconds(
                icon_run_dir,
                timeout,
                'Figure 5 Heuristic-ICON run directory name',
            )
            discovered[
                platform,
                workload,
                timeout,
                algorithm,
            ] = StarvationRun(
                platform=platform,
                workload=workload,
                timeout=timeout,
                algorithm=algorithm,
                run_dir=icon_run_dir,
            )
    if include_oracle:
        for platform, workload, timeout in STARVATION_SCENARIOS:
            discovered[platform, workload, timeout, ORACLE_LABEL] = StarvationRun(platform=platform, workload=workload, timeout=timeout, algorithm=ORACLE_LABEL, run_dir=root / platform / workload / ORACLE_RUN_DIR)
    runs = list(discovered.values())
    runs.sort(key=lambda run: (run.platform.lower(), run.workload.lower(), run.timeout, ALGORITHM_ORDER.index(run.algorithm)))
    if not runs:
        raise RuntimeError(f'No starvation-compatible heuristic-oracle runs found under {root}')
    return runs

def figure5_read_jobs(run: StarvationRun) -> pd.DataFrame:
    raw_path = run.run_dir / 'raw_job_log.csv'
    wait_path = run.run_dir / 'waiting_time_log.csv'
    if not raw_path.is_file():
        raise FileNotFoundError(f'Missing job log: {raw_path}')
    if not wait_path.is_file():
        raise FileNotFoundError(f'Missing waiting-time log: {wait_path}')
    register_source_file(raw_path, 'raw_job_log')
    register_source_file(wait_path, 'waiting_time_log')
    raw = pd.read_csv(raw_path)
    wait = pd.read_csv(wait_path)
    if raw.empty:
        raise ValueError(f'Empty job log: {raw_path}')
    if wait.empty:
        raise ValueError(f'Empty waiting-time log: {wait_path}')
    require_columns(raw, ('job_id', 'res'), raw_path)
    require_columns(wait, ('job_id', 'waiting_time'), wait_path)
    if raw['job_id'].isna().any():
        raise ValueError(f'Null job_id in {raw_path}')
    if wait['job_id'].isna().any():
        raise ValueError(f'Null job_id in {wait_path}')
    if raw['job_id'].duplicated().any():
        duplicates = raw.loc[raw['job_id'].duplicated(keep=False), 'job_id'].tolist()
        raise ValueError(f'Duplicate job_id in {raw_path}: {duplicates}')
    if wait['job_id'].duplicated().any():
        duplicates = wait.loc[wait['job_id'].duplicated(keep=False), 'job_id'].tolist()
        raise ValueError(f'Duplicate job_id in {wait_path}: {duplicates}')
    raw_ids = set(raw['job_id'].tolist())
    wait_ids = set(wait['job_id'].tolist())
    if raw_ids != wait_ids:
        missing_wait = sorted(raw_ids - wait_ids, key=str)
        missing_raw = sorted(wait_ids - raw_ids, key=str)
        raise ValueError(f'Job IDs do not match between starvation logs. Missing in waiting log: {missing_wait}; missing in raw log: {missing_raw}')
    jobs = raw[['job_id', 'res']].merge(wait[['job_id', 'waiting_time']], on='job_id', how='left', sort=False, validate='one_to_one')
    jobs['res'] = strict_numeric(jobs['res'], 'res', raw_path)
    jobs['waiting_time'] = strict_numeric(jobs['waiting_time'], 'waiting_time', wait_path)
    if (jobs['res'] <= 0.0).any():
        bad = jobs.loc[jobs['res'] <= 0.0, ['job_id', 'res']]
        raise ValueError(f'Requested resources must be > 0. Bad rows:\n{bad.to_string(index=False)}')
    if (jobs['waiting_time'] < 0.0).any():
        bad = jobs.loc[jobs['waiting_time'] < 0.0, ['job_id', 'waiting_time']]
        raise ValueError(f'Waiting time must be >= 0. Bad rows:\n{bad.to_string(index=False)}')
    return jobs

def figure5_make_starvation_boxplots(root: Path=HEURISTIC_ORACLE_ROOT, output_dir: Path=STARVATION_OUTPUT_DIR, include_oracle: bool=True) -> int:
    algorithm_order = active_algorithm_order(include_oracle)
    runs = figure5_discover_runs(root, include_oracle)
    runs_by_key: Dict[Tuple[str, str, int], List[StarvationRun]] = {}
    for run in runs:
        runs_by_key.setdefault((run.platform, run.workload, run.timeout), []).append(run)
    selected_groups: List[Tuple[str, str, int, List[StarvationRun]]] = []
    for platform, workload, timeout in STARVATION_SCENARIOS:
        key = (platform, workload, timeout)
        if key not in runs_by_key:
            raise FileNotFoundError(f'Missing starvation scenario: {platform} / {workload} / timeout {timeout}s')
        group = sorted(runs_by_key[key], key=lambda run: algorithm_order.index(run.algorithm))
        missing_algorithms = [algorithm for algorithm in algorithm_order if algorithm not in {run.algorithm for run in group}]
        if missing_algorithms:
            raise RuntimeError(f"Missing algorithms for starvation scenario {platform} / {workload} / timeout {timeout}s: {', '.join(missing_algorithms)}")
        selected_groups.append((platform, workload, timeout, group))
    figure5_output_dir = output_dir / 'Figure5_Starvation'
    figure5_output_dir.mkdir(parents=True, exist_ok=True)
    panel_data: List[Dict[str, object]] = []
    all_resources: List[pd.Series] = []
    for platform, workload, timeout, group in selected_groups:
        data_by_algorithm: Dict[str, pd.DataFrame] = {}
        for run in group:
            jobs = figure5_read_jobs(run)
            subset = jobs[['res', 'waiting_time']].copy()
            data_by_algorithm[run.algorithm] = subset
            all_resources.append(subset['res'])
        if not data_by_algorithm:
            raise RuntimeError(f'No starvation data for {platform} / {workload} / {timeout}')
        panel_data.append({'platform': platform, 'workload': workload, 'timeout': timeout, 'data_by_algorithm': data_by_algorithm})
    combined_resources = pd.concat(all_resources, ignore_index=True)
    _, resource_labels = figure5_assign_resource_bins(combined_resources)

    fig, axes = plt.subplots(2, 1, figsize=((8.592, 3)), sharex=True, squeeze=False)
    panel_axes = axes[:, 0]

    group_width = (0.75)
    box_width = group_width / len(algorithm_order)
    legend_handles: List[Patch] = []
    summary_rows: List[Dict[str, object]] = []
    for style_index, algorithm in enumerate(algorithm_order):
        color = (('#377eb8', '#e41a1c', '#4daf4a', '#984ea3', '#ff7f00', '#a65628', '#009e73'))[style_index % len((('#377eb8', '#e41a1c', '#4daf4a', '#984ea3', '#ff7f00', '#a65628', '#009e73')))]
        hatch = (('///', '\\\\', 'xxx', '---', '+++', '...', 'ooo'))[style_index % len((('///', '\\\\', 'xxx', '---', '+++', '...', 'ooo')))]
        legend_handles.append(Patch(facecolor='white', edgecolor=color, hatch=hatch, label=algorithm))

    for panel_index, (ax, panel) in enumerate(zip(panel_axes, panel_data)):
        platform = panel['platform']
        workload = panel['workload']
        timeout = panel['timeout']
        data_by_algorithm = panel['data_by_algorithm']
        counts_by_algorithm: Dict[str, Dict[str, int]] = {}
        for algorithm in algorithm_order:
            subset = data_by_algorithm[algorithm].copy()
            subset['resource_bin'], _ = figure5_assign_resource_bins(subset['res'])
            counts_by_algorithm[algorithm] = {resource_label: int((subset['resource_bin'] == resource_label).sum()) for resource_label in resource_labels}
            style_index = algorithm_order.index(algorithm)
            color = (('#377eb8', '#e41a1c', '#4daf4a', '#984ea3', '#ff7f00', '#a65628', '#009e73'))[style_index % len((('#377eb8', '#e41a1c', '#4daf4a', '#984ea3', '#ff7f00', '#a65628', '#009e73')))]
            hatch = (('///', '\\\\', 'xxx', '---', '+++', '...', 'ooo'))[style_index % len((('///', '\\\\', 'xxx', '---', '+++', '...', 'ooo')))]
            for bin_index, resource_label in enumerate(resource_labels):
                waits_seconds = subset.loc[subset['resource_bin'] == resource_label, 'waiting_time']
                waits_minutes = waits_seconds / SECONDS_PER_MINUTE
                display_waits = waits_minutes.clip(lower=STARVATION_ZERO_WAIT_MINUTES)
                stats = figure5_box_stats(display_waits, resource_label)
                if stats is None:
                    continue
                position = bin_index + (style_index - (len(algorithm_order) - 1) / 2.0) * box_width
                artists = ax.bxp([stats], positions=[position], widths=box_width * (0.88), patch_artist=True, showfliers=False, manage_ticks=False)
                for box in artists['boxes']:
                    box.set(facecolor='white', edgecolor=color, hatch=hatch, linewidth=(1.2))
                for median in artists['medians']:
                    median.set(color=('#d62728'), linewidth=(1.3))
                for artist_name in ('whiskers', 'caps'):
                    for artist in artists[artist_name]:
                        artist.set(color=color, linewidth=(1.0))
                summary_rows.append({'platform': platform, 'workload': workload, 'timeout_seconds': timeout, 'algorithm': algorithm, 'resource_bin': resource_label, 'jobs': int(len(waits_seconds)), 'wait_q05_minutes': stats['whislo'], 'wait_q25_minutes': stats['q1'], 'wait_median_minutes': stats['med'], 'wait_q75_minutes': stats['q3'], 'wait_q95_minutes': stats['whishi']})
        resource_count_labels: List[str] = []
        for resource_label in resource_labels:
            per_algorithm_counts = [counts_by_algorithm[algorithm][resource_label] for algorithm in algorithm_order]
            if len(set(per_algorithm_counts)) == 1:
                count_value = per_algorithm_counts[0]
            else:
                count_value = sum(per_algorithm_counts)
            resource_count_labels.append(f'n={count_value}')
        ax.set_yscale('log')
        ax.grid(axis='y', which='major', linestyle='--', alpha=(0.55))
        panel_config_label = config_values_title_label(
            [run.run_dir for run in group],
            IMPORTANT_ALGO_CONFIG_KEYS,
        )
        panel_title = (
            f'{platform} / '
            f'{dataset_display_label(workload)} / '
            f'$\\Delta_t$={timeout / 3600.0:g} h'
        )
        if panel_config_label:
            panel_title = f'{panel_title} / {panel_config_label}'
        ax.set_title(
            panel_title,
            x=(0.01),
            y=(0.94),
            ha='left',
            va='bottom',
        )
        for bin_index, count_label in enumerate(resource_count_labels):
            ax.text(bin_index + (-0.52), (0.5),
                    count_label, transform=ax.get_xaxis_transform(),
                    ha='left', va='center', rotation='vertical')
        if panel_index < len(panel_axes) - 1:
            ax.tick_params(labelbottom=False)

    panel_axes[-1].set_xticks(np.arange(len(resource_labels)))
    panel_axes[-1].set_xticklabels(resource_labels)
    panel_axes[-1].set_xlabel('Job size (requested nodes)')
    fig.supylabel('Waiting time (min, log scale)', x=(-0.07))
    panel_axes[0].legend(handles=legend_handles,
                         ncol=len(algorithm_order),
                         frameon=(True),
                         loc=('upper center'),
                         bbox_to_anchor=((0.5, 1.45)),
                         handlelength=(2.0),
                         handletextpad=(0.8),
                         columnspacing=(1.0),
                         labelspacing=(0.5),
                         borderpad=(0.4))

    fig.subplots_adjust(left=(0.0), right=(1.0),
                        top=(1.0), bottom=(0.0),
                        hspace=(0.2))
    stem = 'CustomVisPaper_Figure5_Starvation_2panel_2hour'
    png_path = figure5_output_dir / f'{stem}.png'
    pdf_path = figure5_output_dir / f'{stem}.pdf'
    csv_path = figure5_output_dir / f'{stem}.csv'
    save_figure(
        fig,
        png_path,
        pdf_path,
        png_kwargs={},
        pdf_kwargs={},
    )
    plt.close(fig)
    pd.DataFrame(summary_rows).to_csv(csv_path, index=False)
    write_chart_swept_config(png_path, {f'{platform} / {workload} / timeout-{timeout}': [run.run_dir for run in group] for platform, workload, timeout, group in selected_groups})
    print(f'Wrote {png_path}')
    print(f'Wrote {pdf_path}')
    print(f'Wrote {csv_path}')
    return 1


apply_plot_style()
figure5_make_starvation_boxplots(
    root=HEURISTIC_ORACLE_ROOT,
    output_dir=STARVATION_OUTPUT_DIR,
    include_oracle=False,
)
png_path = OUTPUT_DIR / "Figure5_Starvation" / "CustomVisPaper_Figure5_Starvation_2panel_2hour.png"
display(Image(filename=str(png_path)))
png_path

def figure5_build_comparison_table() -> pd.DataFrame:
    csv_path = OUTPUT_DIR / 'Figure5_Starvation' / 'CustomVisPaper_Figure5_Starvation_2panel_2hour.csv'
    if not csv_path.is_file():
        raise FileNotFoundError(f'Run Figure 5 first. Missing {csv_path}')
    frame = pd.read_csv(csv_path)
    return add_explicit_fcfs_b_ipm_comparisons(
        frame,
        group_columns=['platform', 'workload', 'timeout_seconds', 'resource_bin'],
        metric_specs=[('wait_median_minutes', 'fcfs_b_ipm_wait_median_minutes',
                       'wait_median_difference_minutes', 'wait_median_improvement_percent',
                       'wait_median_comparison', False)],
    )

figure5_comparison_table = display_comparison_table(
    figure5_build_comparison_table(),
    'Figure 5 comparison table',
)

## Figure 6

Reward sweep.


In [ ]:
# Figure 6
# Run the common setup cell first. This cell contains Figure 6-specific code and settings.

# Shared Pareto mechanics are defined once in the common setup cell.

from __future__ import annotations

from copy import copy
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch, Circle
from matplotlib.ticker import FuncFormatter, MaxNLocator

HEURISTIC_ORACLE_ROOT = RESULT_SAVE_3_ROOT / 'Heuristic-IPM' / 'Experiment-64'

SWEEP_ROOT = RESULT_SAVE_3_ROOT / 'Reward-Sweep'
ORACLE_LABEL = 'SNF Oracle'
ORACLE_RUN_DIR = 'snf_oracle_psas_no-timeout'
FIGURE6_PLATFORM = 'AOBA-64'
SWEEP_TIMEOUT = 7200
SWEEP_WORKLOADS = ('DAS2-fs1-0-3000','DAS2-fs2-0-3000','DAS2-fs3-0-3000', 'DAS2-fs4-0-3000','generated-markovian-3000')
SWEEP_RUN_DIR_PREFIX = 'snf_icon'
BASELINE_RUNS = {'SNF+IPM': {'run_dir': f'snf_psas_timeout-{SWEEP_TIMEOUT}', 'marker': '*', 'color': '#777777', 'size': 90}, 'FCFS/B+IPM': {'run_dir': f'easy_psas_timeout-{SWEEP_TIMEOUT}', 'marker': 'X', 'color': '#777777', 'size': 60}, ORACLE_LABEL: {'run_dir': ORACLE_RUN_DIR, 'marker': 'H', 'color': '#009e73', 'size': 75}}
SWEEP_VARIANTS: Dict[str, Dict[str, str]] = {'snf-ssc': {'label': 'SNF-ICON', 'marker': 'o', 'color': '#377eb8'}, 'snf-ssc-nf': {'label': 'SNF-ICON-NF', 'marker': 's', 'color': '#e41a1c'}, 'snf-ssc-ng': {'label': 'SNF-ICON-NG', 'marker': '^', 'color': '#4daf4a'}, 'snf-ssc-ngnf': {'label': 'SNF-ICON-NGNF', 'marker': 'D', 'color': '#984ea3'}}
OLD_SWEEP_PATTERN = re.compile('^alpha-(?P<alpha>\\d+(?:\\.\\d+)?)_beta-(?P<beta>\\d+(?:\\.\\d+)?)$')
NEW_SWEEP_PATTERN = re.compile('^(?P<variant>snf-ssc(?:-nf|-ng|-ngnf|-nfng)?)_alpha-(?P<alpha>\\d+(?:\\.\\d+)?)_beta-(?P<beta>\\d+(?:\\.\\d+)?)$')

@dataclass(frozen=True)
class SweepPoint:
    workload: str
    variant: str
    alpha: float
    beta: float
    waiting_minutes: float
    wasted_energy_mwh: float
    metrics_path: Path

SOURCE_DIR_RECORDS: List[Dict[str, str]] = []
SOURCE_DIR_SEEN: set[Tuple[str, str]] = set()

def source_root_label(path: Path) -> str:
    resolved = path.expanduser().resolve()
    roots = (('Reward sweep', SWEEP_ROOT), ('Heuristic-IPM', HEURISTIC_ORACLE_ROOT))
    for label, root in roots:
        try:
            resolved.relative_to(root.expanduser().resolve())
            return label
        except ValueError:
            continue
    return 'Other'

IMPORTANT_ALGO_CONFIG_KEYS: Tuple[str, ...] = ('run.algo_config.alpha', 'run.algo_config.beta', MARKOV_WINDOW_CONFIG_KEY)
FIGURE_CONFIG_RESULT_DIRS: Dict[Path, set[Path]] = {}

def validated_config_alpha_beta(result_dir: Path, directory_alpha: float, directory_beta: float, source_description: str) -> Tuple[float, float]:
    alpha = config_numeric_value(result_dir, 'run.algo_config.alpha')
    beta = config_numeric_value(result_dir, 'run.algo_config.beta')
    validate_config_numeric_matches(result_dir=result_dir, value_name='alpha', config_value=alpha, source_value=directory_alpha, source_description=source_description)
    validate_config_numeric_matches(result_dir=result_dir, value_name='beta', config_value=beta, source_value=directory_beta, source_description=source_description)
    return (alpha, beta)

def figure6_parse_reward_experiment_name(experiment_name: str) -> Tuple[str, float, float] | None:
    old_match = OLD_SWEEP_PATTERN.fullmatch(experiment_name)
    if old_match is not None:
        return ('snf-ssc', float(old_match.group('alpha')), float(old_match.group('beta')))
    new_match = NEW_SWEEP_PATTERN.fullmatch(experiment_name)
    if new_match is not None:
        variant = new_match.group('variant')
        if variant == 'snf-ssc-nfng':
            variant = 'snf-ssc-ngnf'
        return (variant, float(new_match.group('alpha')), float(new_match.group('beta')))
    return None

def figure6_find_reward_metrics_path(experiment_dir: Path, workload: str) -> Path:
    workload_dir = experiment_dir / 'AOBA-64' / workload
    if not workload_dir.is_dir():
        raise FileNotFoundError(f'Missing sweep workload directory: {workload_dir}')
    exact_candidates = (workload_dir / f'{SWEEP_RUN_DIR_PREFIX}_timeout-{SWEEP_TIMEOUT}' / 'metrics.csv', workload_dir / SWEEP_RUN_DIR_PREFIX / 'metrics.csv')
    for candidate in exact_candidates:
        if candidate.is_file():
            validate_result_config_identity(
                candidate.parent,
                expected_platform=FIGURE6_PLATFORM,
                expected_workload=workload,
                expected_algorithms=('snf_icon',),
            )
            validated_config_timeout_seconds(candidate.parent, SWEEP_TIMEOUT, 'Figure 6 run directory name')
            return candidate
    matches: List[Path] = []
    for run_dir in workload_dir.iterdir():
        if not run_dir.is_dir():
            continue
        if not run_dir.name.startswith(SWEEP_RUN_DIR_PREFIX):
            continue
        metrics_path = run_dir / 'metrics.csv'
        if not metrics_path.is_file():
            continue
        run_timeout = pareto_extract_timeout(run_dir.name)
        expected_timeout = SWEEP_TIMEOUT if run_timeout is None else run_timeout
        validate_result_config_identity(
            run_dir,
            expected_platform=FIGURE6_PLATFORM,
            expected_workload=workload,
            expected_algorithms=('snf_icon',),
        )
        config_timeout = validated_config_timeout_seconds(run_dir, expected_timeout, 'Figure 6 run directory name')
        if config_timeout != SWEEP_TIMEOUT:
            continue
        matches.append(metrics_path)
    if not matches:
        raise FileNotFoundError(f'Missing sweep metrics under {workload_dir}; expected one snf_icon run for timeout {SWEEP_TIMEOUT}s')
    if len(matches) > 1:
        raise ValueError(f'Multiple matching snf_icon runs found; refusing to choose silently: {matches}')
    return matches[0]

def figure6_discover_reward_points() -> List[SweepPoint]:
    if not SWEEP_ROOT.is_dir():
        raise FileNotFoundError(f'Sweep directory does not exist: {SWEEP_ROOT}')
    points: List[SweepPoint] = []
    seen = set()
    combinations_by_workload: Dict[str, set[Tuple[str, float, float]]] = {workload: set() for workload in SWEEP_WORKLOADS}
    missing_paths: List[str] = []
    for experiment_dir in sorted(SWEEP_ROOT.iterdir(), key=lambda path: path.name):
        if not experiment_dir.is_dir():
            continue
        if experiment_dir.name.startswith('.'):
            continue
        parsed = figure6_parse_reward_experiment_name(experiment_dir.name)
        if parsed is None:
            continue
        variant, directory_alpha, directory_beta = parsed
        if variant not in SWEEP_VARIANTS:
            raise ValueError(f'Unknown sweep variant: {variant}')
        for workload in SWEEP_WORKLOADS:
            try:
                metrics_path = figure6_find_reward_metrics_path(experiment_dir, workload)
            except FileNotFoundError:
                missing_paths.append(f'{experiment_dir.name}: {workload}')
                continue
            alpha, beta = validated_config_alpha_beta(metrics_path.parent, directory_alpha, directory_beta, 'Figure 6 experiment directory name')
            config_numeric_value(
                metrics_path.parent,
                MARKOV_WINDOW_CONFIG_KEY,
            )
            combination = (variant, alpha, beta)
            waiting_minutes, wasted_energy_mwh = pareto_read_metrics(metrics_path)
            key = (workload, variant, alpha, beta)
            if key in seen:
                raise ValueError(f'Duplicate sweep point: {workload}, {variant}, alpha={alpha}, beta={beta}')
            seen.add(key)
            combinations_by_workload[workload].add(combination)
            points.append(SweepPoint(workload=workload, variant=variant, alpha=alpha, beta=beta, waiting_minutes=waiting_minutes, wasted_energy_mwh=wasted_energy_mwh, metrics_path=metrics_path))
    if not points:
        raise RuntimeError(f'No sweep points found under {SWEEP_ROOT}')
    all_combinations = set().union(*combinations_by_workload.values())
    incomplete: List[str] = []
    for workload in SWEEP_WORKLOADS:
        missing = sorted(all_combinations - combinations_by_workload[workload], key=lambda item: (item[0], item[1], item[2]))
        for variant, alpha, beta in missing:
            incomplete.append(f'{workload}: {variant}, alpha={alpha:g}, beta={beta:g}')
    if incomplete:
        details = '\n'.join((f'- {item}' for item in incomplete))
        raise RuntimeError(f'Alpha-beta combinations are not complete across all workloads:\n{details}')
    return points

def figure6_discover_baselines(include_oracle: bool=True) -> Dict[str, List[ParetoBaselinePoint]]:
    baselines: Dict[str, List[ParetoBaselinePoint]] = {}
    for workload in SWEEP_WORKLOADS:
        workload_baselines = []
        for label, style in BASELINE_RUNS.items():
            if label == ORACLE_LABEL and (not include_oracle):
                continue
            metrics_path = pareto_heuristic_oracle_workload_dir(
                HEURISTIC_ORACLE_ROOT,
                FIGURE6_PLATFORM,
                workload,
            ) / style['run_dir'] / 'metrics.csv'
            expected_algorithms = None
            if label == 'SNF+IPM':
                expected_algorithms = ('snf_psas', 'snf_ipm')
            elif label == 'FCFS/B+IPM':
                expected_algorithms = ('easy_psas', 'easy_ipm')

            validate_result_config_identity(
                metrics_path.parent,
                expected_platform=FIGURE6_PLATFORM,
                expected_workload=workload,
                expected_algorithms=expected_algorithms,
            )
            if label != ORACLE_LABEL:
                validated_config_timeout_seconds(metrics_path.parent, SWEEP_TIMEOUT, 'Figure 6 baseline run directory name')
            waiting_minutes, wasted_energy_mwh = pareto_read_metrics(metrics_path)
            workload_baselines.append(ParetoBaselinePoint(workload=workload, label=label, waiting_minutes=waiting_minutes, wasted_energy_mwh=wasted_energy_mwh, metrics_path=metrics_path))
        baselines[workload] = workload_baselines
    return baselines

def figure6_select_fixed_beta(
    points: Sequence[SweepPoint],
) -> Tuple[List[SweepPoint], float]:
    """Select one beta and reject an accidental two-dimensional reward sweep.

    Set FIGURE6_FIXED_BETA to a number when the Reward-Sweep directory contains
    stale runs for several beta values. Leave it as None when the directory
    already contains exactly one beta value.
    """
    available_betas = sorted({float(point.beta) for point in points})
    if not available_betas:
        raise RuntimeError('No beta values are available for Figure 6.')

    if (None) is None:
        if len(available_betas) != 1:
            values = ', '.join(f'{value:g}' for value in available_betas)
            raise RuntimeError(
                'Figure 6 varies alpha while keeping beta fixed, but several '
                f'beta values were discovered: {values}. Set '
                'FIGURE6_FIXED_BETA to the beta that should be plotted.'
            )
        fixed_beta = available_betas[0]
    else:
        fixed_beta = float((None))

    selected = [
        point
        for point in points
        if math.isclose(
            float(point.beta),
            fixed_beta,
            rel_tol=1e-12,
            abs_tol=1e-12,
        )
    ]
    if not selected:
        values = ', '.join(f'{value:g}' for value in available_betas)
        raise RuntimeError(
            f'No Figure 6 points use beta={fixed_beta:g}. '
            f'Available beta values: {values}.'
        )

    missing_workloads = [
        workload
        for workload in SWEEP_WORKLOADS
        if not any(point.workload == workload for point in selected)
    ]
    if missing_workloads:
        raise RuntimeError(
            f'beta={fixed_beta:g} is missing workloads: '
            + ', '.join(missing_workloads)
        )

    return selected, fixed_beta


def figure6_make_reward_sweep_pareto(
    points: Sequence[SweepPoint],
    baselines: Dict[str, List[ParetoBaselinePoint]],
) -> Tuple[Path, Path]:
    panel_count = len(SWEEP_WORKLOADS)
    if panel_count != 5:
        raise ValueError(
            'The Figure 6 layout expects exactly five data panels; '
            f'found {panel_count}.'
        )

    encoded_betas = sorted({float(point.beta) for point in points})
    if len(encoded_betas) != 1:
        raise RuntimeError(
            'figure6_make_reward_sweep_pareto expects points from one fixed '
            f'beta, but found: {encoded_betas}'
        )
    fixed_beta = encoded_betas[0]

    encoded_alphas = sorted({float(point.alpha) for point in points})
    if not encoded_alphas:
        raise RuntimeError('No alpha values are available for Figure 6.')

    # Match Figure 7: five data panels plus a dedicated encoding key.
    fig, axes = plt.subplots(
        nrows=3,
        ncols=2,
        figsize=((4.5, 6.0)),
        squeeze=False,
    )
    all_axes = axes.ravel()
    panel_axes = list(all_axes[:panel_count])
    alpha_key_ax = all_axes[panel_count]
    alpha_key_ax.set_axis_off()

    # Alpha values are treated as ordered tested configurations. Mapping by
    # rank prevents values such as 10, 50, 100, 500, and 2000 from collapsing
    # near one end of a linear numeric color scale.
    alpha_cmap = plt.get_cmap(('cividis'), len(encoded_alphas))
    alpha_norm = BoundaryNorm(
        np.arange(-0.5, len(encoded_alphas) + 0.5, 1.0),
        alpha_cmap.N,
    )
    alpha_index = {
        alpha: index
        for index, alpha in enumerate(encoded_alphas)
    }
    alpha_mappable = ScalarMappable(norm=alpha_norm, cmap=alpha_cmap)
    alpha_mappable.set_array(np.arange(len(encoded_alphas), dtype=float))

    # By default, draw the Pareto frontier using the darkest discrete color
    # from the same cividis palette used for alpha. A non-None setting still
    # allows an explicit override.
    pareto_line_color = (
        alpha_cmap(alpha_norm(0))
        if (None) is None
        else (None)
    )

    def alpha_color(alpha: float):
        return alpha_cmap(alpha_norm(alpha_index[float(alpha)]))

    def alpha_marker_size(alpha: float) -> float:
        """Redundant non-color encoding of alpha using marker area."""
        if len(encoded_alphas) == 1:
            fraction = 0.5
        else:
            fraction = alpha_index[float(alpha)] / (len(encoded_alphas) - 1)
        return (
            (32.0)
            + fraction
            * ((100.0) - (32.0))
        )

    for panel_index, (ax, workload) in enumerate(
        zip(panel_axes, SWEEP_WORKLOADS)
    ):
        workload_points = [
            point
            for point in points
            if point.workload == workload
        ]
        if not workload_points:
            raise RuntimeError(f'No Figure 6 points for {workload}')

        workload_baselines = baselines[workload]
        coordinates = [
            (point.waiting_minutes, point.wasted_energy_mwh)
            for point in workload_points
        ]
        coordinates.extend(
            (
                baseline.waiting_minutes,
                baseline.wasted_energy_mwh,
            )
            for baseline in workload_baselines
            if baseline.label != ORACLE_LABEL
        )
        frontier = pareto_frontier(coordinates)
        frontier_coordinates = set(frontier)

        for variant, style in SWEEP_VARIANTS.items():
            variant_points = sorted(
                (
                    point
                    for point in workload_points
                    if point.variant == variant
                ),
                key=lambda point: point.alpha,
                reverse=True,
            )
            if not variant_points:
                continue

            # Draw larger-alpha points first, then smaller points on top, so
            # coincident configurations remain visible instead of being hidden
            # completely by the largest marker.
            for point_index, point in enumerate(variant_points):
                coordinate = (
                    point.waiting_minutes,
                    point.wasted_energy_mwh,
                )
                is_pareto = coordinate in frontier_coordinates
                point_color = alpha_color(point.alpha)
                point_size = alpha_marker_size(point.alpha)

                ax.scatter(
                    point.waiting_minutes,
                    point.wasted_energy_mwh,
                    s=point_size,
                    marker=style['marker'],
                    facecolors=(point_color if is_pareto else 'none'),
                    edgecolors=point_color,
                    linewidths=(
                        (0.75)
                        if is_pareto
                        else (1.1)
                    ),
                    hatch=(
                        None
                        if is_pareto
                        else ('//////')
                    ),
                    label=(
                        style['label']
                        if point_index == 0
                        else '_nolegend_'
                    ),
                    zorder=3,
                )

        for baseline in workload_baselines:
            style = BASELINE_RUNS[baseline.label]
            baseline_coordinate = (
                baseline.waiting_minutes,
                baseline.wasted_energy_mwh,
            )
            baseline_is_pareto = baseline_coordinate in frontier_coordinates
            baseline_color = style['color']

            ax.scatter(
                [baseline.waiting_minutes],
                [baseline.wasted_energy_mwh],
                s=style['size'],
                marker=style['marker'],
                facecolors=(
                    baseline_color
                    if baseline_is_pareto
                    else 'none'
                ),
                edgecolors=baseline_color,
                linewidths=(
                    (0.75)
                    if baseline_is_pareto
                    else (1.1)
                ),
                hatch=(
                    None
                    if baseline_is_pareto
                    else ('//////')
                ),
                label=baseline.label,
                zorder=5,
            )

        if len(frontier) >= 2:
            ax.plot(
                [coordinate[0] for coordinate in frontier],
                [coordinate[1] for coordinate in frontier],
                color=pareto_line_color,
                linestyle=('--'),
                linewidth=(1.0),
                label='Pareto frontier',
                zorder=4,
            )

        workload_result_dirs = [
            point.metrics_path.parent
            for point in workload_points
        ]
        ax.set_title(
            f"({chr(ord('a') + panel_index)}) {FIGURE6_PLATFORM}\n"
            f"{workload_title_label(workload_result_dirs)}",
            y=(1.02),
        )
        visible_fraction_control = (
            ({'DAS2-fs1-0-3000': {'x': 0.25, 'y': 0.25}, 'DAS2-fs2-0-3000': {'x': 0.25, 'y': 0.25}, 'DAS2-fs3-0-3000': {'x': 0.25, 'y': 0.25}, 'DAS2-fs4-0-3000': {'x': 0.5, 'y': 0.25}, 'generated-markovian-3000': {'x': 0.25, 'y': 0.25}})[workload]
        )
        outlier_region_control = (
            ({'DAS2-fs1-0-3000': {'x': 0.25, 'y': 0.25}, 'DAS2-fs2-0-3000': {'x': 0.25, 'y': 0.25}, 'DAS2-fs3-0-3000': {'x': 0.25, 'y': 0.25}, 'DAS2-fs4-0-3000': {'x': 0.25, 'y': 0.25}, 'generated-markovian-3000': {'x': 0.25, 'y': 0.25}})[workload]
        )
        pareto_finalize_panel(
            ax,
            coordinates,
            x_margin=(0.15),
            y_margin=(0.15),
            visible_fraction_threshold=visible_fraction_control,
            outlier_region_fraction=outlier_region_control,
        )

    handles: List[object] = []
    labels: List[str] = []
    seen_labels = set()
    for ax in panel_axes:
        axis_handles, axis_labels = ax.get_legend_handles_labels()
        for handle, label in zip(axis_handles, axis_labels):
            if label in seen_labels:
                continue
            seen_labels.add(label)
            handles.append(handle)
            labels.append(label)

    label_to_handle: Dict[str, object] = dict(zip(labels, handles))
    marker_styles: Dict[str, Tuple[str, float]] = {
        style['label']: (style['marker'], (6.0))
        for style in SWEEP_VARIANTS.values()
    }
    marker_styles.update({
        label: (style['marker'], (6.0))
        for label, style in BASELINE_RUNS.items()
        if label != ORACLE_LABEL
    })

    # Algorithm identity is shape-only. Alpha is encoded separately in the key.
    for label, (marker, _) in marker_styles.items():
        if label not in label_to_handle:
            continue
        size = ({'X': 6.0, '*': 9.0, 'o': 5.5, 's': 5.5, '^': 5.5, 'D': 5.5, 'P': 7.0}).get(
            marker,
            (6.0),
        )
        label_to_handle[label] = Line2D(
            [0], [0],
            linestyle='none',
            marker=marker,
            markerfacecolor=('#333333'),
            markeredgecolor=('#333333'),
            markersize=size,
            label=label,
        )

    circle_radius = (6.0) / 20
    label_to_handle['Pareto'] = Circle(
        (0.5, 0.5),
        radius=circle_radius,
        facecolor=('#333333'),
        edgecolor=('#333333'),
        label='Pareto',
    )
    label_to_handle['Non-Pareto'] = Circle(
        (0.5, 0.5),
        radius=circle_radius,
        facecolor='none',
        edgecolor=('#333333'),
        hatch=('//////'),
        label='Non-Pareto',
    )

    blank_handle = Line2D([0], [0], linestyle='none', marker='', label='')
    legend_columns = [
        [
            'FCFS/B+IPM',
            'SNF+IPM',
            'SNF-ICON',
        ],
        [
            'SNF-ICON-NF',
            'SNF-ICON-NG',
            'SNF-ICON-NGNF',
        ],
        [
            'Pareto frontier',
            'Pareto',
            'Non-Pareto',
        ],
    ]
    rows_per_column = max(len(column) for column in legend_columns)
    legend_handles: List[object] = []
    legend_labels: List[str] = []
    for column in legend_columns:
        for row_index in range(rows_per_column):
            if row_index >= len(column):
                legend_handles.append(blank_handle)
                legend_labels.append('')
                continue
            label = column[row_index]
            if label in label_to_handle:
                legend_handles.append(label_to_handle[label])
                legend_labels.append(label)
            else:
                legend_handles.append(blank_handle)
                legend_labels.append('')


    fig.supxlabel(
        'Average waiting time (min)',
        y=(-0.08),
    )
    fig.supylabel(
        'Total wasted energy (MWh)',
        x=(0.0),
    )
    fig.legend(
        handles=legend_handles,
        labels=legend_labels,
        loc=('upper center'),
        bbox_to_anchor=(((1.0 - 0.84) / 2.0, 1.08, 0.84, 0.12)),
        bbox_transform=fig.transFigure,
        mode='expand',
        ncol=len(legend_columns),
        frameon=(True),
        columnspacing=(2.0),
        handlelength=(1.0),
        handleheight=(1.0),
        handletextpad=(0.5),
        labelspacing=(0.5),
        borderaxespad=0.0,
    )

    # Vertical alpha key. Every tested alpha can be shown without horizontal
    # tick-label collisions, even when the values have very different widths.
    alpha_key_ax.set_title(
        rf'$\alpha$ encoding',
        y=(1.02),
        pad=0,
    )
    colorbar_ax = alpha_key_ax.inset_axes(
        ((0.02, 0.12, 0.18, 0.8)),
        transform=alpha_key_ax.transAxes,
    )
    colorbar = fig.colorbar(
        alpha_mappable,
        cax=colorbar_ax,
        orientation='vertical',
    )

    tick_count = min(
        (12),
        len(encoded_alphas),
    )
    if tick_count == len(encoded_alphas):
        tick_indices = list(range(len(encoded_alphas)))
    else:
        tick_indices = np.floor(
            np.linspace(0, len(encoded_alphas) - 1, tick_count)
        ).astype(int).tolist()
        tick_indices[-1] = len(encoded_alphas) - 1
        tick_indices = sorted(set(tick_indices))

    colorbar.set_ticks(tick_indices)
    colorbar.set_ticklabels([
        f'{encoded_alphas[index]:g}'
        for index in tick_indices
    ])
    colorbar.ax.yaxis.set_ticks_position('right')
    colorbar.ax.tick_params(labelsize=(8.0))

    if (None) is None:
        representative_indices = sorted({
            0,
            len(encoded_alphas) // 2,
            len(encoded_alphas) - 1,
        })
        representative_alphas = [
            encoded_alphas[index]
            for index in representative_indices
        ]
    else:
        representative_alphas = []
        for requested_alpha in (None):
            nearest_alpha = min(
                encoded_alphas,
                key=lambda value: abs(value - float(requested_alpha)),
            )
            if nearest_alpha not in representative_alphas:
                representative_alphas.append(nearest_alpha)

    # Matplotlib displays vertical legend entries from top to bottom. Reverse
    # the values so the largest alpha is at the top, matching the colorbar.
    representative_alphas = sorted(representative_alphas, reverse=True)

    size_handles = [
        Line2D(
            [0], [0],
            linestyle='none',
            marker='o',
            markerfacecolor=alpha_color(alpha),
            markeredgecolor=alpha_color(alpha),
            markersize=math.sqrt(alpha_marker_size(alpha)),
            label=rf'$\alpha={alpha:g}$',
        )
        for alpha in representative_alphas
    ]

    # Dedicated adjustable container for the circle-size legend.
    # Bounds use the unused sixth panel's coordinates:
    #     (left, bottom, width, height)
    # The default full-panel bounds preserve the previous placement exactly.
    circle_legend_ax = alpha_key_ax.inset_axes(
        ((0.0, 0.0, 1.0, 1.0)),
        transform=alpha_key_ax.transAxes,
    )
    circle_legend_ax.set_axis_off()
    circle_legend_ax.legend(
        handles=size_handles,
        loc=('center'),
        bbox_to_anchor=((0.72, 0.5)),
        ncol=(1),
        frameon=(False),
        markerscale=(1.0),
        fontsize=(None),
        handletextpad=(0.3),
        labelspacing=(1.5),
        borderaxespad=(0.0),
    )

    fig.subplots_adjust(
        left=(0.12),
        right=(0.95),
        top=(1.0),
        bottom=(0.0),
        wspace=(0.25),
        hspace=(0.5),
    )

    figure6_output_dir = OUTPUT_DIR / 'Figure6_RewardSweep'
    figure6_output_dir.mkdir(parents=True, exist_ok=True)
    png_path = figure6_output_dir / 'CustomVisPaper_Figure6_RewardSweepPareto.png'
    pdf_path = figure6_output_dir / 'CustomVisPaper_Figure6_RewardSweepPareto.pdf'

    save_figure(
        fig,
        png_path,
        pdf_path,
        png_kwargs={},
        pdf_kwargs={},
    )


    plt.close(fig)
    write_chart_swept_config(
        png_path,
        {
            workload: [
                point.metrics_path.parent
                for point in points
                if point.workload == workload
            ] + [
                baseline.metrics_path.parent
                for baseline in baselines[workload]
            ]
            for workload in SWEEP_WORKLOADS
        },
    )
    return (png_path, pdf_path)

# Generate and display Figure 6

apply_plot_style()
all_reward_points = figure6_discover_reward_points()
points, figure6_fixed_beta = figure6_select_fixed_beta(all_reward_points)
baselines = figure6_discover_baselines(include_oracle=False)
png_path, pdf_path = figure6_make_reward_sweep_pareto(points, baselines)
display(Image(filename=str(png_path)))
(png_path, pdf_path)

# Figure 6 comparison table (FCFS/B+IPM baseline)
def figure6_build_comparison_table(
    points: Sequence[SweepPoint],
    baselines: Mapping[str, Sequence[ParetoBaselinePoint]],
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for workload in SWEEP_WORKLOADS:
        workload_baselines = list(baselines[workload])
        fcfs_matches = [b for b in workload_baselines if b.label == 'FCFS/B+IPM']
        if len(fcfs_matches) != 1:
            raise RuntimeError(
                f'Expected one FCFS/B+IPM baseline for {workload}; '
                f'found {len(fcfs_matches)}.'
            )
        fcfs = fcfs_matches[0]
        workload_points = [p for p in points if p.workload == workload]
        frontier_coordinates = set(
            pareto_frontier(
                [
                    (point.waiting_minutes, point.wasted_energy_mwh)
                    for point in workload_points
                ]
                + [
                    (baseline.waiting_minutes, baseline.wasted_energy_mwh)
                    for baseline in workload_baselines
                    if baseline.label != ORACLE_LABEL
                ]
            )
        )
        for point in workload_points:
            rows.append({
                'workload': dataset_display_label(workload),
                'series_type': 'reward sweep',
                'algorithm': SWEEP_VARIANTS[point.variant]['label'],
                'alpha': point.alpha,
                'beta': point.beta,
                'waiting_time_minutes': point.waiting_minutes,
                'energy_waste_mwh': point.wasted_energy_mwh,
                'is_pareto': (
                    point.waiting_minutes,
                    point.wasted_energy_mwh,
                ) in frontier_coordinates,
                'waiting_time_improvement_vs_fcfs_b_ipm_percent': (
                    comparison_improvement_percent(point.waiting_minutes, fcfs.waiting_minutes)
                ),
                'energy_waste_improvement_vs_fcfs_b_ipm_percent': (
                    comparison_improvement_percent(point.wasted_energy_mwh, fcfs.wasted_energy_mwh)
                ),
            })
        for baseline in workload_baselines:
            rows.append({
                'workload': dataset_display_label(workload),
                'series_type': 'baseline',
                'algorithm': (
                    COMPARISON_BASELINE_DISPLAY_LABEL
                    if baseline.label == 'FCFS/B+IPM'
                    else baseline.label.replace(' baseline', '')
                ),
                'alpha': float('nan'),
                'beta': float('nan'),
                'waiting_time_minutes': baseline.waiting_minutes,
                'energy_waste_mwh': baseline.wasted_energy_mwh,
                'is_pareto': (
                    baseline.label != ORACLE_LABEL
                    and (
                        baseline.waiting_minutes,
                        baseline.wasted_energy_mwh,
                    ) in frontier_coordinates
                ),
                'waiting_time_improvement_vs_fcfs_b_ipm_percent': (
                    comparison_improvement_percent(baseline.waiting_minutes, fcfs.waiting_minutes)
                ),
                'energy_waste_improvement_vs_fcfs_b_ipm_percent': (
                    comparison_improvement_percent(baseline.wasted_energy_mwh, fcfs.wasted_energy_mwh)
                ),
            })
    return add_explicit_fcfs_b_ipm_comparisons(
        pd.DataFrame(rows),
        group_columns=['workload'],
        metric_specs=[
            (
                'waiting_time_minutes',
                'fcfs_b_ipm_waiting_time_minutes',
                'waiting_time_difference_minutes',
                'waiting_time_improvement_percent',
                'waiting_time_comparison',
                False,
            ),
            (
                'energy_waste_mwh',
                'fcfs_b_ipm_energy_waste_mwh',
                'energy_waste_difference_mwh',
                'energy_waste_improvement_percent',
                'energy_waste_comparison',
                False,
            ),
        ],
    )


figure6_comparison_table = display_comparison_table(
    figure6_build_comparison_table(points, baselines),
    'Figure 6 comparison table',
)


## Figure 7

Markov-window sweep.


In [ ]:
# Figure 7
# Run the common setup cell first. This cell contains Figure 7-specific code and settings.

# Shared Pareto mechanics are defined once in the common setup cell.

from __future__ import annotations

from copy import copy
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch, Circle
from matplotlib.ticker import FixedLocator, Formatter, FuncFormatter, MaxNLocator, NullLocator

HEURISTIC_ORACLE_ROOT = RESULT_SAVE_3_ROOT / 'Heuristic-IPM' / 'Experiment-64'

ORACLE_LABEL = 'SNF Oracle'
ORACLE_RUN_DIR = 'snf_oracle_psas_no-timeout'
FIGURE7_PLATFORM = 'AOBA-64'
SWEEP_TIMEOUT = 7200
SWEEP_RUN_DIR_PREFIX = 'snf_icon'
# ================ FIX 1: Star color now matches FCFS/B+IPM =================
BASELINE_RUNS = {'SNF+IPM': {'run_dir': f'snf_psas_timeout-{SWEEP_TIMEOUT}', 'marker': '*', 'color': '#777777', 'size': 90}, 'FCFS/B+IPM': {'run_dir': f'easy_psas_timeout-{SWEEP_TIMEOUT}', 'marker': 'X', 'color': '#777777', 'size': 60}, ORACLE_LABEL: {'run_dir': ORACLE_RUN_DIR, 'marker': 'H', 'color': '#009e73', 'size': 75}}
# ============================================================================
SWEEP_VARIANTS: Dict[str, Dict[str, str]] = {'snf-ssc': {'label': 'SNF-ICON', 'marker': 'o', 'color': '#377eb8'}, 'snf-ssc-nf': {'label': 'SNF-ICON-NF', 'marker': 's', 'color': '#e41a1c'}, 'snf-ssc-ng': {'label': 'SNF-ICON-NG', 'marker': '^', 'color': '#4daf4a'}, 'snf-ssc-ngnf': {'label': 'SNF-ICON-NGNF', 'marker': 'D', 'color': '#984ea3'}}
FIGURE7_ROOT = RESULT_SAVE_3_ROOT / 'Markov-Window-Sweep'
FIGURE7_TIMEOUT = SWEEP_TIMEOUT
FIGURE7_WORKLOADS: Tuple[str, ...] = ('DAS2-fs1-0-3000','DAS2-fs2-0-3000','DAS2-fs3-0-3000', 'DAS2-fs4-0-3000','generated-markovian-3000')
FIGURE7_RUN_DIR_PREFIX = SWEEP_RUN_DIR_PREFIX
FIGURE7_PATTERN = re.compile('^(?P<variant>snf-ssc(?:-nf|-ng|-ngnf|-nfng)?)_MW_(?P<hours>\\d+(?:\\.\\d+)?)h$')
FIGURE7_WINDOW_IGNORED_VARIANTS: Tuple[str, ...] = (
    'snf-ssc-nf',
    'snf-ssc-ngnf',
)

@dataclass(frozen=True)
class MarkovWindowPoint:
    workload: str
    variant: str
    window_hours: float
    waiting_minutes: float
    wasted_energy_mwh: float
    metrics_path: Path

SOURCE_DIR_RECORDS: List[Dict[str, str]] = []
SOURCE_DIR_SEEN: set[Tuple[str, str]] = set()

def source_root_label(path: Path) -> str:
    resolved = path.expanduser().resolve()
    roots = (('Markov-window sweep', FIGURE7_ROOT), ('Heuristic-IPM', HEURISTIC_ORACLE_ROOT))
    for label, root in roots:
        try:
            resolved.relative_to(root.expanduser().resolve())
            return label
        except ValueError:
            continue
    return 'Other'

IMPORTANT_ALGO_CONFIG_KEYS: Tuple[str, ...] = ('run.algo_config.alpha', 'run.algo_config.beta', MARKOV_WINDOW_CONFIG_KEY)
FIGURE_CONFIG_RESULT_DIRS: Dict[Path, set[Path]] = {}

def validated_config_markov_window_hours(result_dir: Path, directory_window_hours: float, source_description: str) -> float:
    window_seconds = config_numeric_value(result_dir, MARKOV_WINDOW_CONFIG_KEY)
    window_hours = window_seconds / 3600.0
    validate_config_numeric_matches(result_dir=result_dir, value_name='Markov window', config_value=window_hours, source_value=directory_window_hours, source_description=source_description)
    return window_hours

def figure7_parse_experiment_name(experiment_name: str) -> Tuple[str, float] | None:
    match = FIGURE7_PATTERN.fullmatch(experiment_name)
    if match is None:
        return None
    variant = match.group('variant')
    if variant == 'snf-ssc-nfng':
        variant = 'snf-ssc-ngnf'
    return (variant, float(match.group('hours')))

def figure7_find_metrics_path(experiment_dir: Path, workload: str) -> Path:
    workload_dir = experiment_dir / 'AOBA-64' / workload
    if not workload_dir.is_dir():
        raise FileNotFoundError(f'Missing Figure 7 workload directory: {workload_dir}')
    exact_candidates = (workload_dir / f'{FIGURE7_RUN_DIR_PREFIX}_timeout-{FIGURE7_TIMEOUT}' / 'metrics.csv', workload_dir / FIGURE7_RUN_DIR_PREFIX / 'metrics.csv')
    for candidate in exact_candidates:
        if candidate.is_file():
            validate_result_config_identity(
                candidate.parent,
                expected_platform=FIGURE7_PLATFORM,
                expected_workload=workload,
                expected_algorithms=('snf_icon',),
            )
            validated_config_timeout_seconds(candidate.parent, FIGURE7_TIMEOUT, 'Figure 7 run directory name')
            return candidate
    matches: List[Path] = []
    for run_dir in sorted(workload_dir.iterdir(), key=lambda path: path.name.lower()):
        if not run_dir.is_dir():
            continue
        if not run_dir.name.startswith(FIGURE7_RUN_DIR_PREFIX):
            continue
        metrics_path = run_dir / 'metrics.csv'
        if not metrics_path.is_file():
            continue
        run_timeout = pareto_extract_timeout(run_dir.name)
        expected_timeout = FIGURE7_TIMEOUT if run_timeout is None else run_timeout
        validate_result_config_identity(
            run_dir,
            expected_platform=FIGURE7_PLATFORM,
            expected_workload=workload,
            expected_algorithms=('snf_icon',),
        )
        config_timeout = validated_config_timeout_seconds(run_dir, expected_timeout, 'Figure 7 run directory name')
        if config_timeout != FIGURE7_TIMEOUT:
            continue
        matches.append(metrics_path)
    if not matches:
        raise FileNotFoundError(f'Missing Figure 7 metrics under {workload_dir}; expected one {FIGURE7_RUN_DIR_PREFIX} run for timeout {FIGURE7_TIMEOUT}s')
    if len(matches) > 1:
        raise ValueError(f'Multiple Figure 7 metrics files found; refusing to choose silently: {matches}')
    return matches[0]

def figure7_discover_points() -> List[MarkovWindowPoint]:
    if not FIGURE7_ROOT.is_dir():
        raise FileNotFoundError(f'Figure 7 directory does not exist: {FIGURE7_ROOT}')
    points: List[MarkovWindowPoint] = []
    seen = set()
    windows_by_variant: Dict[str, set[float]] = {variant: set() for variant in SWEEP_VARIANTS}
    for experiment_dir in sorted(FIGURE7_ROOT.iterdir(), key=lambda path: path.name):
        if not experiment_dir.is_dir():
            continue
        if experiment_dir.name.startswith('.'):
            continue
        parsed = figure7_parse_experiment_name(experiment_dir.name)
        if parsed is None:
            continue
        variant, directory_window_hours = parsed
        if variant not in SWEEP_VARIANTS:
            raise ValueError(f'Unknown Figure 7 variant: {variant}')
        for workload in FIGURE7_WORKLOADS:
            metrics_path = figure7_find_metrics_path(experiment_dir, workload)
            window_hours = validated_config_markov_window_hours(metrics_path.parent, directory_window_hours, 'Figure 7 experiment directory name')
            config_numeric_value(
                metrics_path.parent,
                'run.algo_config.alpha',
            )
            config_numeric_value(
                metrics_path.parent,
                'run.algo_config.beta',
            )
            windows_by_variant[variant].add(window_hours)
            waiting_minutes, wasted_energy_mwh = pareto_read_metrics(metrics_path)
            key = (workload, variant, window_hours)
            if key in seen:
                raise ValueError(f'Duplicate Figure 7 point: {workload}, {variant}, window={window_hours:g}h')
            seen.add(key)
            points.append(MarkovWindowPoint(workload=workload, variant=variant, window_hours=window_hours, waiting_minutes=waiting_minutes, wasted_energy_mwh=wasted_energy_mwh, metrics_path=metrics_path))
    if not points:
        raise RuntimeError(f'No Figure 7 points found under {FIGURE7_ROOT}')

    all_windows = set().union(*windows_by_variant.values())
    missing_windows: List[str] = []
    for variant in SWEEP_VARIANTS:
        for window_hours in sorted(
            all_windows - windows_by_variant[variant]
        ):
            missing_windows.append(
                f'{variant}: {window_hours:g} h'
            )
    if missing_windows:
        raise RuntimeError(
            'Figure 7 is missing Markov-window data:\n- '
            + '\n- '.join(missing_windows)
        )

    figure7_validate_ignored_window_points(points)

    points.sort(key=lambda point: (FIGURE7_WORKLOADS.index(point.workload), list(SWEEP_VARIANTS).index(point.variant), point.window_hours))
    return points

def figure7_validate_ignored_window_points(
    points: Sequence[MarkovWindowPoint],
) -> None:
    """Require identical plotted metrics across ignored window sweeps.

    NF and NGNF disable fallback, so the Markov-window value must not
    affect their results. Every swept directory is still read and
    validated, but all points for one workload/variant must have the
    same waiting time and wasted energy.
    """
    for workload in FIGURE7_WORKLOADS:
        for variant in FIGURE7_WINDOW_IGNORED_VARIANTS:
            variant_points = sorted(
                (
                    point
                    for point in points
                    if (
                        point.workload == workload
                        and point.variant == variant
                    )
                ),
                key=lambda point: point.window_hours,
            )

            if not variant_points:
                raise RuntimeError(
                    f'No Figure 7 points for '
                    f'{workload}/{variant}'
                )

            reference = variant_points[0]
            different: List[str] = []

            for point in variant_points[1:]:
                waiting_matches = math.isclose(
                    point.waiting_minutes,
                    reference.waiting_minutes,
                    rel_tol=1e-12,
                    abs_tol=1e-12,
                )
                energy_matches = math.isclose(
                    point.wasted_energy_mwh,
                    reference.wasted_energy_mwh,
                    rel_tol=1e-12,
                    abs_tol=1e-12,
                )

                if waiting_matches and energy_matches:
                    continue

                different.append(
                    f'{point.window_hours:g}h: '
                    f'waiting={point.waiting_minutes!r}, '
                    f'energy={point.wasted_energy_mwh!r}'
                )

            if different:
                raise RuntimeError(
                    f'Figure 7 {variant} ignores the Markov '
                    f'window, but its metrics differ for '
                    f'{workload}. Reference '
                    f'{reference.window_hours:g}h: '
                    f'waiting={reference.waiting_minutes!r}, '
                    f'energy={reference.wasted_energy_mwh!r}; '
                    f'different values:\n- '
                    + '\n- '.join(different)
                )


def figure7_discover_baselines(include_oracle: bool=True) -> Dict[str, List[ParetoBaselinePoint]]:
    baselines: Dict[str, List[ParetoBaselinePoint]] = {}
    for workload in FIGURE7_WORKLOADS:
        workload_baselines: List[ParetoBaselinePoint] = []
        for label, style in BASELINE_RUNS.items():
            if label == ORACLE_LABEL and (not include_oracle):
                continue
            metrics_path = pareto_heuristic_oracle_workload_dir(
                HEURISTIC_ORACLE_ROOT,
                FIGURE7_PLATFORM,
                workload,
            ) / style['run_dir'] / 'metrics.csv'
            expected_algorithms = None
            if label == 'SNF+IPM':
                expected_algorithms = ('snf_psas', 'snf_ipm')
            elif label == 'FCFS/B+IPM':
                expected_algorithms = ('easy_psas', 'easy_ipm')

            validate_result_config_identity(
                metrics_path.parent,
                expected_platform=FIGURE7_PLATFORM,
                expected_workload=workload,
                expected_algorithms=expected_algorithms,
            )
            if label != ORACLE_LABEL:
                validated_config_timeout_seconds(metrics_path.parent, FIGURE7_TIMEOUT, 'Figure 7 baseline run directory name')
            waiting_minutes, wasted_energy_mwh = pareto_read_metrics(metrics_path)
            workload_baselines.append(ParetoBaselinePoint(workload=workload, label=label, waiting_minutes=waiting_minutes, wasted_energy_mwh=wasted_energy_mwh, metrics_path=metrics_path))
        baselines[workload] = workload_baselines
    return baselines

def figure7_make_markov_window_sweep_pareto(points: Sequence[MarkovWindowPoint], baselines: Dict[str, List[ParetoBaselinePoint]]) -> Tuple[Path, Path]:
    panel_count = len(FIGURE7_WORKLOADS)
    if panel_count != 5:
        raise ValueError(
            'The Figure 7 layout expects exactly five data panels; '
            f'found {panel_count}.'
        )

    # Match Figure 9: three rows, two columns, row-major panel placement.
    # The sixth axis is reserved for the Markov-window encoding key.
    fig, axes = plt.subplots(
        nrows=3,
        ncols=2,
        figsize=((4.5, 6.0)),
        squeeze=False,
    )
    all_axes = axes.ravel()
    panel_axes = list(all_axes[:panel_count])
    tm_key_ax = all_axes[panel_count]
    tm_key_ax.set_axis_off()

    # T_M is meaningful only for variants that actually use the Markov window.
    encoded_windows = sorted({
        float(point.window_hours)
        for point in points
        if point.variant not in FIGURE7_WINDOW_IGNORED_VARIANTS
    })
    if not encoded_windows:
        raise RuntimeError('No Markov-window values are available for Figure 7.')

    # A discrete cividis scale is color-vision-deficiency friendly and has
    # monotonic luminance, so its ordering also survives grayscale conversion.
    tm_cmap = plt.get_cmap(('cividis'), len(encoded_windows))
    tm_norm = BoundaryNorm(
        np.arange(-0.5, len(encoded_windows) + 0.5, 1.0),
        tm_cmap.N,
    )
    tm_index = {
        window_hours: index
        for index, window_hours in enumerate(encoded_windows)
    }
    tm_mappable = ScalarMappable(norm=tm_norm, cmap=tm_cmap)
    tm_mappable.set_array(np.arange(len(encoded_windows), dtype=float))

    # By default, draw the Pareto frontier using the darkest discrete color
    # from the same cividis palette used for T_M. A non-None setting still
    # allows an explicit override.
    pareto_line_color = (
        tm_cmap(tm_norm(0))
        if (None) is None
        else (None)
    )

    def tm_color(window_hours: float):
        return tm_cmap(tm_norm(tm_index[float(window_hours)]))

    def tm_marker_size(window_hours: float) -> float:
        """Redundant non-color encoding of T_M using marker area."""
        if len(encoded_windows) == 1:
            fraction = 0.5
        else:
            fraction = tm_index[float(window_hours)] / (len(encoded_windows) - 1)
        return (
            (32.0)
            + fraction
            * ((100.0) - (32.0))
        )

    for panel_index, (ax, workload) in enumerate(
        zip(panel_axes, FIGURE7_WORKLOADS)
    ):
        workload_points = [point for point in points if point.workload == workload]
        if not workload_points:
            raise RuntimeError(f'No Figure 7 points for {workload}')
        workload_baselines = baselines[workload]
        coordinates = [
            (point.waiting_minutes, point.wasted_energy_mwh)
            for point in workload_points
        ]
        coordinates.extend(
            (
                baseline.waiting_minutes,
                baseline.wasted_energy_mwh,
            )
            for baseline in workload_baselines
            if baseline.label != ORACLE_LABEL
        )
        frontier = pareto_frontier(coordinates)
        frontier_coordinates = set(frontier)

        for variant, style in SWEEP_VARIANTS.items():
            variant_points = sorted(
                (
                    point
                    for point in workload_points
                    if point.variant == variant
                ),
                key=lambda point: point.window_hours,
            )
            if not variant_points:
                continue

            window_is_inactive = variant in FIGURE7_WINDOW_IGNORED_VARIANTS
            plotted_variant_points = (
                variant_points[:1]
                if window_is_inactive
                else list(reversed(variant_points))
            )

            for point_index, point in enumerate(plotted_variant_points):
                coordinate = (
                    point.waiting_minutes,
                    point.wasted_energy_mwh,
                )
                is_pareto = coordinate in frontier_coordinates

                if window_is_inactive:
                    point_color = ('#9a9a9a')
                    point_size = (45.0)
                else:
                    point_color = tm_color(point.window_hours)
                    point_size = tm_marker_size(point.window_hours)

                # Pareto status is independent of hue:
                #   Pareto     -> solid T_M-colored marker
                #   Non-Pareto -> hollow, T_M-colored outline and hatch
                # Marker area remains a redundant non-color cue for T_M.
                ax.scatter(
                    point.waiting_minutes,
                    point.wasted_energy_mwh,
                    s=point_size,
                    marker=style['marker'],
                    facecolors=(point_color if is_pareto else 'none'),
                    edgecolors=point_color,
                    linewidths=(
                        (0.75)
                        if is_pareto
                        else (1.1)
                    ),
                    hatch=(
                        None
                        if is_pareto
                        else ('//////')
                    ),
                    label=(
                        style['label']
                        if point_index == 0
                        else '_nolegend_'
                    ),
                    zorder=3,
                )

        for baseline in workload_baselines:
            style = BASELINE_RUNS[baseline.label]
            baseline_coordinate = (
                baseline.waiting_minutes,
                baseline.wasted_energy_mwh,
            )
            baseline_is_pareto = (
                baseline_coordinate in frontier_coordinates
            )
            baseline_color = style['color']
            ax.scatter(
                [baseline.waiting_minutes],
                [baseline.wasted_energy_mwh],
                s=style['size'],
                marker=style['marker'],
                facecolors=(
                    baseline_color
                    if baseline_is_pareto
                    else 'none'
                ),
                edgecolors=baseline_color,
                linewidths=(
                    (0.75)
                    if baseline_is_pareto
                    else (1.1)
                ),
                hatch=(
                    None
                    if baseline_is_pareto
                    else ('//////')
                ),
                label=baseline.label,
                zorder=5,
            )

        if len(frontier) >= 2:
            ax.plot(
                [point[0] for point in frontier],
                [point[1] for point in frontier],
                color=pareto_line_color,
                linestyle=('--'),
                linewidth=(1.0),
                label='Pareto frontier',
                zorder=4,
            )

        workload_result_dirs = [
            point.metrics_path.parent
            for point in workload_points
        ]
        # Match Figure 9's two-line panel-title structure.
        ax.set_title(
            f"({chr(ord('a') + panel_index)}) {FIGURE7_PLATFORM}\n"
            f"{workload_title_label(workload_result_dirs)}",
            y=(1.02),
        )
        visible_fraction_control = (
            ({'DAS2-fs1-0-3000': {'x': 0.25, 'y': 0.25}, 'DAS2-fs2-0-3000': {'x': 0.25, 'y': 0.25}, 'DAS2-fs3-0-3000': {'x': 0.25, 'y': 0.25}, 'DAS2-fs4-0-3000': {'x': 0.25, 'y': 0.25}, 'generated-markovian-3000': {'x': 0.25, 'y': 0.25}})[workload]
        )
        outlier_region_control = (
            ({'DAS2-fs1-0-3000': {'x': 0.25, 'y': 0.25}, 'DAS2-fs2-0-3000': {'x': 0.25, 'y': 0.25}, 'DAS2-fs3-0-3000': {'x': 0.25, 'y': 0.25}, 'DAS2-fs4-0-3000': {'x': 0.25, 'y': 0.25}, 'generated-markovian-3000': {'x': 0.25, 'y': 0.25}})[workload]
        )
        pareto_finalize_panel(
            ax,
            coordinates,
            x_margin=(0.15),
            y_margin=(0.15),
            visible_fraction_threshold=visible_fraction_control,
            outlier_region_fraction=outlier_region_control,
        )

    handles: List[object] = []
    labels: List[str] = []
    seen_labels = set()

    for ax in panel_axes:
        axis_handles, axis_labels = ax.get_legend_handles_labels()
        for handle, label in zip(axis_handles, axis_labels):
            if label in seen_labels:
                continue
            seen_labels.add(label)
            handles.append(handle)
            labels.append(label)

    label_to_handle: Dict[str, object] = dict(zip(labels, handles))

    marker_styles: Dict[str, Tuple[str, float]] = {
        style['label']: (style['marker'], (6.0))
        for style in SWEEP_VARIANTS.values()
    }
    marker_styles.update({
        label: (style['marker'], (6.0))
        for label, style in BASELINE_RUNS.items()
        if label != ORACLE_LABEL
    })

    # ================ FIX 2: Use configurable legend marker sizes ================
    # Now using the global variable FIGURE7_LEGEND_MARKER_SIZE_COMPENSATION
    # =============================================================================

    # Algorithm identity is shape-only in the main legend. This prevents the
    # legend from suggesting that algorithms have fixed colors.
    for label, (marker, _) in marker_styles.items():
        if label not in label_to_handle:
            continue
        # Use the compensation dict, fallback to the base size if marker not found
        size = ({'X': 6.0, '*': 9.0, 'o': 5.5, 's': 5.5, '^': 5.5, 'D': 5.5, 'P': 7.0}).get(marker, (6.0))
        label_to_handle[label] = Line2D(
            [0], [0],
            linestyle='none',
            marker=marker,
            markerfacecolor=('#333333'),
            markeredgecolor=('#333333'),
            markersize=size,
            label=label,
        )

    circle_radius = (6.0) / 20
    label_to_handle['Pareto'] = Circle(
        (0.5, 0.5),
        radius=circle_radius,
        facecolor=('#333333'),
        edgecolor=('#333333'),
        label='Pareto',
    )
    label_to_handle['Non-Pareto'] = Circle(
        (0.5, 0.5),
        radius=circle_radius,
        facecolor='none',
        edgecolor=('#333333'),
        hatch=('//////'),
        label='Non-Pareto',
    )

    blank_handle = Line2D([0], [0], linestyle='none', marker='', label='')
    legend_columns = [
        [
            'FCFS/B+IPM',
            'SNF+IPM',
            'SNF-ICON',
        ],
        [
            'SNF-ICON-NF',
            'SNF-ICON-NG',
            'SNF-ICON-NGNF',
        ],
        [
            'Pareto frontier',
            'Pareto',
            'Non-Pareto',
        ],
    ]

    rows_per_column = max(len(column) for column in legend_columns)
    legend_handles: List[object] = []
    legend_labels: List[str] = []

    for column in legend_columns:
        for row_index in range(rows_per_column):
            if row_index >= len(column):
                legend_handles.append(blank_handle)
                legend_labels.append('')
                continue

            label = column[row_index]
            if label in label_to_handle:
                legend_handles.append(label_to_handle[label])
                legend_labels.append(label)
            else:
                legend_handles.append(blank_handle)
                legend_labels.append('')

    fig.supxlabel(
        'Average waiting time (min)',
        y=(-0.08),
    )
    fig.supylabel(
        'Total wasted energy (MWh)',
        x=(0.0),
    )
    fig.legend(
        handles=legend_handles,
        labels=legend_labels,
        loc=('upper center'),
        bbox_to_anchor=(((1.0 - 0.84) / 2.0, 1.08, 0.84, 0.12)),
        bbox_transform=fig.transFigure,
        mode='expand',
        ncol=len(legend_columns),
        frameon=(True),
        columnspacing=(2.0),
        handlelength=(1.0),
        handleheight=(1.0),
        handletextpad=(0.5),
        labelspacing=(0.5),
        borderaxespad=0.0,
    )

    # Vertical Markov-window key in the unused sixth panel. A vertical bar
    # avoids horizontal tick-label collisions and matches Figure 6.
    tm_key_ax.set_title(
        r'$T_M$ encoding',
        y=(1.02),
        pad=0,
    )

    colorbar_ax = tm_key_ax.inset_axes(
        ((0.02, 0.12, 0.18, 0.8)),
        transform=tm_key_ax.transAxes,
    )
    colorbar = fig.colorbar(
        tm_mappable,
        cax=colorbar_ax,
        orientation='vertical',
    )

    tick_count = min(
        (4),
        len(encoded_windows),
    )
    if tick_count == len(encoded_windows):
        tick_indices = list(range(len(encoded_windows)))
    else:
        # Floor-spaced indices keep representative values stable. For a 1--24 h
        # sweep with four labels this produces 1, 8, 16, and 24 h.
        tick_indices = np.floor(
            np.linspace(0, len(encoded_windows) - 1, tick_count)
        ).astype(int).tolist()
        tick_indices[-1] = len(encoded_windows) - 1
        tick_indices = sorted(set(tick_indices))

    colorbar.set_ticks(tick_indices)
    colorbar.set_ticklabels([
        f'{encoded_windows[index]:g} h'
        for index in tick_indices
    ])
    colorbar.ax.yaxis.set_ticks_position('right')
    colorbar.ax.tick_params(labelsize=(8.0))

    if (None) is None:
        representative_indices = sorted({
            0,
            len(encoded_windows) // 2,
            len(encoded_windows) - 1,
        })
        representative_windows = [
            encoded_windows[index]
            for index in representative_indices
        ]
    else:
        representative_windows = []
        for requested_window in (None):
            nearest_window = min(
                encoded_windows,
                key=lambda value: abs(value - float(requested_window)),
            )
            if nearest_window not in representative_windows:
                representative_windows.append(nearest_window)

    # Matplotlib displays vertical legend entries from top to bottom. Reverse
    # the values so the largest T_M is at the top, matching the colorbar.
    representative_windows = sorted(representative_windows, reverse=True)

    size_handles = [
        Line2D(
            [0], [0],
            linestyle='none',
            marker='o',
            markerfacecolor=tm_color(window_hours),
            markeredgecolor=tm_color(window_hours),
            markersize=math.sqrt(tm_marker_size(window_hours)),
            label=f'{window_hours:g} h',
        )
        for window_hours in representative_windows
    ]

    # Dedicated adjustable container for the circle-size legend.
    # Bounds use the unused sixth panel's coordinates:
    #     (left, bottom, width, height)
    # The default full-panel bounds preserve the previous placement exactly.
    circle_legend_ax = tm_key_ax.inset_axes(
        ((0.0, 0.0, 1.0, 1.0)),
        transform=tm_key_ax.transAxes,
    )
    circle_legend_ax.set_axis_off()
    circle_legend_ax.legend(
        handles=size_handles,
        loc=('center'),
        bbox_to_anchor=((0.72, 0.5)),
        ncol=(1),
        frameon=(False),
        markerscale=(1.0),
        fontsize=(None),
        handletextpad=(0.3),
        labelspacing=(1.5),
        borderaxespad=(0.0),
    )

    # Match Figure 9's subplot geometry and spacing.
    fig.subplots_adjust(
        left=(0.12),
        right=(0.95),
        top=(1.0),
        bottom=(0.0),
        wspace=(0.25),
        hspace=(0.5),
    )

    figure7_output_dir = OUTPUT_DIR / 'Figure7_MarkovWindowSweep'
    figure7_output_dir.mkdir(parents=True, exist_ok=True)
    png_path = figure7_output_dir / 'CustomVisPaper_Figure7_MarkovWindowSweepPareto.png'
    pdf_path = figure7_output_dir / 'CustomVisPaper_Figure7_MarkovWindowSweepPareto.pdf'

    save_figure(
        fig,
        png_path,
        pdf_path,
        png_kwargs={},
        pdf_kwargs={},
    )


    plt.close(fig)
    write_chart_swept_config(
        png_path,
        {
            workload: [
                point.metrics_path.parent
                for point in points
                if point.workload == workload
            ] + [
                baseline.metrics_path.parent
                for baseline in baselines[workload]
            ]
            for workload in FIGURE7_WORKLOADS
        },
    )
    return (png_path, pdf_path)

# Generate and display Figure 7

apply_plot_style()
points = figure7_discover_points()
baselines = figure7_discover_baselines(include_oracle=False)
png_path, pdf_path = figure7_make_markov_window_sweep_pareto(points, baselines)
display(Image(filename=str(png_path)))
(png_path, pdf_path)

# Figure 7 comparison table (FCFS/B+IPM baseline)
def figure7_build_comparison_table(
    points: Sequence[MarkovWindowPoint],
    baselines: Mapping[str, Sequence[ParetoBaselinePoint]],
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for workload in FIGURE7_WORKLOADS:
        workload_baselines = list(baselines[workload])
        fcfs_matches = [b for b in workload_baselines if b.label == 'FCFS/B+IPM']
        if len(fcfs_matches) != 1:
            raise RuntimeError(
                f'Expected one FCFS/B+IPM baseline for {workload}; '
                f'found {len(fcfs_matches)}.'
            )
        fcfs = fcfs_matches[0]
        workload_points = [p for p in points if p.workload == workload]
        frontier_coordinates = set(
            pareto_frontier(
                [
                    (point.waiting_minutes, point.wasted_energy_mwh)
                    for point in workload_points
                ]
                + [
                    (baseline.waiting_minutes, baseline.wasted_energy_mwh)
                    for baseline in workload_baselines
                    if baseline.label != ORACLE_LABEL
                ]
            )
        )
        for point in workload_points:
            rows.append({
                'workload': dataset_display_label(workload),
                'series_type': 'Markov-window sweep',
                'algorithm': SWEEP_VARIANTS[point.variant]['label'],
                'markov_window_hours': point.window_hours,
                'waiting_time_minutes': point.waiting_minutes,
                'energy_waste_mwh': point.wasted_energy_mwh,
                'is_pareto': (
                    point.waiting_minutes,
                    point.wasted_energy_mwh,
                ) in frontier_coordinates,
                'waiting_time_improvement_vs_fcfs_b_ipm_percent': (
                    comparison_improvement_percent(point.waiting_minutes, fcfs.waiting_minutes)
                ),
                'energy_waste_improvement_vs_fcfs_b_ipm_percent': (
                    comparison_improvement_percent(point.wasted_energy_mwh, fcfs.wasted_energy_mwh)
                ),
            })
        for baseline in workload_baselines:
            rows.append({
                'workload': dataset_display_label(workload),
                'series_type': 'baseline',
                'algorithm': (
                    COMPARISON_BASELINE_DISPLAY_LABEL
                    if baseline.label == 'FCFS/B+IPM'
                    else baseline.label.replace(' baseline', '')
                ),
                'markov_window_hours': float('nan'),
                'waiting_time_minutes': baseline.waiting_minutes,
                'energy_waste_mwh': baseline.wasted_energy_mwh,
                'is_pareto': (
                    baseline.label != ORACLE_LABEL
                    and (
                        baseline.waiting_minutes,
                        baseline.wasted_energy_mwh,
                    ) in frontier_coordinates
                ),
                'waiting_time_improvement_vs_fcfs_b_ipm_percent': (
                    comparison_improvement_percent(baseline.waiting_minutes, fcfs.waiting_minutes)
                ),
                'energy_waste_improvement_vs_fcfs_b_ipm_percent': (
                    comparison_improvement_percent(baseline.wasted_energy_mwh, fcfs.wasted_energy_mwh)
                ),
            })
    return add_explicit_fcfs_b_ipm_comparisons(
        pd.DataFrame(rows),
        group_columns=['workload'],
        metric_specs=[
            (
                'waiting_time_minutes',
                'fcfs_b_ipm_waiting_time_minutes',
                'waiting_time_difference_minutes',
                'waiting_time_improvement_percent',
                'waiting_time_comparison',
                False,
            ),
            (
                'energy_waste_mwh',
                'fcfs_b_ipm_energy_waste_mwh',
                'energy_waste_difference_mwh',
                'energy_waste_improvement_percent',
                'energy_waste_comparison',
                False,
            ),
        ],
    )


figure7_comparison_table = display_comparison_table(
    figure7_build_comparison_table(points, baselines),
    'Figure 7 comparison table',
)


## Figure 9

Platform comparison.


In [ ]:
# Figure 9
# Run the common setup cell and Figure 1 first.

# Shared Pareto mechanics are defined once in the common setup cell.

from __future__ import annotations

from matplotlib.patches import Patch, Circle
from matplotlib.ticker import FormatStrFormatter, MultipleLocator, FuncFormatter, MaxNLocator

FIGURE9_PLATFORMS: Tuple[str, ...] = ('AOBA-64', 'Taurus-64')
FIGURE9_WORKLOADS: Tuple[str, ...] = (
    'DAS2-fs2-0-3000',
    'generated-markovian-3000',
    'SDSC-BLUE-2000-4.2-cln-0-3000',
)
FIGURE9_SDSC_WORKLOAD = 'SDSC-BLUE-2000-4.2-cln-0-3000'
FIGURE9_SDSC_PLATFORM_BY_ROW: Dict[str, str] = {
    'AOBA-64': 'AOBA-1152',
    'Taurus-64': 'Taurus-1152',
}
FIGURE9_TIMEOUT = RESULT_SAVE_3_SELECTED_TIMEOUT

FIGURE9_ALGORITHM_ORDER: Tuple[str, ...] = (
    'FCFS/B+IPM',
    'SNF+IPM',
    'SNF-ICON',
    'SNF-ICON-NF',
    'SNF-ICON-NG',
    'SNF-ICON-NGNF',
)

FIGURE9_ICON_VARIANTS: Dict[str, str] = {
    'SNF-ICON': 'snf-ssc',
    'SNF-ICON-NF': 'snf-ssc-nf',
    'SNF-ICON-NG': 'snf-ssc-ng',
    'SNF-ICON-NGNF': 'snf-ssc-ngnf',
}

FIGURE9_IPM_RUN_DIRS: Dict[str, str] = {
    'FCFS/B+IPM': f'easy_psas_timeout-{FIGURE9_TIMEOUT}',
    'SNF+IPM': f'snf_psas_timeout-{FIGURE9_TIMEOUT}',
}

FIGURE9_HEURISTIC_ICON_ROOT = RESULT_SAVE_3_ROOT / 'Heuristic-ICON'
FIGURE9_HEURISTIC_IPM_ROOT = RESULT_SAVE_3_ROOT / 'Heuristic-IPM'


@dataclass(frozen=True)
class Figure9Record:
    platform: str
    workload: str
    algorithm: str
    waiting_seconds: float
    wasted_energy_mwh: float
    metrics_path: Path

    @property
    def dataset(self) -> str:
        return self.workload


def figure9_platform_node_count(platform: str) -> int:
    match = re.search(r'-(\d+)(?:-|$)', platform)
    if match is None:
        raise ValueError(f'Could not infer node count from platform name: {platform}')
    return int(match.group(1))


def figure9_panel_platform(row_platform: str, workload: str) -> str | None:
    if workload == FIGURE9_SDSC_WORKLOAD:
        return FIGURE9_SDSC_PLATFORM_BY_ROW.get(row_platform)
    return row_platform


def figure9_icon_metrics_path(platform: str, workload: str, variant: str, timeout: int) -> Path:
    experiment_dir = (
        FIGURE9_HEURISTIC_ICON_ROOT
        / f'{variant}_{figure9_platform_node_count(platform)}'
        / platform
        / workload
    )
    run_dir = experiment_dir / f'snf_icon_timeout-{timeout}'
    metrics_path = run_dir / 'metrics.csv'
    if not metrics_path.is_file():
        raise FileNotFoundError(f'Missing Figure 9 ICON metrics: {metrics_path}')
    validate_result_config_identity(run_dir, expected_platform=platform, expected_workload=workload, expected_algorithms=('snf_icon',))
    validated_config_timeout_seconds(run_dir, timeout, 'Figure 9 ICON run directory name')
    return metrics_path


def figure9_ipm_metrics_path(platform: str, workload: str, algorithm: str, timeout: int) -> Path:
    run_dir = (
        FIGURE9_HEURISTIC_IPM_ROOT
        / f'Experiment-{figure9_platform_node_count(platform)}'
        / platform
        / workload
        / FIGURE9_IPM_RUN_DIRS[algorithm]
    )
    metrics_path = run_dir / 'metrics.csv'
    if not metrics_path.is_file():
        raise FileNotFoundError(f'Missing Figure 9 IPM metrics: {metrics_path}')
    expected_algorithms = ('easy_psas', 'easy_ipm') if algorithm == 'FCFS/B+IPM' else ('snf_psas', 'snf_ipm')
    validate_result_config_identity(run_dir, expected_platform=platform, expected_workload=workload, expected_algorithms=expected_algorithms)
    validated_config_timeout_seconds(run_dir, timeout, 'Figure 9 IPM run directory name')
    return metrics_path


def figure9_discover_records() -> List[Figure9Record]:
    records = []
    for row_platform in FIGURE9_PLATFORMS:
        for workload in FIGURE9_WORKLOADS:
            platform = figure9_panel_platform(row_platform, workload)
            if platform is None:
                continue
            for algorithm in ('FCFS/B+IPM', 'SNF+IPM'):
                metrics_path = figure9_ipm_metrics_path(platform, workload, algorithm, FIGURE9_TIMEOUT)
                wait, energy = pareto_read_metrics(
        metrics_path,
        waiting_divisor=1.0,
        source_role='figure9_pareto_metrics',
    )
                records.append(Figure9Record(platform, workload, algorithm, wait, energy, metrics_path))
            for algorithm, variant in FIGURE9_ICON_VARIANTS.items():
                metrics_path = figure9_icon_metrics_path(platform, workload, variant, FIGURE9_TIMEOUT)
                wait, energy = pareto_read_metrics(
        metrics_path,
        waiting_divisor=1.0,
        source_role='figure9_pareto_metrics',
    )
                records.append(Figure9Record(platform, workload, algorithm, wait, energy, metrics_path))
    if not records:
        raise RuntimeError('No Figure 9 records were discovered')
    return records


def figure9_workload_display_label(workload: str) -> str:
    if 'dataset_display_label' in globals():
        return dataset_display_label(workload)
    return workload


def figure9_make_pareto(records: Sequence[Figure9Record]) -> Tuple[Path, Path]:
    if not records:
        raise ValueError('No Figure 9 records supplied for Pareto plotting')

    styles = {
        alg: {'marker': (('X', '*', 'o', 's', '^', 'D'))[i], 'size': ((60, 90, 45, 45, 45, 45))[i]}
        for i, alg in enumerate(FIGURE9_ALGORITHM_ORDER)
    }

    fig, axes = plt.subplots(
        nrows=len(FIGURE9_WORKLOADS),
        ncols=len(FIGURE9_PLATFORMS),
        figsize=((4.5, 6.0)),
        squeeze=False,
    )
    panel_axes = axes.ravel()
    panel_index = 0

    for workload in FIGURE9_WORKLOADS:
        for row_platform in FIGURE9_PLATFORMS:
            ax = panel_axes[panel_index]
            platform = figure9_panel_platform(row_platform, workload)
            if platform is None:
                ax.set_visible(False)
                panel_index += 1
                continue

            panel_records = [r for r in records if r.platform == platform and r.workload == workload]
            lookup = {r.algorithm: r for r in panel_records}
            missing = [a for a in FIGURE9_ALGORITHM_ORDER if a not in lookup]
            if missing:
                raise RuntimeError(f'Figure 9 is missing {", ".join(missing)} for {platform}/{workload}')

            coords = [(r.waiting_seconds, r.wasted_energy_mwh) for r in panel_records]
            frontier = pareto_frontier(coords)
            frontier_coords = set(frontier)

            for alg in FIGURE9_ALGORITHM_ORDER:
                rec = lookup[alg]
                style = styles[alg]
                coord = (rec.waiting_seconds, rec.wasted_energy_mwh)
                is_pareto = coord in frontier_coords
                ax.scatter(
                    rec.waiting_seconds,
                    rec.wasted_energy_mwh,
                    s=style['size'],
                    marker=style['marker'],
                    facecolors=(plt.get_cmap('cividis')(0.0)) if is_pareto else 'none',
                    edgecolors=(plt.get_cmap('cividis')(0.0)) if is_pareto else (plt.get_cmap('cividis')(0.7)),
                    hatch=None if is_pareto else ('//////'),
                    label=alg,
                    zorder=3,
                )

            if len(frontier) >= 2:
                ax.plot([p[0] for p in frontier], [p[1] for p in frontier],
                        color=(plt.get_cmap('cividis')(0.0)), linestyle=('--'),
                        linewidth=(1.0), label='Pareto frontier', zorder=4)

            workload_label = figure9_workload_display_label(workload)
            ax.set_title(
                f"({chr(ord('a') + panel_index)}) {platform}\n{workload_label}",
                y=(1.02),
            )

            visible_fraction_control = (
                ({(row_platform, workload): {'x': 0.25, 'y': 0.25} for row_platform in FIGURE9_PLATFORMS for workload in FIGURE9_WORKLOADS})[
                    (row_platform, workload)
                ]
            )
            outlier_region_control = (
                ({('AOBA-64', 'DAS2-fs2-0-3000'): {'x': 0.25, 'y': 0.25}, ('AOBA-64', 'generated-markovian-3000'): {'x': 0.25, 'y': 0.25}, ('AOBA-64', 'SDSC-BLUE-2000-4.2-cln-0-3000'): {'x': 0.25, 'y': 0.25}, ('Taurus-64', 'DAS2-fs2-0-3000'): {'x': 0.25, 'y': 0.25}, ('Taurus-64', 'generated-markovian-3000'): {'x': 0.25, 'y': 0.25}, ('Taurus-64', 'SDSC-BLUE-2000-4.2-cln-0-3000'): {'x': 0.25, 'y': 0.25}})[
                    (row_platform, workload)
                ]
            )
            pareto_finalize_panel(
                ax,
                coords,
                x_margin=(0.15),
                y_margin=(0.15),
                visible_fraction_threshold=visible_fraction_control,
                outlier_region_fraction=outlier_region_control,
            )
            panel_index += 1

    # ---- Legend construction ----
    handles, labels = [], []
    seen = set()
    for ax in panel_axes:
        if not ax.get_visible():
            continue
        for h, l in zip(*ax.get_legend_handles_labels()):
            if l not in seen:
                seen.add(l)
                handles.append(h)
                labels.append(l)

    label_to_handle = dict(zip(labels, handles))

    # ================ FIX: Use compensation dictionary ================
    for alg in FIGURE9_ALGORITHM_ORDER:
        if alg not in label_to_handle:
            continue
        style = styles[alg]
        marker = style['marker']
        size = ({'X': 6.0, '*': 9.0, 'o': 5.5, 's': 5.5, '^': 5.5, 'D': 5.5, 'P': 7.0}).get(marker, (6.0))
        label_to_handle[alg] = Line2D(
            [0], [0], linestyle='none',
            marker=marker,
            markerfacecolor=('#333333'),
            markeredgecolor=('#333333'),
            markersize=size,
            label=alg,
        )

    circle_radius = (6.0) / 20
    label_to_handle['Pareto'] = Circle(
        (0.5, 0.5), radius=circle_radius,
        facecolor=(plt.get_cmap('cividis')(0.0)),
        edgecolor=(plt.get_cmap('cividis')(0.0)),
        label='Pareto',
    )
    label_to_handle['Non-Pareto'] = Circle(
        (0.5, 0.5), radius=circle_radius,
        facecolor='none',
        edgecolor=(plt.get_cmap('cividis')(0.7)),
        hatch=('//////'),
        label='Non-Pareto',
    )

    blank = Line2D([0], [0], linestyle='none', marker='', label='')

    column_order = [
        ['FCFS/B+IPM', 'SNF+IPM', 'SNF-ICON'],
        ['SNF-ICON-NF', 'SNF-ICON-NG', 'SNF-ICON-NGNF'],
        ['Pareto frontier', 'Pareto', 'Non-Pareto'],
    ]

    rows_per_col = max(len(c) for c in column_order)
    legend_handles, legend_labels = [], []
    for col in column_order:
        for i in range(rows_per_col):
            if i < len(col):
                label = col[i]
                if label not in label_to_handle:
                    legend_handles.append(blank)
                    legend_labels.append('')
                else:
                    legend_handles.append(label_to_handle[label])
                    legend_labels.append(label)
            else:
                legend_handles.append(blank)
                legend_labels.append('')

    # ---- Figure labels ----
    fig.supxlabel('Mean waiting time (seconds)', y=(-0.06))
    fig.supylabel('Energy waste (MWh)', x=(0.0))

    fig.legend(
        handles=legend_handles,
        labels=legend_labels,
        loc=('upper center'),
        bbox_to_anchor=((0.5, 1.22)),
        ncol=len(column_order),
        frameon=(True),
        columnspacing=(2.0),
        handlelength=(1.0),
        handleheight=(1.0),
        handletextpad=(0.5),
        labelspacing=(0.5),
    )

    fig.subplots_adjust(
        left=(0.12),
        right=(0.97),
        top=(1.0),
        bottom=(0.0),
        wspace=(0.2),
        hspace=(0.5),
    )

    output_dir = OUTPUT_DIR / 'Figure9_Pareto'
    output_dir.mkdir(parents=True, exist_ok=True)
    png_path = output_dir / 'CustomVisPaper_Figure9_Pareto.png'
    pdf_path = output_dir / 'CustomVisPaper_Figure9_Pareto.pdf'

    save_figure(fig, png_path, pdf_path, png_kwargs={}, pdf_kwargs={})
    plt.close(fig)

    write_chart_swept_config(
        png_path,
        {
            f'{platform} / {workload}': [
                r.metrics_path.parent
                for r in records
                if r.platform == platform and r.workload == workload
            ]
            for platform, workload in sorted(
                {(r.platform, r.workload) for r in records}
            )
        },
    )
    return png_path, pdf_path


apply_plot_style()
figure9_records = figure9_discover_records()
png_path, pdf_path = figure9_make_pareto(figure9_records)
display(Image(filename=str(png_path)))
(png_path, pdf_path)


def figure9_build_comparison_table(records: Sequence[Figure9Record]) -> pd.DataFrame:
    rows = []
    group_keys = sorted({(r.platform, r.workload) for r in records}, key=lambda x: (x[0].lower(), x[1].lower()))
    for platform, workload in group_keys:
        group = [r for r in records if r.platform == platform and r.workload == workload]
        baseline = next(r for r in group if r.algorithm == 'FCFS/B+IPM')
        frontier_coords = set(pareto_frontier([(r.waiting_seconds, r.wasted_energy_mwh) for r in group]))
        for record in group:
            rows.append({
                'platform': platform,
                'workload': dataset_display_label(workload),
                'algorithm': COMPARISON_BASELINE_DISPLAY_LABEL if record.algorithm == 'FCFS/B+IPM' else record.algorithm,
                'waiting_time_seconds': record.waiting_seconds,
                'energy_waste_mwh': record.wasted_energy_mwh,
                'is_pareto': (record.waiting_seconds, record.wasted_energy_mwh) in frontier_coords,
                'waiting_time_improvement_vs_fcfs_b_ipm_percent': comparison_improvement_percent(record.waiting_seconds, baseline.waiting_seconds),
                'energy_waste_improvement_vs_fcfs_b_ipm_percent': comparison_improvement_percent(record.wasted_energy_mwh, baseline.wasted_energy_mwh),
            })
    return add_explicit_fcfs_b_ipm_comparisons(
        pd.DataFrame(rows),
        group_columns=['platform', 'workload'],
        metric_specs=[
            ('waiting_time_seconds', 'fcfs_b_ipm_waiting_time_seconds', 'waiting_time_difference_seconds',
             'waiting_time_improvement_percent', 'waiting_time_comparison', False),
            ('energy_waste_mwh', 'fcfs_b_ipm_energy_waste_mwh', 'energy_waste_difference_mwh',
             'energy_waste_improvement_percent', 'energy_waste_comparison', False),
        ],
    )

figure9_comparison_table = display_comparison_table(
    figure9_build_comparison_table(figure9_records),
    'Figure 9 comparison table',
)